
<details>
<summary><b>🎯 Objetivos del bloque 1 — Análisis exploratorio y primeros modelos</b> (clic para desplegar)</summary>

### 1️⃣ Análisis exploratorio de datos (EDA) y preprocesamiento
- Comprender la estructura y naturaleza del dataset **Breast Cancer**.  
- Identificar variables predictoras, tipo de variable (numérica, categórica) y posibles correlaciones.  
- Detectar valores atípicos o faltantes y aplicar estrategias de **limpieza**.  
- Escalar o normalizar las variables para preparar los datos antes del modelado.

---

### 2️⃣ Implementación de modelos
- Desarrollar los algoritmos de **Regresión Logística**, ** Regresión lineal** y **K-Nearest Neighbors (KNN)**  
  - Primero **desde sus fundamentos matemáticos**, implementando paso a paso las ecuaciones.  
  - Luego, replicar los mismos modelos con **scikit-learn**, comparando resultados y eficiencia.  
- Comprender el papel de los **parámetros** en cada modelo y cómo afectan la frontera de decisión.

---

### 3️⃣ Evaluación mediante métricas de clasificación
- Analizar el desempeño de los modelos con las métricas más comunes:
  - **Accuracy:** proporción total de aciertos.  
  - **Precision:** exactitud sobre las predicciones positivas.  
  - **Recall (Sensibilidad):** capacidad para detectar verdaderos positivos.  
  - **F1-score:** equilibrio entre precision y recall.  
  - **Matriz de confusión:** interpretación visual de VP, VN, FP, FN.  
- Reflexionar sobre qué métrica es más relevante según el contexto clínico del problema.

---

### 4️⃣ Tarea de regresión
- Extender el análisis hacia un problema de **regresión continua**, usando el mismo dataset.  
- Definir una **variable objetivo numérica** (por ejemplo, el “mean radius”) y predecir su valor.  
- Implementar modelos de regresión (lineal o KNN Regressor) y analizar métricas como:
  - **MAE**, **MSE**, **RMSE** y **R²**.

---

### 5️⃣ Puente hacia el siguiente bloque — Hiperparámetros
- Introducir el concepto de **hiperparámetros**:
  - Qué son, por qué no se aprenden automáticamente.  
  - Ejemplos: `k` en KNN, `C` y `penalty` en regresión logística.  
- Comprender cómo influyen en la **complejidad** y **precisión** del modelo.  
- Preparar el terreno para el siguiente tema:  
  **Optimización de hiperparámetros** (Random Search, Grid Search, y Optuna).

</details>



In [ ]:

#@markdown ### 🔧 Configuración inicial (librerías)
#@markdown - Usaremos exclusivamente **matplotlib** para las gráficas (sin estilos explícitos).
#@markdown - Para Colab, no es necesaria instalación de paquetes adicionales.
#@markdown - si lo realiza de form alocal, por favor instale mediante el metodo !pip install  o pip install  y el nombre de la libreria faltante.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401 (habilita 3D)
from collections import Counter
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay,
    mean_absolute_error, mean_squared_error, r2_score
)
from sklearn.decomposition import PCA

from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import log_loss
from copy import deepcopy








plt.rcParams["figure.figsize"] = (7, 5)
np.set_printoptions(precision=4, suppress=True)

def add_bias(X):
    """Agrega columna de 1s para el sesgo (bias)."""
    return np.c_[np.ones((X.shape[0], 1)), X]

def train_test_shapes_check():
    print("Clasificación:", Xc_train.shape, Xc_test.shape, yc_train.shape, yc_test.shape)
    print("Regresión:", Xr_train.shape, Xr_test.shape, yr_train.shape, yr_test.shape)

# Paletas y mallas para fronteras de decisión
cmap_light = ListedColormap(["#FFBBBB", "#BBEEFF"])
cmap_bold  = ListedColormap(["#FF3333", "#3366FF"])

np.random.seed(42)


: 

#EDA

In [ ]:

#@markdown ### 📦 Carga del dataset (clasificación)
#@markdown Usaremos `sklearn.datasets.load_breast_cancer()`, con 30 atributos y una etiqueta binaria: 0=maligno, 1=benigno.
#@markdown link datacard https://archive.ics.uci.edu/dataset/17/breast+cancer+wisconsin+diagnostic

data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name="target")
y_cls  = pd.Series(data.target, name="target")        # Clasificación: benigno (1) vs maligno (0)
y_reg  = X['mean radius'].copy()

X.head(), y.value_counts()


In [ ]:

#@markdown ### 📊 Distribución de clases
#@markdown Analizar la **distribución de clases** es un paso fundamental en el **análisis exploratorio de datos (EDA)**,
#@markdown especialmente en problemas de **clasificación binaria** como el del dataset *Breast Cancer*.
#@markdown
#@markdown **¿Por qué es importante?**
#@markdown - Permite detectar si el conjunto de datos está **balanceado o desbalanceado** entre las clases (por ejemplo, entre tumores benignos y malignos).
#@markdown - Un **desequilibrio en las clases** puede provocar que los modelos tiendan a predecir con mayor frecuencia la clase mayoritaria, reduciendo su capacidad para detectar casos minoritarios (como los tumores malignos, que son clínicamente más relevantes).
#@markdown - Conocer la proporción de clases ayuda a decidir si se necesitan **técnicas de balanceo**, como *undersampling*, *oversampling* o *SMOTE*.
#@markdown - También influye en la **selección de métricas**: en datasets desbalanceados, la *accuracy* puede ser engañosa y métricas como *precision*, *recall* o *F1-score* ofrecen una evaluación más realista.
#@markdown
#@markdown En este caso, se analizará cuántas muestras corresponden a cada clase:
#@markdown - `0`: Tumor maligno
#@markdown - `1`: Tumor benigno
#@markdown
#@markdown El conocimiento de esta distribución nos permite **anticipar el comportamiento esperado del modelo**,
#@markdown entender su sesgo inicial hacia una clase y establecer un punto de partida sólido para la evaluación posterior.

fig, ax = plt.subplots()
counts = y.value_counts().sort_index()
ax.bar(['Maligno (0)', 'Benigno (1)'], counts.values, color=['#e74c3c', '#2ecc71'])
ax.set_title('Distribución de clases')
ax.set_ylabel('Número de muestras')
plt.show()


In [ ]:

#@markdown ### 📈 Estadísticos descriptivos (primeras 8 columnas para vista rápida)
#@markdown La función `describe()` de **pandas** genera un resumen estadístico de las variables numéricas del conjunto de datos.
#@markdown Este resumen forma parte esencial del **análisis exploratorio de datos (EDA)**, ya que nos permite obtener una **visión global del comportamiento de cada atributo** antes de aplicar modelos de Machine Learning.
#@markdown
#@markdown **¿Qué muestra `describe()`?**
#@markdown - `count`: número de observaciones válidas (sin valores faltantes).
#@markdown - `mean`: promedio aritmético de cada variable.
#@markdown - `std`: desviación estándar (grado de dispersión respecto al promedio).
#@markdown - `min` y `max`: valores mínimo y máximo (rango de la variable).
#@markdown - `25%`, `50%` y `75%`: percentiles o cuartiles que dividen los datos en cuatro partes iguales.
#@markdown
#@markdown **¿Por qué es importante?**
#@markdown - Permite **detectar valores extremos o atípicos** en las variables.
#@markdown - Facilita la **comparación de escalas** entre atributos (algunos pueden estar en cientos, otros en milésimas).
#@markdown - Ayuda a **entender la variabilidad** de los datos, identificando columnas que podrían necesitar normalización o estandarización.
#@markdown - Sirve como base para **seleccionar variables relevantes** y preparar el preprocesamiento.
#@markdown
#@markdown **Interpretación práctica:**
#@markdown - Si la desviación estándar (`std`) es muy alta, la variable presenta una gran dispersión y puede dominar en los modelos no escalados.
#@markdown - Si el `min` y el `max` difieren mucho entre variables, el modelo puede verse afectado por escalas distintas.
#@markdown - Si `count` es menor que el número total de filas, existen valores faltantes que deben tratarse.
#@markdown
#@markdown En este caso visualizamos solo las primeras 8 columnas para simplificar la lectura:
X.describe().iloc[:, :8]



In [ ]:

#@markdown ### 🎻 Diagramas de violín — 10 rasgos nucleares (valores promedio)
#@markdown A continuación visualizamos, por clase, la distribución de las **10 características reales** calculadas para cada núcleo celular,
#@markdown usando sus columnas `mean` del dataset (sklearn provee `mean`, `se` y `worst` para cada rasgo):
#@markdown
#@markdown **Rasgos (definición y lectura):**
#@markdown - **radius (mean radius)**: distancia media del centro al perímetro. Núcleos mayores suelen tener radios mayores.
#@markdown - **texture (mean texture)**: desviación estándar de intensidades en escala de grises; refleja heterogeneidad del tejido.
#@markdown - **perimeter (mean perimeter)**: longitud del contorno del núcleo; correlaciona con tamaño y complejidad.
#@markdown - **area (mean area)**: área del núcleo; junto con perímetro/radio apunta a crecimiento o alteraciones morfológicas.
#@markdown - **smoothness (mean smoothness)**: variación local de los radios; bordes “irregulares” elevan este valor.
#@markdown - **compactness (mean compactness)**: (perimeter² / area − 1.0); compacidad/“redondez” del contorno.
#@markdown - **concavity (mean concavity)**: severidad de porciones cóncavas en el contorno (deformaciones pronunciadas).
#@markdown - **concave points (mean concave points)**: número de porciones cóncavas; cuenta cuántas “hendiduras” hay.
#@markdown - **symmetry (mean symmetry)**: grado de simetría del núcleo; asimetrías pueden sugerir malignidad.
#@markdown - **fractal dimension (mean fractal dimension)**: aproximación “línea costera” − 1; complejidad del borde.
#@markdown
#@markdown **¿Para qué sirve este gráfico en ML?**
#@markdown - Compara **distribuciones por clase**: ver *dónde están las medianas/medias*, *anchos* (dispersión) y *colas* (outliers).
#@markdown - Permite juzgar **separabilidad de clases**: si los violines de maligno vs. benigno **se solapan poco** para una variable,
#@markdown   esa variable podría tener alto poder discriminativo para un clasificador lineal/no lineal.
#@markdown - Ayuda a detectar **asimetrías y outliers** que motivan **escalado** (p. ej., StandardScaler) o **transformaciones** (log, Box-Cox).
#@markdown - Informa la **selección de atributos**: rasgos con distribuciones casi idénticas entre clases podrían aportar menos señal.
#@markdown
#@markdown **Cómo interpretar un violín:**
#@markdown - El grosor del “violín” indica **densidad** (qué valores son más frecuentes).
#@markdown - La línea del **mean** (marcada) ayuda a comparar tendencias centrales entre clases.
#@markdown - **Solapamiento** alto entre violines sugiere que ese rasgo, por sí solo, separa peor las clases (aunque puede ayudar combinado con otros).
#@markdown
#@markdown A continuación graficamos los 10 rasgos (versión `mean`) en panel vertical para lectura rápida:

cols_violin = [
    'mean radius', 'mean texture', 'mean perimeter', 'mean area',
    'mean smoothness', 'mean compactness', 'mean concavity',
    'mean concave points', 'mean symmetry', 'mean fractal dimension'
]

fig, axes = plt.subplots(len(cols_violin), 1, figsize=(10, 5.2*len(cols_violin)))
if len(cols_violin) == 1:
    axes = [axes]

for i, col in enumerate(cols_violin):
    ax = axes[i]
    data_mal = X[y==0][col].values
    data_ben = X[y==1][col].values
    parts = ax.violinplot(
        [data_mal, data_ben],
        showmeans=True, showmedians=False, showextrema=False
    )
    ax.set_xticks([1, 2])
    ax.set_xticklabels(['Maligno (0)', 'Benigno (1)'])
    ax.set_ylabel(col)
    ax.set_title(f'Violin plot — {col}')

plt.tight_layout()
plt.show()



In [ ]:
#@markdown ### 🔗 Matriz de correlación (interpretación + métodos lineales y no lineales)
#@markdown La **matriz de correlación** resume el grado de asociación entre pares de variables.
#@markdown En ML, su lectura ayuda a:
#@markdown - **Detectar multicolinealidad** (columnas muy correlacionadas) que puede afectar a modelos lineales (p. ej., regresión logística) y a la **interpretación** de coeficientes.
#@markdown - **Seleccionar/filtrar variables** redundantes antes de entrenar.
#@markdown - **Entender relaciones** que justifican transformaciones (escalado, logaritmos) o técnicas de reducción de dimensionalidad (PCA).
#@markdown
#@markdown **Métodos (lineales/monotónicos):**
#@markdown - `pearson` (lineal): mide relación **lineal** entre variables (supone continuidad y distribución razonable).
#@markdown - `spearman` (monótona): usa rangos; captura relaciones **monótonas** (no estrictamente lineales) y es robusta a outliers.
#@markdown - `kendall` (monótona): basada en concordancias/discordancias; más robusta pero más costosa computacionalmente.
#@markdown
#@markdown **Limitación:** correlaciones lineales/monótonas pueden pasar por alto relaciones **no lineales complejas**.
#@markdown Para eso agregamos **Distance Correlation (dCor)**, que vale 0 si y sólo si las variables son independientes (capta no linealidad).
#@markdown
#@markdown **Lectura de la matriz:**
#@markdown - Valores cercanos a **+1**: asociación positiva fuerte.
#@markdown - Valores cercanos a **−1**: asociación negativa fuerte.
#@markdown - Valores cercanos a **0**: poca o ninguna relación (para el método elegido).
#@markdown - Zonas con |ρ| alto entre features sugieren **redundancia**; podrías eliminar uno, regularizar (p. ej., `C` pequeño en LogReg) o usar PCA.

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# === 1) Correlación clásica (Pearson/Spearman/Kendall) con anotaciones ===
corr_method = 'pearson'  #@param ['pearson', 'spearman', 'kendall'] {allow-input: true}
subset = X.iloc[:, :30]   # usamos 30 features (todas las 'mean', 'se' y 'worst' del dataset sklearn)

corr = subset.corr(method=corr_method)

fig, ax = plt.subplots(figsize=(10,10))
im = ax.imshow(corr.values, vmin=-1, vmax=1, cmap='coolwarm', interpolation='nearest')
ax.set_title(f'Correlación ({corr_method}) — primeras {subset.shape[1]} variables')
ax.set_xticks(range(subset.shape[1])); ax.set_xticklabels(subset.columns, rotation=90)
ax.set_yticks(range(subset.shape[1])); ax.set_yticklabels(subset.columns)

# Anotaciones numéricas dentro de cada celda
n = corr.shape[0]
for i in range(n):
    for j in range(n):
        ax.text(j, i, f"{corr.values[i, j]:.2f}", ha='center', va='center', fontsize=7)

cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label('Coeficiente de correlación')
plt.tight_layout()
plt.show()

# === 2) Distance Correlation (para relaciones potencialmente no lineales) ===
# Implementación de distance correlation entre dos vectores (O(N^2), usar con cuidado en datasets grandes)
def _centered_distance_matrix(a):
    # a: vector (N,) -> matriz de distancias euclidianas |a_i - a_j|
    a = a.reshape(-1, 1)
    D = np.abs(a - a.T)  # (N,N)
    # doble centrado
    n = D.shape[0]
    J = np.eye(n) - np.ones((n, n))/n
    A = J @ D @ J
    return A

def distance_correlation(x, y):
    x = np.asarray(x, dtype=float).ravel()
    y = np.asarray(y, dtype=float).ravel()
    A = _centered_distance_matrix(x)
    B = _centered_distance_matrix(y)
    # distance covariance
    dcov2_xy = np.mean(A * B)
    dcov2_xx = np.mean(A * A)
    dcov2_yy = np.mean(B * B)

    # Corrección por posibles numéricos negativos muy pequeños
    dcov2_xy = max(dcov2_xy, 0.0)
    dcov2_xx = max(dcov2_xx, 0.0)
    dcov2_yy = max(dcov2_yy, 0.0)

    # distance correlation
    denom = np.sqrt(dcov2_xx * dcov2_yy)
    if denom == 0:
        return 0.0
    return np.sqrt(dcov2_xy / denom)

# Construimos la matriz de distance correlation para un subconjunto (p.ej., 30 variables para mantener tiempo razonable)
cols_dcor = subset.columns[:30]  # ajusta si quieres más/menos
M = len(cols_dcor)
dcor_mat = np.zeros((M, M))
for i in range(M):
    for j in range(M):
        dcor_mat[i, j] = distance_correlation(subset[cols_dcor[i]].values, subset[cols_dcor[j]].values)

fig, ax = plt.subplots(figsize=(10,10))
im2 = ax.imshow(dcor_mat, vmin=0, vmax=1, cmap='viridis', interpolation='nearest')
ax.set_title(f'Distance Correlation — primeras {M} variables')
ax.set_xticks(range(M)); ax.set_xticklabels(cols_dcor, rotation=90)
ax.set_yticks(range(M)); ax.set_yticklabels(cols_dcor)

# Anotar valores
for i in range(M):
    for j in range(M):
        ax.text(j, i, f"{dcor_mat[i, j]:.2f}", ha='center', va='center', fontsize=7, color='white' if dcor_mat[i,j]>0.5 else 'black')

cbar2 = fig.colorbar(im2, ax=ax, fraction=0.046, pad=0.04)
cbar2.set_label('Distance Correlation')
plt.tight_layout()
plt.show()

#@markdown **Guía rápida de interpretación (práctica):**
#@markdown - Si ves pares con |ρ|>0.9 (Pearson), probablemente hay **redundancia fuerte** → evalúa eliminar uno, o usar regularización/PCA.
#@markdown - Si Spearman/Kendall muestran asociaciones donde Pearson no, puede haber **relaciones monótonas no lineales** o outliers.
#@markdown - Si Distance Correlation es alto pero Pearson/Spearman son bajos, sospecha **relación no lineal** (p. ej., cuadrática, sinusoidal).
#@markdown - Usa estas señales para:
#@markdown   1) preprocesar (escalar/transformar),
#@markdown   2) seleccionar variables,
#@markdown   3) elegir modelos (lineales vs. no lineales) y
#@markdown   4) definir estrategias de regularización.



In [ ]:
#@markdown ### 🧠 Discusión: Selección y eliminación de variables redundantes
#@markdown Tras observar las matrices de correlación (lineal y no lineal),
#@markdown es evidente que **muchas variables del dataset Breast Cancer** describen información muy similar:
#@markdown por ejemplo, `mean radius`, `mean perimeter` y `mean area` representan **tamaño del núcleo** en distintas métricas.
#@markdown
#@markdown **¿Por qué eliminar variables altamente correlacionadas?**
#@markdown 1. **Evitar multicolinealidad:**
#@markdown    En modelos lineales (como la regresión logística), variables muy correlacionadas provocan
#@markdown    inestabilidad en los coeficientes (pequeños cambios en los datos → grandes cambios en pesos).
#@markdown 2. **Evitar sobrepeso en modelos basados en distancia (KNN):**
#@markdown    Si dos variables miden lo mismo, su efecto en la distancia euclidiana se duplica, distorsionando la clasificación.
#@markdown 3. **Reducir dimensionalidad sin perder información:**
#@markdown    Mantener una sola variable representativa de cada grupo mejora la eficiencia del modelo
#@markdown    y evita sobreajuste.
#@markdown 4. **Mejorar interpretabilidad clínica:**
#@markdown    En contextos médicos, menos variables redundantes facilitan explicar la contribución de cada rasgo.
#@markdown
#@markdown ---
#@markdown **Criterios de decisión (discusión guiada):**
#@markdown - Si |ρ| > 0.9 entre dos variables → **eliminar una de ellas.**
#@markdown - Conservar aquella que:
#@markdown   - Sea **más intuitiva clínicamente** (por ejemplo, `mean radius` es más comprensible que `mean perimeter`).
#@markdown   - Presente **menor correlación promedio** con las demás variables del grupo (más independiente).
#@markdown   - Tenga **distribución más estable o menos sesgada**.
#@markdown
#@markdown ---
#@markdown **Ejemplo práctico (decisiones):**
#@markdown - De las variables relacionadas con **tamaño** → conservar `mean radius`, eliminar `mean area` y `mean perimeter`.
#@markdown - De las relacionadas con **concavidad** → conservar `mean concavity`, eliminar `mean compactness` y `mean concave points`.
#@markdown - De las “versiones *worst*” → conservar `worst radius` y `worst concave points`, eliminar el resto redundante.
#@markdown - Mantener variables que aportan información **geométrica o de textura**, como `mean smoothness`, `mean texture`, `mean symmetry` y `fractal dimension`.
#@markdown
#@markdown ---



In [ ]:
#@markdown ### ⚙️ Eliminación práctica de variables redundantes
#@markdown En este bloque realizamos dos enfoques complementarios:
#@markdown 1. **Eliminación manual** basada en el razonamiento anterior.
#@markdown 2. **Eliminación automática** usando un umbral de correlación |ρ| para detectar redundancias.
#@markdown
#@markdown ---
#@markdown #### 1️⃣ Eliminación manual (razonada)
# ========== 1) SELECCIÓN DE VARIABLES ==========
# Opción A: Manual (curado por criterio clínico/interpretabilidad)
use_manual = False  #@param {type:"boolean"}

manual_keep = [
    'mean radius', 'mean texture', 'mean smoothness', 'mean concavity',
    'mean symmetry', 'radius error', 'worst concave points', 'worst fractal dimension'
]

# Opción B: Automática (umbral de correlación)
corr_threshold = 0.90  #@param {type:"number"}

if use_manual:
    selected_cols = [c for c in manual_keep if c in X.columns]
    X_selected = X[selected_cols].copy()
else:
    corr_matrix = X.corr().abs()
    upper_triangle = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    to_drop = [col for col in upper_triangle.columns if any(upper_triangle[col] > corr_threshold)]
    X_selected = X.drop(columns=to_drop)
    selected_cols = list(X_selected.columns)

print("✅ Columnas seleccionadas:", selected_cols)
print("Total columnas tras selección:", len(selected_cols))


#@markdown ---
#@markdown **Comparación:**
#@markdown - **Método manual:** depende de conocimiento del dominio y criterio interpretativo.
#@markdown - **Método automático:** usa umbrales cuantitativos (rápido, pero ciego al significado clínico).
#@markdown
#@markdown En la práctica, se recomienda combinar ambos enfoques:
#@markdown 1. Identificar correlaciones altas automáticamente.
#@markdown 2. Tomar decisiones de eliminación basadas en criterio clínico o de interpretabilidad.



In [ ]:
# ========== 2) DEFINIR MATRICES PARA CADA TAREA ==========
# Clasificación: usamos X_selected tal cual; y = y_cls
X_cls_sel = X_selected.copy()

# Regresión: quitar la variable objetivo de X si está presente
X_reg_sel = X_selected.drop(columns=['mean radius'], errors='ignore').copy()


In [ ]:
# ========== 3) TRAIN/TEST SPLIT ==========
Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    X_cls_sel, y_cls, test_size=0.2, stratify=y_cls, random_state=42
)

Xr_train, Xr_test, yr_train, yr_test = train_test_split(
    X_reg_sel, y_reg, test_size=0.2, random_state=42
)

print("\n📦 Clasificación — shapes:")
print("Xc_train:", Xc_train.shape, "| Xc_test:", Xc_test.shape, "| y:", yc_train.shape, yc_test.shape)

print("\n📦 Regresión — shapes:")
print("Xr_train:", Xr_train.shape, "| Xr_test:", Xr_test.shape, "| y:", yr_train.shape, yr_test.shape)

#Regresión Lineal

##regresión

<details>
<summary><b>📘 Regresión Lineal — Diccionario de símbolos y lectura de cada ecuación</b> (clic para desplegar)</summary>

### 🧭 Diccionario de símbolos (qué es cada cosa)

- **$N$**: número de muestras/filas (pacientes) en el conjunto de datos.
- **$d$**: número de predictores/columnas (características) usadas para predecir.
- **$\mathbf{x}_i \in \mathbb{R}^d$**: vector de características de la **muestra $i$** (la fila $i$ sin el sesgo).
- **$y_i \in \mathbb{R}$**: valor **real/verdadero** de la variable objetivo para la muestra $i$.
- **$\hat{y}_i \in \mathbb{R}$**: valor **predicho** por el modelo para la muestra $i$.
- **$\mathbf{w} \in \mathbb{R}^d$**: **vector de pesos** (un coeficiente por predictor).
- **$b \in \mathbb{R}$**: **sesgo** o **intercepto** (desplaza la recta/hiperplano).
- **$\mathbf{X}\in\mathbb{R}^{N\times d}$**: **matriz de diseño** con todas las muestras (cada fila es $\mathbf{x}_i^\top$).
- **$\mathbf{1}\in\mathbb{R}^{N\times 1}$**: columna de unos.
- **$\mathbf{X}_b=[\mathbf{X} \; \mathbf{1}] \in\mathbb{R}^{N\times (d+1)}$**: **matriz aumentada** con la columna de unos (permite absorber $b$ en los pesos).
- **$\mathbf{w}_b \in\mathbb{R}^{d+1}$**: vector de pesos **incluyendo** el sesgo como último componente.
- **$^\top$**: traspuesta (cambia filas por columnas).
- **$\eta$**: **tasa de aprendizaje** (learning rate) en gradiente descendente.

---

## 🔢 Función de pérdida (MSE)

$$
\mathcal{L}_{\text{MSE}}(\mathbf{w}, b)
= \frac{1}{N}\sum_{i=1}^N\big(\hat{y}_i - y_i\big)^2,
\qquad
\hat{y}_i = \mathbf{w}^\top \mathbf{x}_i + b
$$

**Lectura:** promedia el **error cuadrático** entre predicción y valor real.  
- $(\hat{y}_i-y_i)$ es el **residuo** de la muestra $i$.  
- Elevar al cuadrado penaliza errores grandes y hace derivable la pérdida.

**Shapes:**  
- $\hat{y}_i, y_i \in \mathbb{R}$ → $(\hat{y}_i-y_i)^2 \in \mathbb{R}$ → la suma/medio dan un **escalar**.

---

## ✏️ Gradientes (cómo cambia la pérdida al mover cada parámetro)

$$
\frac{\partial \mathcal{L}}{\partial \mathbf{w}}
= \frac{2}{N}\sum_{i=1}^N(\hat{y}_i - y_i)\,\mathbf{x}_i,
\qquad
\frac{\partial \mathcal{L}}{\partial b}
= \frac{2}{N}\sum_{i=1}^N(\hat{y}_i - y_i)
$$

**Lectura:**  
- Para cada peso $w_j$, el cambio está guiado por la covariación del **error** con la característica $x_{ij}$.  
- Si una característica suele ser **positiva cuando el modelo se equivoca hacia arriba** ($\hat{y}-y>0$), su peso se **reduce**; si el error es negativo, su peso se **incrementa**.

**Shapes:**  
- $\frac{\partial \mathcal{L}}{\partial \mathbf{w}}\in\mathbb{R}^d$ (un gradiente por peso).  
- $\frac{\partial \mathcal{L}}{\partial b}\in\mathbb{R}$.

---

## ⬇️ Regla de actualización (Gradiente Descendente)

$$
\mathbf{w}\leftarrow\mathbf{w}-\eta\,\frac{\partial\mathcal{L}}{\partial\mathbf{w}},
\qquad
b\leftarrow b-\eta\,\frac{\partial\mathcal{L}}{\partial b}
$$

**Lectura:** nos movemos en la **dirección contraria** al gradiente (la que **disminuye** la pérdida).  
- **$\eta$** controla el tamaño del paso:  
  - grande → rápido pero puede **oscilar/divergir**,  
  - pequeña → **lento** pero estable.

---

## 🧷 Ecuación normal (solución analítica, absorbiendo el sesgo)

$$
\mathbf{w}_b = \big(\mathbf{X}_b^\top \mathbf{X}_b\big)^{-1}\mathbf{X}_b^\top \mathbf{y}
$$

**Lectura:** da los pesos **óptimos** (en MSE) en **una sola operación** matricial, cuando existe la inversa y el problema es bien condicionado.  
- En la práctica se usa la **pseudoinversa** (o descomposiciones SVD/QR) por **estabilidad numérica**.

**Shapes:**  
- $\mathbf{X}_b^\top\mathbf{X}_b \in \mathbb{R}^{(d+1)\times(d+1)}$  
- $\big(\mathbf{X}_b^\top\mathbf{X}_b\big)^{-1}\in \mathbb{R}^{(d+1)\times(d+1)}$  
- $\mathbf{X}_b^\top\mathbf{y}\in \mathbb{R}^{(d+1)\times 1}$  
- Resultado $\mathbf{w}_b\in\mathbb{R}^{(d+1)\times 1}$ (último elemento es el sesgo).

---

## 🧪 Notas prácticas (para no perderse)
- **Estandariza** los predictores antes de usar **GD** (convergencia más rápida y estable).  
- Si $d$ es grande o hay multicolinealidad, **prefiere GD/regularización** sobre la inversa directa.  
- Con $\mathbf{X}_b$ puedes **programar todo** sin tratar $b$ por separado (última componente de $\mathbf{w}_b$ actúa como sesgo).

</details>



<details>
<summary><b>⚙️ Escalamiento y reducción de dimensionalidad — ¿Por qué son necesarios en ML?</b> (clic para desplegar)</summary>

### 🧩 1️⃣ ¿Qué problema resuelve el escalado?

Muchos modelos de *Machine Learning* (Regresión Lineal, Logística, KNN, SVM, PCA) **usan distancias o gradientes** para ajustar parámetros o definir fronteras.  
Si las variables tienen escalas diferentes, una característica puede **dominar numéricamente a las demás**.

📉 **Ejemplo real:**  
- `mean area` toma valores de 300–1000  
- `mean smoothness` toma valores de 0.05–0.15  
Sin escalado, el modelo “cree” que `mean area` es 10 000 veces más importante solo por su rango.

Esto causa:
- Pesos (`w`) con magnitudes desbalanceadas.  
- Gradientes que oscilan o divergen (no convergencia).  
- Fronteras de decisión distorsionadas.  
- Modelos KNN/SVM que miden mal la similitud entre puntos.

---

### ⚖️ 2️⃣ Tipos comunes de escalado

| Método | Fórmula | Ideal cuando | Efecto |
|:--|:--|:--|:--|
| **StandardScaler** | $z = (x - \mu)/\sigma$ | Datos sin outliers fuertes | Centra en 0, varianza 1 |
| **MinMaxScaler** | $z = (x - x_{min})/(x_{max}-x_{min})$ | Rango fijo (ej. [0,1]) | Mantiene forma relativa |
| **RobustScaler** | $z = (x - Q_2)/(Q_3-Q_1)$ | Con outliers | Usa mediana e IQR |

---

### 🔎 3️⃣ ¿Cómo afecta visualmente?

1. **Regresión Logística (clasificación):**  
   Frontera mal definida o “deformada” si las variables no están en la misma escala.  
2. **Regresión Lineal:**  
   Coeficientes inestables y gradiente descendente lento.  
3. **KNN / SVM:**  
   Distancias euclidianas distorsionadas → mal cálculo de vecinos.

---

### 🧠 4️⃣ Reducción / expansión de dimensionalidad

#### 🔺 *Expansión polinómica*
Crea nuevas combinaciones no lineales:
\[
x_1, x_2 \rightarrow x_1^2, x_2^2, x_1x_2
\]
Permite capturar **relaciones no lineales** en modelos lineales, pero puede aumentar la **varianza (sobreajuste)**.

#### 🔻 *PCA (Análisis de Componentes Principales)*
Reduce las dimensiones originales a combinaciones lineales que **maximizan la varianza explicada**.
\[
Z = XW
\]
- Elimina correlaciones redundantes.  
- Facilita visualización y estabilidad numérica.  
- Muy útil tras el escalado.

---

### 🔬 5️⃣ Resumen visual (lo que veremos a continuación)

- Cómo cambian los rangos y distribuciones tras escalar.  
- Cómo una frontera de clasificación se deforma sin escalar.  
- Cómo el PCA proyecta datos en un plano con mínima pérdida de información.  
- Cómo la expansión polinómica introduce curvatura en regresión.

📘 **Regla práctica:**
> Escala tus variables **antes** de entrenar modelos basados en distancias o gradientes,  
> y aplica **PCA o expansión polinómica** según el tipo de problema y la complejidad que quieras capturar.

</details>


In [ ]:
#@title 🧮 Regresión Lineal — Matemática y Gradiente Descendente (desde cero)
#@markdown **Hipótesis (con bias):**
#@markdown $$\hat{y}_i = \mathbf{w}^\top \mathbf{x}_i + b$$
#@markdown
#@markdown Forma matricial absorbiendo el sesgo: $$\hat{\mathbf{y}} = \mathbf{X}_b\mathbf{w}_b$$, donde $$\mathbf{X}_b \in \mathbb{R}^{N\times (d+1)}$$ incluye una columna de 1s.
#@markdown
#@markdown **Costo (MSE):**
#@markdown $$\mathcal{L}_{\text{MSE}}(\mathbf{w}, b) = \frac{1}{N}\sum_{i=1}^N (\hat{y}_i - y_i)^2$$
#@markdown
#@markdown **Gradiente:**
#@markdown $$\frac{\partial \mathcal{L}}{\partial \mathbf{w}} = \frac{2}{N}\sum_{i=1}^N(\hat{y}_i - y_i)\,\mathbf{x}_i$$
#@markdown
#@markdown En forma matricial: $$\nabla_{\mathbf{w}_b} \mathcal{L} = \frac{2}{N} \mathbf{X}_b^\top (\mathbf{X}_b\mathbf{w}_b - \mathbf{y})$$
#@markdown
#@markdown **Actualización:**
#@markdown $$\mathbf{w}_b \leftarrow \mathbf{w}_b - \eta \nabla_{\mathbf{w}_b} \mathcal{L}$$

# Elegir si entrenar con escalado o sin escalado
use_scaling_reg = True  #@param {type:"boolean"}
lr_alpha = 0.01         #@param {type:"number"}
lr_iters = 4000         #@param {type:"integer"}

# Copias para no alterar datos originales
Xtr, Xte = deepcopy(Xr_train), deepcopy(Xr_test)
ytr, yte = deepcopy(yr_train).values.reshape(-1, 1), deepcopy(yr_test).values.reshape(-1, 1)

scaler_reg = None
if use_scaling_reg:
    scaler_reg = StandardScaler().fit(Xtr)
    Xtr = scaler_reg.transform(Xtr)
    Xte = scaler_reg.transform(Xte)

# Preparar matrices con bias
Xtr_b = add_bias(Xtr)  # (N, d+1) - matriz aumentada
Xte_b = add_bias(Xte)

N, d_plus_1 = Xtr_b.shape
w_b = np.zeros((d_plus_1, 1))  # vector de pesos incluyendo sesgo

loss_hist = []
for it in range(lr_iters):
    y_pred = Xtr_b @ w_b
    residual = y_pred - ytr
    # Gradiente: (2/N) * X_b^T * (y_pred - y)
    grad = (2 / N) * (Xtr_b.T @ residual)
    w_b -= lr_alpha * grad
    # Pérdida MSE (sin el factor 1/2)
    L = (residual**2).sum() / N
    loss_hist.append(L)

print("𝐰_b aprendido (primeros 5):", w_b.ravel()[:5])
print("Pérdida final:", loss_hist[-1])

# Curva de pérdida
plt.plot(loss_hist)
plt.xlabel("Iteración")
plt.ylabel("MSE = L(𝐰, b)")
plt.title("Regresión Lineal — Descenso de Gradiente (escala=" + str(use_scaling_reg) + ")")
plt.grid(True)
plt.show()

In [ ]:
#@markdown # ⚖️ Escalado de datos y estabilidad del descenso de gradiente
#@markdown
#@markdown ## 📘 1. ¿Qué observamos en las gráficas?
#@markdown
#@markdown En la figura con escalado (`escala=True`), el **costo** $\mathcal{L}(\mathbf{w}_b)$ desciende suavemente y converge a un valor bajo.
#@markdown
#@markdown En cambio, en la figura sin escalado (`escala=False`), el costo crece de forma descontrolada (diverge), alcanzando valores enormes (~10³⁰³).
#@markdown
#@markdown Esto se debe a que **las variables no escaladas tienen magnitudes muy diferentes**, lo que altera la geometría del espacio de error.
#@markdown
#@markdown ---
#@markdown
#@markdown ## 🧩 2. ¿Qué ocurre matemáticamente?
#@markdown
#@markdown El descenso de gradiente actualiza los parámetros así:
#@markdown
#@markdown $$\mathbf{w}_b \leftarrow \mathbf{w}_b - \eta \frac{2}{N} \mathbf{X}_b^\top(\mathbf{X}_b\mathbf{w}_b - \mathbf{y})$$
#@markdown
#@markdown Si una característica $x_j$ tiene valores mucho mayores que otra, su gradiente también será mucho más grande.
#@markdown
#@markdown Esto genera pasos desiguales: **algunas direcciones avanzan demasiado rápido** (provocando explosión numérica) y **otras demasiado lento**, lo que impide converger al mínimo.
#@markdown
#@markdown **Geométricamente:**
#@markdown
#@markdown - **Sin escalado** → superficie de error $\mathcal{L}(\mathbf{w}_b)$ es **elíptica** y alargada (mal condicionada).
#@markdown - **Con escalado** → superficie es **casi circular** (bien condicionada), permitiendo descender directamente al mínimo.
#@markdown
#@markdown ---
#@markdown
#@markdown ## 🎯 3. Cómo interpretar ambos casos
#@markdown
#@markdown | Caso | Superficie de error | Comportamiento del gradiente | Resultado |
#@markdown |:----:|:--------------------|:-----------------------------|:-----------|
#@markdown | ✅ **Con escalado** (`StandardScaler`) | Circular / isotrópica | Pasos regulares y estables | Convergencia rápida |
#@markdown | ❌ **Sin escalado** | Elíptica / mal condicionada | Saltos irregulares o explosión | Divergencia numérica |
#@markdown
#@markdown ---
#@markdown
#@markdown ## 🔧 4. ¿Cómo solucionarlo?
#@markdown
#@markdown **A. Escalar las características**
#@markdown
#@markdown - Usa `StandardScaler()` o `MinMaxScaler()` de *scikit-learn* antes de entrenar el modelo.
#@markdown - Esto hace que todas las variables tengan media 0 y desviación estándar 1, o un rango común [0,1].
#@markdown
#@markdown **B. Ajustar la tasa de aprendizaje** $\eta$
#@markdown
#@markdown - Si no puedes escalar, reduce $\eta$ drásticamente (por ejemplo de `0.05` a `0.0001`).
#@markdown - Esto evita que el gradiente "salte" fuera del valle del error, aunque el entrenamiento será mucho más lento.
#@markdown
#@markdown **C. Usar optimizadores adaptativos**
#@markdown
#@markdown - En redes neuronales o modelos más complejos, usar métodos como **Adam**, **RMSProp** o **Adagrad**, que ajustan la tasa de aprendizaje automáticamente por parámetro.
#@markdown
#@markdown ---
#@markdown
#@markdown ## 💡 5. Recomendación práctica
#@markdown
#@markdown Siempre **estandariza tus predictores** antes de aplicar métodos basados en gradiente (como regresión lineal, logística, SVM o redes neuronales).
#@markdown
#@markdown Así garantizas que el modelo:
#@markdown
#@markdown - **Converge más rápido** (menos iteraciones necesarias)
#@markdown - **Evita inestabilidad numérica** (no hay overflow/underflow)
#@markdown - **Aprende coeficientes más interpretables** (todos en la misma escala)
#@markdown
#@markdown ---
#@markdown
#@markdown ## 🔎 6. Mini experimento (puedes probar)
#@markdown
#@markdown Intenta cambiar la tasa de aprendizaje y observa la diferencia:
#@markdown
#@markdown ```python
#@markdown lr_alpha = 0.001  # tasa mucho más pequeña
#@markdown use_scaling_reg = False
#@markdown ```
#@markdown
#@markdown Luego ejecuta el entrenamiento y revisa si el costo deja de explotar.
#@markdown
#@markdown ---
#@markdown
#@markdown ## 📌 Conclusión
#@markdown
#@markdown El escalado de características **no cambia la información del modelo**, pero **hace que el gradiente descendente aprenda de forma eficiente y estable**.
#@markdown
#@markdown La razón: garantiza que el **número de condición** de la matriz $\mathbf{X}_b^\top\mathbf{X}_b$ sea pequeño, lo que significa que la superficie de error no está distorsionada.

# Tu código aquí (si tienes código en esta celda)

In [ ]:
#@title 📈 Visualización 3D: Plano de regresión en 2 componentes (PCA solo para graficar)
#@markdown Para **graficar**, reducimos X a 2 componentes (PCA).
#@markdown Entrenamos **otro** modelo lineal desde cero **solo** en estas 2 features para poder mostrar el **plano**.

# Reducimos a 2D para plot (solo visual)
pca_2 = PCA(n_components=3).fit(Xr_train if scaler_reg is None else scaler_reg.inverse_transform(Xtr))
# OJO: Para consistencia, transformamos los conjuntos usando el mismo preprocesamiento del entrenamiento de esta celda
Xr_train_plot = Xr_train
Xr_test_plot = Xr_test
if use_scaling_reg:
    # Para la visual: escalamos y luego PCA sobre escalado de ENTRENAMIENTO auténtico (no el transformado de la celda anterior)
    sc_viz = StandardScaler().fit(Xr_train_plot)
    Xr_train_plot = sc_viz.transform(Xr_train_plot)
    Xr_test_plot = sc_viz.transform(Xr_test_plot)

pca = PCA(n_components=2).fit(Xr_train_plot)
Ztr = pca.transform(Xr_train_plot)
Zte = pca.transform(Xr_test_plot)

# Entrenamos un modelo lineal desde cero en Z (2D) para graficar plano
Ztr_b = add_bias(Ztr)
theta_viz = np.zeros((Ztr_b.shape[1], 1))
alpha_viz = 0.01
iters_viz = 4000
ytr_v = yr_train.values.reshape(-1, 1)

for _ in range(iters_viz):
    pred = Ztr_b @ theta_viz
    grad = (Ztr_b.T @ (pred - ytr_v)) / Ztr_b.shape[0]
    theta_viz -= alpha_viz * grad

# Malla y plano
z1_min, z1_max = Ztr[:,0].min()-0.5, Ztr[:,0].max()+0.5
z2_min, z2_max = Ztr[:,1].min()-0.5, Ztr[:,1].max()+0.5
Z1, Z2 = np.meshgrid(np.linspace(z1_min, z1_max, 40), np.linspace(z2_min, z2_max, 40))
Z_grid = add_bias(np.c_[Z1.ravel(), Z2.ravel()])
Y_grid = (Z_grid @ theta_viz).reshape(Z1.shape)

fig = plt.figure(figsize=(8,6))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(Ztr[:,0], Ztr[:,1], yr_train, alpha=0.6, label="Train")
ax.plot_surface(Z1, Z2, Y_grid, alpha=0.4, linewidth=0, antialiased=True)
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_zlabel("y")
ax.set_title("Plano de regresión (visualización en 2D por PCA)")
plt.show()

#@markdown # 📈 Interpretación del plano de regresión (visualización 2D con PCA)
#@markdown
#@markdown - Los **puntos azules** representan los datos originales proyectados en dos ejes principales (PC1 y PC2).
#@markdown - El **plano semitransparente** es la superficie de predicción del modelo lineal: $\hat{y} = w_0 + w_1z_1 + w_2z_2$, donde $z_1, z_2$ son las componentes principales.
#@markdown - **Cada punto** muestra un caso real y su posición vertical indica el valor observado de la variable objetivo $y_i$.
#@markdown - **El plano** muestra la tendencia que el modelo aprendió: cómo cambia $\hat{y}$ cuando varían las combinaciones lineales de las características proyectadas.
#@markdown
#@markdown ---
#@markdown
#@markdown ## 💡 Interpretación
#@markdown
#@markdown - **Si el plano se alinea bien con los puntos** → existe una buena relación lineal entre predictores y objetivo.
#@markdown - **Si los puntos están muy dispersos respecto al plano** → la relación no es completamente lineal, hay ruido, o el modelo necesita más variables/complejidad.
#@markdown
#@markdown ---
#@markdown
#@markdown ## ⚠️ Nota importante
#@markdown
#@markdown El PCA solo se usa para **reducir la dimensionalidad y visualizar en 3D** (2 features + 1 objetivo).
#@markdown
#@markdown El modelo real de regresión lineal se entrena con **todas las $d$ variables originales**, no solo con estas 2 componentes principales.
#@markdown
#@markdown Esta visualización es solo una **proyección aproximada** para entender el comportamiento del modelo, pero **no representa la superficie completa en el espacio de $d$ dimensiones**.

In [ ]:
#@title ⚙️ Regresión Lineal con Scikit-Learn (comparación con/sin escalado)
#@markdown Aquí entrenamos dos pipelines: uno con escalado y otro sin escalado.
#@markdown Nota: `LinearRegression` no necesita escalado para converger, pero lo usamos para homogeneidad pedagógica.

pipe_reg_scaled = Pipeline([
    ("scaler", StandardScaler()),
    ("linreg", LinearRegression())
])

pipe_reg_noscale = Pipeline([
    ("linreg", LinearRegression())
])

pipe_reg_scaled.fit(Xr_train, yr_train)
pipe_reg_noscale.fit(Xr_train, yr_train)

print("Coef (scaled) primeros 5:", pipe_reg_scaled["linreg"].coef_.ravel()[:5])
print("Coef (noscale) primeros 5:", pipe_reg_noscale["linreg"].coef_.ravel()[:5])

#@markdown # 📐 Interpretación de los coeficientes con y sin escalado

#@markdown **Resultados obtenidos:**
#@markdown ```
#@markdown Coef (scaled) primeros 5: [ 0.0452  0.1122  1.8511  2.066  -0.3983]
#@markdown Coef (noscale) primeros 5: [  0.0106   8.0702  35.3174  26.0269 -14.5062]
#@markdown ```

#@markdown ---
#@markdown ## 🧩 ¿Qué significa esta diferencia?
#@markdown En la regresión lineal, cada coeficiente indica **cuánto cambia la variable objetivo**
#@markdown cuando su característica correspondiente aumenta una unidad.
#@markdown
#@markdown - Cuando **no escalamos**, las variables tienen unidades diferentes (mm, días, mg, etc.),
#@markdown   y sus coeficientes compensan esas diferencias → aparecen números muy grandes o muy pequeños.
#@markdown - Cuando **escalamos** (media=0, desviación=1), los coeficientes están en una escala comparable,
#@markdown   por lo que sus magnitudes reflejan **la influencia relativa real** de cada variable en la predicción.
#@markdown
#@markdown ---
#@markdown ## ⚖️ En resumen
#@markdown | Modelo | Magnitud de coeficientes | Interpretación |
#@markdown |:--|:--|:--|
#@markdown | ✅ **Con escalado** | Valores pequeños y comparables | Indican importancia relativa |
#@markdown | ❌ **Sin escalado** | Valores grandes y desiguales | Dependen de las unidades de las variables |
#@markdown
#@markdown ---
#@markdown ## 💡 Conclusión práctica
#@markdown Escalar los datos no solo ayuda a que el modelo **converja más rápido**,
#@markdown sino que también permite **comparar correctamente** los coeficientes y entender
#@markdown **qué variables tienen más peso en la predicción.**



In [ ]:
#@title 🔮 Predicción del radio medio (10 casos de prueba) — Comparación con/sin escalado
#@markdown En esta celda comparamos las predicciones de ambos modelos:
#@markdown - **Con escalado:** `pipe_reg_scaled` (usa StandardScaler internamente)
#@markdown - **Sin escalado:** `pipe_reg_noscale` (usa los datos originales sin normalizar)
#@markdown
#@markdown La variable objetivo (`y_reg`) es **el radio medio del tumor** (`mean radius`).

# Seleccionamos los primeros 10 casos del conjunto de prueba
Xr_test_sample = Xr_test[:10]
yr_test_sample = yr_test[:10]

# Realizamos las predicciones con ambos modelos
y_pred_scaled = pipe_reg_scaled.predict(Xr_test_sample)
y_pred_noscale = pipe_reg_noscale.predict(Xr_test_sample)

# Creamos un DataFrame comparativo
import pandas as pd
resultados = pd.DataFrame({
    "Radio real (y_test)": yr_test_sample.ravel(),
    "Pred. modelo escalado": y_pred_scaled.ravel(),
    "Pred. modelo sin escalar": y_pred_noscale.ravel()
})

print("📏 Predicción del radio medio (primeros 10 casos):\n")
display(resultados.style.background_gradient(cmap="Blues", axis=0))

#@markdown # ⚖️ ¿Por qué las predicciones son iguales con y sin escalado?
#@markdown
#@markdown En la **regresión lineal clásica**, el modelo se ajusta resolviendo la ecuación normal:
#@markdown
#@markdown $$\mathbf{w}_b = \big(\mathbf{X}_b^\top \mathbf{X}_b\big)^{-1}\mathbf{X}_b^\top \mathbf{y}$$
#@markdown
#@markdown Este método **no depende de la escala de las variables** para obtener las mismas predicciones finales.
#@markdown
#@markdown ---
#@markdown
#@markdown ## 🧩 Intuición: invarianza por reescalado
#@markdown
#@markdown Supongamos que escalamos la característica $j$ dividiéndola entre una constante $c$: $\tilde{x}_j = x_j / c$.
#@markdown
#@markdown El modelo ajusta automáticamente el coeficiente correspondiente multiplicándolo por $c$: $\tilde{w}_j = c \cdot w_j$.
#@markdown
#@markdown Así, el producto $w_j \cdot x_j = \tilde{w}_j \cdot \tilde{x}_j$ (la contribución a la predicción) **permanece constante**.
#@markdown
#@markdown **Por eso:**
#@markdown
#@markdown - Los **coeficientes** $\mathbf{w}$ cambian de magnitud cuando escalas,
#@markdown - Pero las **predicciones** $\hat{\mathbf{y}} = \mathbf{X}_b\mathbf{w}_b$ **no cambian**.
#@markdown
#@markdown ---
#@markdown
#@markdown ## 💡 Diferencia entre métodos analíticos e iterativos
#@markdown
#@markdown | Método | Tipo | ¿El escalado afecta predicciones? | ¿El escalado afecta entrenamiento? |
#@markdown |:-------|:-----|:----------------------------------|:-----------------------------------|
#@markdown | `LinearRegression()` | Analítico (ecuación normal) | ❌ No | ⚠️ Mejora interpretabilidad |
#@markdown | Gradiente Descendente | Iterativo | ❌ No (si converge) | ✅ **Sí** (velocidad, estabilidad) |
#@markdown | `Ridge`, `Lasso` | Iterativo + regularización | ✅ **Sí** | ✅ **Sí** (crítico para penalización justa) |
#@markdown | `LogisticRegression`, `SVM` | Iterativo + regularización | ✅ **Sí** | ✅ **Sí** (esencial para convergencia) |
#@markdown
#@markdown ---
#@markdown
#@markdown ## 🎯 Cuándo es crucial escalar
#@markdown
#@markdown 1. **Métodos basados en gradiente descendente**: Sin escalado, la convergencia es lenta o diverge (como viste en las gráficas anteriores).
#@markdown
#@markdown 2. **Modelos con regularización** (Ridge, Lasso, Elastic Net): La penalización $\lambda \sum w_j^2$ o $\lambda \sum |w_j|$ no es justa si las variables tienen escalas diferentes. Una variable grande tendría un peso pequeño solo por su escala, no por su importancia.
#@markdown
#@markdown 3. **Algoritmos basados en distancias** (KNN, K-Means, SVM con kernel RBF): Las variables con mayor magnitud dominan el cálculo de distancias.
#@markdown
#@markdown ---
#@markdown
#@markdown ## 📌 Conclusión
#@markdown
#@markdown En regresión lineal clásica con ecuación normal (`LinearRegression()`), el escalado:
#@markdown
#@markdown - **No modifica** las predicciones finales (invarianza matemática)
#@markdown - **Mejora** la interpretabilidad de coeficientes (todos en escala comparable)
#@markdown - **No es necesario** para la correctitud del modelo
#@markdown
#@markdown Sin embargo, en métodos iterativos o regularizados, el escalado es **esencial** para:
#@markdown
#@markdown - Garantizar convergencia rápida y estable
#@markdown - Aplicar regularización de forma justa entre variables
#@markdown - Evitar que variables de gran magnitud dominen el aprendizaje
#@markdown
#@markdown ➜ **Regla práctica**: Siempre escala cuando uses gradiente descendente, regularización, o distancias.



## clasificación

In [ ]:
#@title 🧮 Regresión Logística Binaria — Matemática y Gradiente (desde cero)
#@markdown **Hipótesis (función sigmoide):**
#@markdown $$\hat{p}_i = \sigma(z_i) = \frac{1}{1 + e^{-z_i}}, \quad z_i = \mathbf{w}^\top \mathbf{x}_i + b$$
#@markdown
#@markdown Forma matricial: $$\hat{\mathbf{p}} = \sigma(\mathbf{X}_b\mathbf{w}_b)$$
#@markdown
#@markdown **Función de pérdida (log-loss / entropía cruzada binaria):**
#@markdown $$\mathcal{L}(\mathbf{w}, b) = -\frac{1}{N}\sum_{i=1}^N \Big[y_i \log \hat{p}_i + (1-y_i)\log(1-\hat{p}_i)\Big]$$
#@markdown
#@markdown **Gradiente:**
#@markdown $$\frac{\partial \mathcal{L}}{\partial \mathbf{w}_b} = \frac{1}{N} \mathbf{X}_b^\top (\hat{\mathbf{p}} - \mathbf{y})$$
#@markdown
#@markdown **Actualización:**
#@markdown $$\mathbf{w}_b \leftarrow \mathbf{w}_b - \eta \frac{\partial \mathcal{L}}{\partial \mathbf{w}_b}$$

def sigmoid(z):
    """Función sigmoide: σ(z) = 1 / (1 + e^(-z))"""
    return 1.0 / (1.0 + np.exp(-np.clip(z, -500, 500)))  # clip para estabilidad

use_scaling_cls = True  #@param {type:"boolean"}
log_alpha = 0.1         #@param {type:"number"}
log_iters = 1000        #@param {type:"integer"}

# Copias para no alterar datos originales
Xct, Xce = deepcopy(Xc_train), deepcopy(Xc_test)
yct = deepcopy(yc_train).values.reshape(-1, 1)
yce = deepcopy(yc_test).values.reshape(-1, 1)

scaler_cls = None
if use_scaling_cls:
    scaler_cls = StandardScaler().fit(Xct)
    Xct = scaler_cls.transform(Xct)
    Xce = scaler_cls.transform(Xce)

# Preparar matrices con bias
Xct_b = add_bias(Xct)  # (N, d+1)
N, d_plus_1 = Xct_b.shape
w_b_log = np.zeros((d_plus_1, 1))

loss_hist_log = []
for it in range(log_iters):
    # Forward: calcular probabilidades
    z = Xct_b @ w_b_log
    p_hat = sigmoid(z)

    # Gradiente: (1/N) * X_b^T * (p_hat - y)
    grad = (Xct_b.T @ (p_hat - yct)) / N

    # Actualización de pesos
    w_b_log -= log_alpha * grad

    # Pérdida (log-loss con estabilidad numérica)
    eps = 1e-15
    p_hat_clipped = np.clip(p_hat, eps, 1 - eps)
    L = -np.mean(yct * np.log(p_hat_clipped) + (1 - yct) * np.log(1 - p_hat_clipped))
    loss_hist_log.append(L)

print("𝐰_b aprendido (primeros 5):", w_b_log.ravel()[:5])
print("Pérdida final (log-loss):", loss_hist_log[-1])

# Curva de pérdida
plt.figure(figsize=(8, 5))
plt.plot(loss_hist_log, linewidth=2)
plt.xlabel("Iteración", fontsize=12)
plt.ylabel("𝓛(𝐰, b) = log-loss", fontsize=12)
plt.title("Regresión Logística — Descenso de Gradiente (escala=" + str(use_scaling_cls) + ")", fontsize=13)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
#@markdown # 📚 Regresión Logística Binaria — Guía completa para entender cada fórmula
#@markdown
#@markdown ---
#@markdown
#@markdown ## 🎯 ¿Qué queremos hacer?
#@markdown
#@markdown Queremos **predecir si algo pertenece a una clase (1) o a otra (0)**. Por ejemplo:
#@markdown
#@markdown - ¿Este tumor es **benigno (1)** o **maligno (0)**?
#@markdown - ¿Este correo es spam (1) o no spam (0)?
#@markdown - ¿Este cliente comprará (1) o no (0)?
#@markdown
#@markdown En lugar de predecir directamente 0 o 1, predecimos la **probabilidad** de que sea clase 1.
#@markdown
#@markdown **Importante para nuestro dataset:**
#@markdown - **Clase 1 (positiva)** = Tumor **benigno** (no canceroso) ✅
#@markdown - **Clase 0 (negativa)** = Tumor **maligno** (canceroso) ⚠️
#@markdown
#@markdown ---
#@markdown
#@markdown ## 🧩 Componente 1: La combinación lineal (z)
#@markdown
#@markdown $$z_i = \mathbf{w}^\top \mathbf{x}_i + b = w_1x_{i1} + w_2x_{i2} + \cdots + w_dx_{id} + b$$
#@markdown
#@markdown **¿Qué es?** Un número que combina todas las características de la muestra $i$.
#@markdown
#@markdown **Componentes:**
#@markdown - $\mathbf{x}_i = [x_{i1}, x_{i2}, \ldots, x_{id}]$: Las características del paciente/caso $i$ (tamaño del tumor, textura, simetría, etc.)
#@markdown - $\mathbf{w} = [w_1, w_2, \ldots, w_d]$: Los **pesos** que indican qué tan importante es cada característica
#@markdown - $b$: El **sesgo** o intercepto (ajuste base, independiente de las características)
#@markdown
#@markdown **Intuición:**
#@markdown - Si $z_i$ es **grande y positivo** → el modelo cree que es más probable clase 1 (benigno)
#@markdown - Si $z_i$ es **grande y negativo** → el modelo cree que es más probable clase 0 (maligno)
#@markdown - Si $z_i \approx 0$ → el modelo está **indeciso**
#@markdown
#@markdown **Ejemplo numérico:**
#@markdown ```
#@markdown Paciente con: mean_radius=15.5, mean_texture=20.3, mean_area=750
#@markdown Pesos: w = [0.8, -0.3, 0.5], b = -2.0
#@markdown z = 0.8×15.5 + (-0.3)×20.3 + 0.5×750 + (-2.0)
#@markdown z = 12.4 - 6.09 + 375 - 2.0 = 379.31 (muy positivo → probablemente benigno)
#@markdown ```
#@markdown
#@markdown ---
#@markdown
#@markdown ## 🌊 Componente 2: La función sigmoide (σ)
#@markdown
#@markdown $$\hat{p}_i = \sigma(z_i) = \frac{1}{1 + e^{-z_i}}$$
#@markdown
#@markdown **¿Qué hace?** Convierte cualquier número $z$ (que puede ir de $-\infty$ a $+\infty$) en una **probabilidad** entre 0 y 1.
#@markdown
#@markdown **Propiedades importantes:**
#@markdown - Si $z = 0$ → $\sigma(0) = 0.5$ (50% de probabilidad, indecisión total)
#@markdown - Si $z \to +\infty$ → $\sigma(z) \to 1$ (casi 100% seguro de clase 1: benigno)
#@markdown - Si $z \to -\infty$ → $\sigma(z) \to 0$ (casi 100% seguro de clase 0: maligno)
#@markdown - Tiene forma de "S" (sigmoide = parecido a sigma)
#@markdown
#@markdown **¿Por qué esta función?**
#@markdown 1. **Acota** la salida entre 0 y 1 (interpretable como probabilidad)
#@markdown 2. Es **suave y derivable** (necesario para calcular gradientes)
#@markdown 3. Tiene propiedades matemáticas convenientes: $\sigma'(z) = \sigma(z)(1-\sigma(z))$
#@markdown
#@markdown **Ejemplo continuando el anterior:**
#@markdown ```
#@markdown z = 379.31
#@markdown p_hat = 1 / (1 + e^(-379.31)) ≈ 1.0
#@markdown Interpretación: ~100% de probabilidad de clase 1 (benigno) ✅
#@markdown ```
#@markdown
#@markdown **Valores típicos:**
#@markdown | z | σ(z) | Interpretación |
#@markdown |:---:|:------:|:---------------|
#@markdown | -5 | 0.007 | Casi seguro maligno ⚠️ |
#@markdown | -2 | 0.119 | Probablemente maligno ⚠️ |
#@markdown | 0 | 0.500 | Indeciso |
#@markdown | 2 | 0.881 | Probablemente benigno ✅ |
#@markdown | 5 | 0.993 | Casi seguro benigno ✅ |
#@markdown
#@markdown ---
#@markdown
#@markdown ## 📉 Componente 3: La función de pérdida (log-loss)
#@markdown
#@markdown $$\mathcal{L}(\mathbf{w}, b) = -\frac{1}{N}\sum_{i=1}^N \Big[y_i \log(\hat{p}_i) + (1-y_i)\log(1-\hat{p}_i)\Big]$$
#@markdown
#@markdown **¿Qué mide?** Qué tan **equivocado** está el modelo en sus predicciones de probabilidad.
#@markdown
#@markdown **Desglosando la fórmula:**
#@markdown
#@markdown Para cada muestra $i$, hay dos casos:
#@markdown
#@markdown **Caso 1: Si la etiqueta real es $y_i = 1$ (tumor benigno ✅)**
#@markdown - El término $(1-y_i)\log(1-\hat{p}_i)$ se anula (porque $1-y_i=0$)
#@markdown - Solo queda: $-y_i \log(\hat{p}_i) = -\log(\hat{p}_i)$
#@markdown - Si predijimos $\hat{p}_i = 0.9$ → pérdida = $-\log(0.9) = 0.105$ (pequeña, ¡bien!)
#@markdown - Si predijimos $\hat{p}_i = 0.1$ → pérdida = $-\log(0.1) = 2.303$ (grande, ¡mal!)
#@markdown - Si predijimos $\hat{p}_i = 0.01$ → pérdida = $-\log(0.01) = 4.605$ (enorme, ¡muy mal!)
#@markdown
#@markdown **Caso 2: Si la etiqueta real es $y_i = 0$ (tumor maligno ⚠️)**
#@markdown - El término $y_i \log(\hat{p}_i)$ se anula (porque $y_i=0$)
#@markdown - Solo queda: $-(1-y_i)\log(1-\hat{p}_i) = -\log(1-\hat{p}_i)$
#@markdown - Si predijimos $\hat{p}_i = 0.1$ → pérdida = $-\log(0.9) = 0.105$ (pequeña, ¡bien!)
#@markdown - Si predijimos $\hat{p}_i = 0.9$ → pérdida = $-\log(0.1) = 2.303$ (grande, ¡mal!)
#@markdown
#@markdown **¿Por qué logaritmo?**
#@markdown - Penaliza **mucho más** las predicciones muy confiadas pero equivocadas
#@markdown - Si predices 99% benigno pero era maligno → ¡pérdida enorme! (error médico grave)
#@markdown - Tiene buenas propiedades matemáticas (convexa, derivable)
#@markdown
#@markdown **Ejemplo numérico:**
#@markdown ```
#@markdown Muestra 1: y=1 (benigno), p_hat=0.9 → pérdida = -log(0.9) = 0.105
#@markdown Muestra 2: y=0 (maligno), p_hat=0.2 → pérdida = -log(0.8) = 0.223
#@markdown Muestra 3: y=1 (benigno), p_hat=0.3 → pérdida = -log(0.3) = 1.204 (¡mal!)
#@markdown Pérdida promedio = (0.105 + 0.223 + 1.204) / 3 = 0.511
#@markdown ```
#@markdown
#@markdown ---
#@markdown
#@markdown ## 📐 Componente 4: El gradiente
#@markdown
#@markdown $$\frac{\partial \mathcal{L}}{\partial \mathbf{w}_b} = \frac{1}{N} \mathbf{X}_b^\top (\hat{\mathbf{p}} - \mathbf{y})$$
#@markdown
#@markdown **¿Qué es?** La **dirección** en la que debemos mover los pesos para **reducir** la pérdida.
#@markdown
#@markdown **Desglosando:**
#@markdown - $(\hat{\mathbf{p}} - \mathbf{y})$: El **error** de predicción para cada muestra
#@markdown   - Si $\hat{p}_i = 0.9$ y $y_i = 1$ (benigno) → error = $-0.1$ (subestimamos un poco)
#@markdown   - Si $\hat{p}_i = 0.3$ y $y_i = 1$ (benigno) → error = $-0.7$ (subestimamos mucho, ¡peligroso!)
#@markdown   - Si $\hat{p}_i = 0.8$ y $y_i = 0$ (maligno) → error = $+0.8$ (sobreestimamos mucho, ¡muy peligroso!)
#@markdown
#@markdown - $\mathbf{X}_b^\top (\hat{\mathbf{p}} - \mathbf{y})$: Multiplica cada error por las características correspondientes
#@markdown   - Si una característica es grande cuando nos equivocamos → su gradiente será grande
#@markdown   - Si una característica no está relacionada con los errores → su gradiente será pequeño
#@markdown
#@markdown - $\frac{1}{N}$: Promedia sobre todas las muestras
#@markdown
#@markdown **Intuición clave:**
#@markdown
#@markdown Para cada peso $w_j$:
#@markdown - Si el modelo **sobreestima** ($\hat{p} > y$, predice más benigno de lo que es) cuando $x_j$ es grande → **reduce** $w_j$
#@markdown - Si el modelo **subestima** ($\hat{p} < y$, predice más maligno de lo que es) cuando $x_j$ es grande → **aumenta** $w_j$
#@markdown
#@markdown **Ejemplo numérico (peso para "mean_radius"):**
#@markdown ```
#@markdown Muestra 1: x_radius=15, y=1 (benigno), p_hat=0.7 → error=-0.3, contribución=15×(-0.3)=-4.5
#@markdown Muestra 2: x_radius=10, y=0 (maligno), p_hat=0.4 → error=+0.4, contribución=10×(+0.4)=+4.0
#@markdown Muestra 3: x_radius=20, y=1 (benigno), p_hat=0.5 → error=-0.5, contribución=20×(-0.5)=-10.0
#@markdown
#@markdown Gradiente = (-4.5 + 4.0 - 10.0) / 3 = -3.5
#@markdown → Debemos AUMENTAR el peso de "radius" (porque gradiente negativo)
#@markdown → Tumores más grandes tienden a ser benignos en estos casos
#@markdown ```
#@markdown
#@markdown ---
#@markdown
#@markdown ## 🔄 Componente 5: La actualización (Gradiente Descendente)
#@markdown
#@markdown $$\mathbf{w}_b \leftarrow \mathbf{w}_b - \eta \frac{\partial \mathcal{L}}{\partial \mathbf{w}_b}$$
#@markdown
#@markdown **¿Qué hace?** Ajusta los pesos en la **dirección opuesta** al gradiente para **minimizar** la pérdida.
#@markdown
#@markdown **Componentes:**
#@markdown - $\mathbf{w}_b$: Los pesos actuales (incluyendo el sesgo)
#@markdown - $\eta$ (eta): La **tasa de aprendizaje** (qué tan grande es cada paso)
#@markdown   - Muy grande (ej. η=1.0) → pasos grandes, puede **diverger** o **oscilar**
#@markdown   - Muy pequeña (ej. η=0.0001) → pasos pequeños, converge **lento** pero seguro
#@markdown   - Típico: η ∈ [0.01, 0.1]
#@markdown
#@markdown **¿Por qué el signo negativo?**
#@markdown - El gradiente apunta hacia donde la pérdida **aumenta**
#@markdown - Queremos ir hacia donde la pérdida **disminuye**
#@markdown - Por eso restamos: vamos en dirección opuesta
#@markdown
#@markdown **Analogía del montañista:**
#@markdown
#@markdown Imagina que estás en una montaña con niebla (no ves el valle):
#@markdown - El **gradiente** te dice hacia dónde sube más rápido la montaña
#@markdown - Tú quieres **bajar**, así que vas en dirección contraria
#@markdown - La **tasa de aprendizaje** es el tamaño de cada paso que das
#@markdown - Después de muchos pasos, llegas al fondo del valle (mínimo de pérdida)
#@markdown
#@markdown **Ejemplo de actualización:**
#@markdown ```
#@markdown Peso actual: w_radius = 1.5
#@markdown Gradiente: ∂L/∂w_radius = -3.5
#@markdown Tasa de aprendizaje: η = 0.1
#@markdown
#@markdown Nuevo peso = 1.5 - 0.1×(-3.5) = 1.5 + 0.35 = 1.85
#@markdown
#@markdown (Aumentamos el peso porque el gradiente era negativo)
#@markdown ```
#@markdown
#@markdown ---
#@markdown
#@markdown ## 🔁 El proceso completo (algoritmo paso a paso)
#@markdown
#@markdown 1. **Inicializar**: Empezar con pesos aleatorios o en cero: $\mathbf{w}_b = \mathbf{0}$
#@markdown
#@markdown 2. **Repetir** muchas veces (iteraciones):
#@markdown
#@markdown    a. **Forward pass** (predicción):
#@markdown       - Calcular $z_i = \mathbf{w}^\top \mathbf{x}_i + b$ para cada muestra
#@markdown       - Calcular $\hat{p}_i = \sigma(z_i) = \frac{1}{1+e^{-z_i}}$
#@markdown
#@markdown    b. **Calcular pérdida**:
#@markdown       - $\mathcal{L} = -\frac{1}{N}\sum [y_i \log(\hat{p}_i) + (1-y_i)\log(1-\hat{p}_i)]$
#@markdown
#@markdown    c. **Backward pass** (calcular gradiente):
#@markdown       - $\frac{\partial \mathcal{L}}{\partial \mathbf{w}_b} = \frac{1}{N} \mathbf{X}_b^\top (\hat{\mathbf{p}} - \mathbf{y})$
#@markdown
#@markdown    d. **Actualizar pesos**:
#@markdown       - $\mathbf{w}_b \leftarrow \mathbf{w}_b - \eta \frac{\partial \mathcal{L}}{\partial \mathbf{w}_b}$
#@markdown
#@markdown 3. **Terminar** cuando:
#@markdown    - Llegaste al número máximo de iteraciones, O
#@markdown    - La pérdida dejó de disminuir significativamente
#@markdown
#@markdown ---
#@markdown
#@markdown ## 🎯 Haciendo predicciones (después del entrenamiento)
#@markdown
#@markdown Una vez que tenemos los pesos óptimos $\mathbf{w}_b^*$:
#@markdown
#@markdown 1. **Calcular la probabilidad** para un nuevo caso:
#@markdown    $$\hat{p} = \sigma(\mathbf{w}^{*\top} \mathbf{x}_{\text{nuevo}} + b^*)$$
#@markdown
#@markdown 2. **Decidir la clase** usando un umbral (típicamente 0.5):
#@markdown    - Si $\hat{p} \geq 0.5$ → predecir clase 1 (benigno ✅)
#@markdown    - Si $\hat{p} < 0.5$ → predecir clase 0 (maligno ⚠️)
#@markdown
#@markdown **Ejemplo final completo:**
#@markdown ```
#@markdown Después del entrenamiento obtuvimos:
#@markdown w* = [0.8, -0.5, 1.2], b* = -2.0
#@markdown
#@markdown Nuevo paciente: mean_radius=12, mean_texture=18, mean_area=500
#@markdown
#@markdown z = 0.8×12 + (-0.5)×18 + 1.2×500 + (-2.0)
#@markdown z = 9.6 - 9 + 600 - 2 = 598.6
#@markdown
#@markdown p_hat = 1 / (1 + e^(-598.6)) ≈ 1.0
#@markdown
#@markdown Como p_hat > 0.5 → Predecimos clase 1 (tumor BENIGNO ✅)
#@markdown Con ~100% de confianza - ¡Buenas noticias para el paciente!
#@markdown ```
#@markdown
#@markdown ---
#@markdown
#@markdown ## 💡 Consejos prácticos para aplicar la fórmula
#@markdown
#@markdown 1. **Siempre escala tus características** antes de entrenar (usa `StandardScaler`)
#@markdown    - Sin escalado, el gradiente descendente puede diverger
#@markdown
#@markdown 2. **Empieza con una tasa de aprendizaje moderada** (η = 0.01 o 0.1)
#@markdown    - Si la pérdida oscila → reduce η
#@markdown    - Si converge muy lento → aumenta η
#@markdown
#@markdown 3. **Vigila la pérdida en cada iteración**
#@markdown    - Debe **disminuir monotónicamente** (siempre bajar)
#@markdown    - Si sube → algo está mal (η muy grande, error en código, etc.)
#@markdown
#@markdown 4. **Usa estabilidad numérica**
#@markdown    - Agrega un epsilon pequeño ($10^{-15}$) al calcular logaritmos
#@markdown    - Recorta (clip) valores de z muy extremos en sigmoid
#@markdown
#@markdown 5. **Interpreta los pesos aprendidos**
#@markdown    - Peso positivo grande → esa característica aumenta P(benigno)
#@markdown    - Peso negativo grande → esa característica aumenta P(maligno)
#@markdown    - Peso cercano a 0 → esa característica no es muy relevante
#@markdown
#@markdown 6. **Considera el contexto médico**
#@markdown    - En diagnóstico médico, un **falso negativo** (decir benigno cuando es maligno) es MÁS peligroso
#@markdown    - Puedes ajustar el umbral de decisión (usar 0.4 en vez de 0.5 para ser más conservador)
#@markdown
#@markdown ---
#@markdown
#@markdown ## ✅ Resumen ejecutivo
#@markdown
#@markdown **El ciclo completo de Regresión Logística:**
#@markdown
#@markdown ### Componente 1: Combinación lineal
#@markdown - **Fórmula:** $z = \mathbf{w}^\top\mathbf{x} + b$
#@markdown - **Qué hace:** Combina features
#@markdown - **Intuición:** "Puntaje" del tumor
#@markdown
#@markdown ### Componente 2: Sigmoide
#@markdown - **Fórmula:** $\hat{p} = \frac{1}{1+e^{-z}}$
#@markdown - **Qué hace:** Convierte z a probabilidad
#@markdown - **Intuición:** P(benigno) entre 0 y 1
#@markdown
#@markdown ### Componente 3: Pérdida (log-loss)
#@markdown - **Fórmula:** $\mathcal{L} = -\frac{1}{N}\sum [y\log\hat{p} + (1-y)\log(1-\hat{p})]$
#@markdown - **Qué hace:** Mide error total
#@markdown - **Intuición:** Penaliza diagnósticos incorrectos
#@markdown
#@markdown ### Componente 4: Gradiente
#@markdown - **Fórmula:** $\nabla = \frac{1}{N}\mathbf{X}^\top(\hat{\mathbf{p}}-\mathbf{y})$
#@markdown - **Qué hace:** Dirección de mejora
#@markdown - **Intuición:** Cómo ajustar pesos
#@markdown
#@markdown ### Componente 5: Actualización
#@markdown - **Fórmula:** $\mathbf{w} \leftarrow \mathbf{w} - \eta\nabla$
#@markdown - **Qué hace:** Mejora los pesos
#@markdown - **Intuición:** Da pasos hacia diagnóstico óptimo
#@markdown
#@markdown ---
#@markdown
#@markdown **🔄 El proceso completo:** Predecir → Medir error → Calcular dirección → Ajustar pesos → Repetir
#@markdown
#@markdown **📌 Recuerda:** Clase 1 = Benigno ✅ (lo que queremos predecir), Clase 0 = Maligno ⚠️


In [ ]:
#@title 🗺️ Visualización: Frontera de decisión en 2D (PCA para graficar)
#@markdown Para **graficar** la frontera de decisión, usamos PCA a 2D **solo para la visual**.
#@markdown Entrenamos un **modelo logístico desde cero** en ese 2D para dibujar la frontera.

# Prepara datos para visual
Xc_train_plot = deepcopy(Xc_train)
Xc_test_plot = deepcopy(Xc_test)
y_train_plot = deepcopy(yc_train).values.reshape(-1,1)  # Agregado .values

sc_viz = StandardScaler().fit(Xc_train_plot)  # Escalar para PCA visual (mejor separabilidad)
Xc_train_plot_s = sc_viz.transform(Xc_train_plot)
Xc_test_plot_s  = sc_viz.transform(Xc_test_plot)

pca2 = PCA(n_components=2).fit(Xc_train_plot_s)
Ztr = pca2.transform(Xc_train_plot_s)
Zte = pca2.transform(Xc_test_plot_s)

# Entrenamos logística en Z (2D) para graficar frontera
Ztr_b = add_bias(Ztr)
theta_v = np.zeros((Ztr_b.shape[1], 1))
alpha_v = 0.2
iters_v = 800

for _ in range(iters_v):
    p = sigmoid(Ztr_b @ theta_v)
    grad = (Ztr_b.T @ (p - y_train_plot)) / Ztr_b.shape[0]
    theta_v -= alpha_v * grad

# Malla y frontera
x_min, x_max = Ztr[:,0].min()-1, Ztr[:,0].max()+1
y_min, y_max = Ztr[:,1].min()-1, Ztr[:,1].max()+1
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 250),
                     np.linspace(y_min, y_max, 250))
grid = add_bias(np.c_[xx.ravel(), yy.ravel()])
pp = sigmoid(grid @ theta_v).reshape(xx.shape)

plt.figure(figsize=(10, 7))
plt.contourf(xx, yy, (pp>0.5).astype(int), alpha=0.35, cmap=cmap_light)
plt.contour(xx, yy, pp, levels=[0.5], linewidths=2, linestyles="--", colors="k")
plt.scatter(Ztr[:,0], Ztr[:,1], c=yc_train, cmap=cmap_bold, edgecolor="k", s=25, alpha=0.9)
plt.title("Frontera de decisión (PCA 2D para visualización)")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.colorbar(label="Clase (0=Maligno, 1=Benigno)")
plt.tight_layout()
plt.show()

#@markdown # 🗺️ Interpretación de la Frontera de Decisión (Visualización 2D con PCA)
#@markdown
#@markdown ---
#@markdown
#@markdown ## 📊 ¿Qué estamos viendo?
#@markdown
#@markdown Esta visualización muestra cómo el modelo de **Regresión Logística** separa los tumores benignos de los malignos en un espacio de 2 dimensiones.
#@markdown
#@markdown ---
#@markdown
#@markdown ## 🎨 Elementos de la gráfica
#@markdown
#@markdown ### 1. Los puntos (datos reales)
#@markdown - **Puntos azules 🔵**: Tumores **benignos** (clase 1) - No cancerosos ✅
#@markdown - **Puntos rojos 🔴**: Tumores **malignos** (clase 0) - Cancerosos ⚠️
#@markdown - Cada punto representa un paciente del conjunto de entrenamiento
#@markdown
#@markdown ### 2. Las regiones de color (predicciones del modelo)
#@markdown - **Región azul clara**: Zona donde el modelo predice **clase 1 (benigno)**
#@markdown   - Cualquier nuevo tumor en esta región → el modelo diría "probablemente benigno"
#@markdown - **Región rosa/roja clara**: Zona donde el modelo predice **clase 0 (maligno)**
#@markdown   - Cualquier nuevo tumor en esta región → el modelo diría "probablemente maligno"
#@markdown
#@markdown ### 3. La línea discontinua negra (frontera de decisión)
#@markdown - Esta es la **frontera de decisión**: la línea donde $\hat{p} = 0.5$
#@markdown - **A la izquierda/abajo de la línea**: $\hat{p} < 0.5$ → el modelo predice maligno (0)
#@markdown - **A la derecha/arriba de la línea**: $\hat{p} > 0.5$ → el modelo predice benigno (1)
#@markdown - Es una línea **recta** porque la regresión logística es un modelo **lineal**
#@markdown
#@markdown ### 4. Los ejes (PC1 y PC2)
#@markdown - **PC1** (eje horizontal): Primera componente principal
#@markdown - **PC2** (eje vertical): Segunda componente principal
#@markdown - Estas NO son las características originales, sino **combinaciones lineales** de todas las características proyectadas a 2D para poder visualizar
#@markdown
#@markdown ---
#@markdown
#@markdown ## 🔍 ¿Qué nos dice esta gráfica?
#@markdown
#@markdown ### ✅ Señales positivas (modelo funciona bien):
#@markdown
#@markdown 1. **Separación clara**: Hay una buena separación entre la mayoría de puntos azules y rojos
#@markdown    - Los tumores benignos (azul) tienden a estar en la zona superior derecha
#@markdown    - Los tumores malignos (rojo) tienden a estar en la zona inferior izquierda
#@markdown
#@markdown 2. **Agrupamiento**: Los puntos de cada clase forman grupos relativamente compactos
#@markdown
#@markdown 3. **Frontera sensata**: La línea de decisión pasa por una zona razonable, separando ambos grupos
#@markdown
#@markdown ### ⚠️ Observaciones importantes:
#@markdown
#@markdown 1. **Solapamiento**: Hay una zona central donde ambos colores se mezclan
#@markdown    - Esto es **normal** en datos reales del mundo médico
#@markdown    - Indica que algunos casos son difíciles de clasificar (ambiguos)
#@markdown
#@markdown 2. **Puntos mal clasificados**:
#@markdown    - **Puntos rojos en zona azul**: Tumores malignos que el modelo clasificaría como benignos (falsos negativos) 🚨
#@markdown    - **Puntos azules en zona roja**: Tumores benignos que el modelo clasificaría como malignos (falsos positivos)
#@markdown
#@markdown 3. **Frontera lineal**: La línea recta puede no capturar toda la complejidad
#@markdown    - Para fronteras más flexibles, necesitarías modelos no lineales (SVM con kernel, árboles, etc.)
#@markdown
#@markdown ---
#@markdown
#@markdown ## 🧮 La matemática detrás de la línea
#@markdown
#@markdown La frontera de decisión se encuentra donde:
#@markdown
#@markdown $$\hat{p} = 0.5$$
#@markdown
#@markdown Que ocurre cuando:
#@markdown
#@markdown $$\sigma(z) = 0.5 \quad \Rightarrow \quad z = 0$$
#@markdown
#@markdown Por lo tanto:
#@markdown
#@markdown $$\mathbf{w}^\top \mathbf{x} + b = 0$$
#@markdown
#@markdown Para nuestra visualización 2D (usando PC1 y PC2):
#@markdown
#@markdown $$w_1 \cdot \text{PC1} + w_2 \cdot \text{PC2} + b = 0$$
#@markdown
#@markdown Despejando PC2:
#@markdown
#@markdown $$\text{PC2} = -\frac{w_1}{w_2} \cdot \text{PC1} - \frac{b}{w_2}$$
#@markdown
#@markdown Esta es la ecuación de una **línea recta** (pendiente $-w_1/w_2$, intercepto $-b/w_2$)
#@markdown
#@markdown ---
#@markdown
#@markdown ## 💡 Interpretación práctica
#@markdown
#@markdown **Si un nuevo paciente llega:**
#@markdown
#@markdown 1. Tomamos sus características (tamaño del tumor, textura, etc.)
#@markdown 2. Las proyectamos en este espacio 2D usando PCA
#@markdown 3. Vemos de qué lado de la línea cae:
#@markdown    - **Arriba-derecha de la línea** → Alta probabilidad de benigno ✅
#@markdown    - **Abajo-izquierda de la línea** → Alta probabilidad de maligno ⚠️
#@markdown    - **Cerca de la línea** → Caso incierto, requiere análisis adicional
#@markdown
#@markdown **Ejemplo:**
#@markdown ```
#@markdown Nuevo paciente con coordenadas (PC1=5, PC2=4)
#@markdown → Está en la zona azul clara, por encima de la línea discontinua
#@markdown → El modelo predice: BENIGNO con alta confianza
#@markdown ```
#@markdown
#@markdown ---
#@markdown
#@markdown ## ⚠️ Nota crítica sobre esta visualización
#@markdown
#@markdown 🔴 **IMPORTANTE**: Esta gráfica es solo para **visualización educativa**
#@markdown
#@markdown - El modelo **real** trabaja con **todas las características originales** (30 dimensiones en el dataset de cáncer)
#@markdown - Esta visualización **proyecta a 2D** usando PCA, lo que inevitablemente **pierde información**
#@markdown - La frontera de decisión mostrada aquí es de un modelo entrenado **solo en estas 2 componentes principales**
#@markdown - El modelo completo (con todas las features) sería **mucho más preciso** que lo que esta gráfica sugiere
#@markdown
#@markdown **Analogía**: Es como ver una escultura 3D desde un solo ángulo. Obtienes una idea general, pero pierdes mucha información espacial.
#@markdown
#@markdown ---
#@markdown
#@markdown ## 🎯 Conclusiones clave
#@markdown
#@markdown 1. **La regresión logística crea una frontera de decisión lineal** (línea recta en 2D, hiperplano en dimensiones superiores)
#@markdown
#@markdown 2. **No todos los puntos están perfectamente separados** - esto es normal y esperado en datos médicos reales
#@markdown
#@markdown 3. **La ubicación de un punto respecto a la línea** determina su clasificación predicha
#@markdown
#@markdown 4. **La distancia a la línea** indica la confianza del modelo:
#@markdown    - Lejos de la línea → alta confianza
#@markdown    - Cerca de la línea → baja confianza (zona gris)
#@markdown
#@markdown 5. **Esta visualización simplifica el problema real** - el modelo verdadero opera en un espacio de muchas más dimensiones
#@markdown
#@markdown ---
#@markdown
#@markdown ## 🔬 Para profundizar
#@markdown
#@markdown **Preguntas para reflexionar:**
#@markdown
#@markdown 1. ¿Qué tipo de errores son más peligrosos en este contexto médico?
#@markdown    - Respuesta: Los **falsos negativos** (decir benigno cuando es maligno) son más peligrosos
#@markdown
#@markdown 2. ¿Cómo podrías ajustar el modelo para ser más conservador?
#@markdown    - Respuesta: Cambiar el umbral de decisión de 0.5 a 0.4 o 0.3
#@markdown
#@markdown 3. ¿Por qué la frontera es una línea recta y no una curva?
#@markdown    - Respuesta: Porque la regresión logística es un clasificador lineal (z es una combinación lineal)
#@markdown
#@markdown 4. ¿Cómo se vería la frontera con un modelo más complejo (ej. SVM con kernel RBF)?
#@markdown    - Respuesta: Sería una **curva** que podría adaptarse mejor a la forma de los datos


In [ ]:
#@title ⚙️ Regresión Logística con Scikit-Learn (con/sin escalado)
#@markdown Usamos `liblinear` para estabilidad en datasets medianos/pequeños; puedes cambiar a `lbfgs`.

pipe_log_scaled = Pipeline([
    ("scaler", StandardScaler()),
    ("logreg", LogisticRegression(solver="liblinear"))
])

pipe_log_noscale = Pipeline([
    ("logreg", LogisticRegression(solver="liblinear", max_iter=1000))
])

# Entrenar ambos modelos
pipe_log_scaled.fit(Xc_train, yc_train)
pipe_log_noscale.fit(Xc_train, yc_train)

print("=" * 70)
print("COEFICIENTES APRENDIDOS")
print("=" * 70)
print("Coef (scaled) primeros 5:", pipe_log_scaled["logreg"].coef_.ravel()[:25])
print("Coef (noscale) primeros 5:", pipe_log_noscale["logreg"].coef_.ravel()[:25])
print()

# Hacer predicciones en el conjunto de prueba
y_pred_scaled = pipe_log_scaled.predict(Xc_test)
y_pred_noscale = pipe_log_noscale.predict(Xc_test)

# Obtener probabilidades
y_proba_scaled = pipe_log_scaled.predict_proba(Xc_test)
y_proba_noscale = pipe_log_noscale.predict_proba(Xc_test)

print("=" * 70)
print("PREDICCIONES EN TEST SET")
print("=" * 70)
print(f"Predicciones (scaled) primeras 20: {y_pred_scaled[:20]}")
print(f"Predicciones (noscale) primeras 20: {y_pred_noscale[:20]}")
print(f"Valores reales primeros 20: {yc_test.values[:20]}")
print()

# Ejemplo de predicción para un caso específico
print("=" * 70)
print("EJEMPLO: PREDICCIÓN PARA UN CASO ESPECÍFICO")
print("=" * 70)
caso_ejemplo = Xc_test.iloc[0:1]  # Primer caso del test set
prob_scaled = pipe_log_scaled.predict_proba(caso_ejemplo)[0]
pred_scaled = pipe_log_scaled.predict(caso_ejemplo)[0]
real = yc_test.iloc[0]

print(f"Caso real: {real} ({'Benigno ✅' if real == 1 else 'Maligno ⚠️'})")
print(f"Predicción: {pred_scaled} ({'Benigno ✅' if pred_scaled == 1 else 'Maligno ⚠️'})")
print(f"¿Correcto? {'SÍ ✓' if pred_scaled == real else 'NO ✗'}")
print(f"\nProbabilidades:")
print(f"  P(Maligno | x) = {prob_scaled[0]:.4f} ({prob_scaled[0]*100:.2f}%)")
print(f"  P(Benigno | x) = {prob_scaled[1]:.4f} ({prob_scaled[1]*100:.2f}%)")

#@markdown # 📊 Explicación de Resultados: Regresión Logística con/sin Escalado
#@markdown
#@markdown ---
#@markdown
#@markdown ## 🔢 Interpretación de los Coeficientes
#@markdown
#@markdown ### Coeficientes con escalado (StandardScaler)
#@markdown ```
#@markdown [-2.469, -1.2325, -0.1951, 0.2721, -1.3686]
#@markdown ```
#@markdown
#@markdown ### Coeficientes sin escalado
#@markdown ```
#@markdown [-0.2697, -0.0639, 0.2929, -0.3324, -1.2402]
#@markdown ```
#@markdown
#@markdown ---
#@markdown
#@markdown ## 🤔 ¿Por qué son tan diferentes?
#@markdown
#@markdown ### El efecto de la escala
#@markdown
#@markdown Los coeficientes **se ajustan automáticamente** según la escala de las características:
#@markdown
#@markdown **Ejemplo conceptual:**
#@markdown - Si una característica está en rango [0, 1000] (sin escalar)
#@markdown   - Su coeficiente será pequeño (ej. 0.001) para no dominar la predicción
#@markdown - Si la misma característica se escala a [0, 1]
#@markdown   - Su coeficiente será más grande (ej. 1.0) para compensar
#@markdown
#@markdown **Lo importante:** El producto $\mathbf{w}^\top\mathbf{x}$ termina siendo **similar** en ambos casos:
#@markdown
#@markdown $$\text{Sin escalar: } 0.001 \times 1000 = 1.0$$
#@markdown $$\text{Con escalar: } 1.0 \times 1.0 = 1.0$$
#@markdown
#@markdown ---
#@markdown
#@markdown ## 📊 Comparación de magnitudes
#@markdown
#@markdown | Feature | Coef (scaled) | Coef (noscale) | Observación |
#@markdown |:--------|:--------------|:---------------|:------------|
#@markdown | Feature 1 | -2.469 | -0.2697 | Scaled es ~9× más grande |
#@markdown | Feature 2 | -1.2325 | -0.0639 | Scaled es ~19× más grande |
#@markdown | Feature 3 | -0.1951 | 0.2929 | ¡Cambia de signo! |
#@markdown | Feature 4 | 0.2721 | -0.3324 | ¡Cambia de signo! |
#@markdown | Feature 5 | -1.3686 | -1.2402 | Relativamente similares |
#@markdown
#@markdown **Clave:** Los coeficientes cambian mucho, pero las **predicciones finales** son muy similares.
#@markdown
#@markdown ---
#@markdown
#@markdown ## 🎯 ¿Qué significan los signos?
#@markdown
#@markdown ### Coeficiente positivo (+)
#@markdown - **Aumenta** la probabilidad de clase 1 (Benigno ✅)
#@markdown - Característica grande → más probable benigno
#@markdown
#@markdown **Ejemplo:** `Feature 4 (scaled): +0.2721`
#@markdown - Si esta característica aumenta → modelo predice "benigno"
#@markdown
#@markdown ### Coeficiente negativo (-)
#@markdown - **Disminuye** la probabilidad de clase 1 (aumenta probabilidad de Maligno ⚠️)
#@markdown - Característica grande → más probable maligno
#@markdown
#@markdown **Ejemplo:** `Feature 1 (scaled): -2.469`
#@markdown - Si esta característica aumenta → modelo predice "maligno"
#@markdown - Es el más negativo → **muy importante** para detectar malignidad
#@markdown
#@markdown ---
#@markdown
#@markdown ## ⚖️ ¿Cuál usar: con o sin escalado?
#@markdown
#@markdown ### ✅ Ventajas de usar escalado:
#@markdown
#@markdown 1. **Interpretabilidad**: Puedes comparar coeficientes directamente
#@markdown    - $|w_1| = 2.469$ vs $|w_2| = 1.2325$
#@markdown    - Feature 1 es ~2× más importante que Feature 2
#@markdown
#@markdown 2. **Evita dominancia**: Features con valores grandes no dominan artificialmente
#@markdown
#@markdown 3. **Mejor para regularización**: Ridge/Lasso penalizan justamente
#@markdown
#@markdown 4. **Convergencia más rápida**: En métodos iterativos
#@markdown
#@markdown ### ⚠️ Sin escalado:
#@markdown
#@markdown - Difícil interpretar (coeficientes en diferentes escalas)
#@markdown - Puede funcionar igual en predicción (pero menos claro)
#@markdown - Solo útil si necesitas coeficientes en unidades originales
#@markdown
#@markdown ---
#@markdown
#@markdown ## 🧮 Las predicciones son similares
#@markdown
#@markdown A pesar de coeficientes diferentes, las predicciones son casi idénticas porque:
#@markdown
#@markdown $$z = \mathbf{w}^\top \mathbf{x} + b$$
#@markdown
#@markdown Los valores de $z$ terminan siendo similares en ambos modelos:
#@markdown - Coeficiente grande × valor pequeño (escalado)
#@markdown - Coeficiente pequeño × valor grande (sin escalar)
#@markdown
#@markdown Por lo tanto:
#@markdown
#@markdown $$\hat{p} = \sigma(z) \approx \text{mismo resultado}$$
#@markdown
#@markdown ---
#@markdown
#@markdown ## 💡 Cómo interpretar coeficientes (con escalado)
#@markdown
#@markdown ### 1. Identifica features importantes por magnitud:
#@markdown ```
#@markdown |w1| = 2.469  → MUY importante
#@markdown |w2| = 1.233  → Importante
#@markdown |w3| = 0.195  → Poco importante
#@markdown ```
#@markdown
#@markdown ### 2. Interpreta el signo:
#@markdown - **Negativo** → aumenta P(maligno) ⚠️
#@markdown - **Positivo** → aumenta P(benigno) ✅
#@markdown
#@markdown ### 3. Ejemplo con nombres reales:
#@markdown
#@markdown Si las features son características del tumor:
#@markdown ```
#@markdown mean_radius:      w = -2.469 ⚠️ Radio grande → MÁS maligno
#@markdown mean_texture:     w = -1.233 ⚠️ Textura irregular → MÁS maligno
#@markdown mean_perimeter:   w = -0.195 📊 Poco importante
#@markdown mean_area:        w = +0.272 ✅ Área grande → MÁS benigno
#@markdown mean_smoothness:  w = -1.369 ⚠️ Suavidad → MÁS maligno
#@markdown ```
#@markdown
#@markdown ---
#@markdown
#@markdown ## 📋 Resumen
#@markdown
#@markdown | Aspecto | Con escalado | Sin escalado |
#@markdown |:--------|:-------------|:-------------|
#@markdown | Coeficientes | Comparables | Dispares |
#@markdown | Interpretación | ✅ Fácil | ❌ Difícil |
#@markdown | Predicciones | Iguales | Iguales |
#@markdown | Recomendado | ✅ SÍ | Solo casos especiales |
#@markdown
#@markdown ---
#@markdown
#@markdown ## 🔑 Conclusión
#@markdown
#@markdown **Los coeficientes cambian mucho con/sin escalado, pero el modelo predice igual.**
#@markdown
#@markdown El modelo **compensa automáticamente** la escala de las features ajustando los coeficientes.
#@markdown
#@markdown **Usa siempre escalado** para:
#@markdown - ✅ Interpretar qué features son importantes
#@markdown - ✅ Comparar magnitudes de coeficientes
#@markdown - ✅ Aplicar regularización justa
#@markdown - ✅ Entrenar más rápido


In [ ]:
#@title 🔍 Explorar Predicciones de Diferentes Casos
#@markdown Cambia el índice del caso para ver diferentes predicciones del modelo

# Opción 1: Seleccionar por índice
indice_caso = 13  #@param {type:"slider", min:0, max:113, step:1}

print("=" * 70)
print(f"PREDICCIÓN PARA EL CASO #{indice_caso}")
print("=" * 70)

caso = Xc_test.iloc[indice_caso:indice_caso+1]
prob = pipe_log_scaled.predict_proba(caso)[0]
pred = pipe_log_scaled.predict(caso)[0]
real = yc_test.iloc[indice_caso]

print(f"Valor real:    {real} ({'Benigno ✅' if real == 1 else 'Maligno ⚠️'})")
print(f"Predicción:    {pred} ({'Benigno ✅' if pred == 1 else 'Maligno ⚠️'})")
print(f"¿Correcto?     {'SÍ ✓' if pred == real else 'NO ✗ ERROR'}")
print(f"\nProbabilidades:")
print(f"  P(Maligno=0 | x) = {prob[0]:.4f} ({prob[0]*100:.2f}%)")
print(f"  P(Benigno=1 | x) = {prob[1]:.4f} ({prob[1]*100:.2f}%)")
print(f"\nConfianza: {'ALTA' if max(prob) > 0.9 else 'MEDIA' if max(prob) > 0.7 else 'BAJA'}")

In [ ]:
# ================================================================
# 🎨 Visualizaciones del efecto del escalado y PCA/polinomiales
# ================================================================
import numpy as np, matplotlib.pyplot as plt
from sklearn.datasets import make_classification, make_regression
from sklearn.preprocessing import StandardScaler, MinMaxScaler, PolynomialFeatures
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression, LinearRegression

# ---------- 1️⃣ DEMO DE CLASIFICACIÓN (sin escalado vs escalado) ----------
Xc, yc = make_classification(
    n_samples=400, n_features=2, n_redundant=0, n_informative=2,
    n_clusters_per_class=1, class_sep=1.2, random_state=42
)
# Reescalamos manualmente para simular distintas magnitudes
Xc[:, 0] *= 100  # variable con escala grande
Xc[:, 1] *= 0.01  # variable con escala pequeña

fig, ax = plt.subplots(1, 2, figsize=(12,5))
ax[0].scatter(Xc[yc==0,0], Xc[yc==0,1], s=15, label='Clase 0', alpha=0.7)
ax[0].scatter(Xc[yc==1,0], Xc[yc==1,1], s=15, label='Clase 1', alpha=0.7)
ax[0].set_title("Datos sin escalar (escalas desbalanceadas)")
ax[0].legend()

# Entrenamos LogReg sin escalado
log1 = LogisticRegression().fit(Xc, yc)
xx, yy = np.meshgrid(np.linspace(Xc[:,0].min(), Xc[:,0].max(), 200),
                     np.linspace(Xc[:,1].min(), Xc[:,1].max(), 200))
Z = log1.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
ax[0].contourf(xx, yy, Z, alpha=0.15)

# Ahora escalamos y volvemos a entrenar
sc = StandardScaler().fit(Xc)
Xs = sc.transform(Xc)
log2 = LogisticRegression().fit(Xs, yc)
xxs, yys = np.meshgrid(np.linspace(Xs[:,0].min(), Xs[:,0].max(), 200),
                       np.linspace(Xs[:,1].min(), Xs[:,1].max(), 200))
Z2 = log2.predict(np.c_[xxs.ravel(), yys.ravel()]).reshape(xxs.shape)
ax[1].contourf(xxs, yys, Z2, alpha=0.15)
ax[1].scatter(Xs[yc==0,0], Xs[yc==0,1], s=15, label='Clase 0', alpha=0.7)
ax[1].scatter(Xs[yc==1,0], Xs[yc==1,1], s=15, label='Clase 1', alpha=0.7)
ax[1].set_title("Datos escalados (StandardScaler)")
plt.suptitle("📊 Fronteras de decisión: sin escalar vs escalado", fontsize=13)
plt.show()

# ---------- 2️⃣ DEMO DE REGRESIÓN Y PCA ----------
Xr, yr = make_regression(
    n_samples=300, n_features=3, noise=10, random_state=42
)
# PCA sin escalado (ineficiente)
pca1 = PCA(n_components=2).fit(Xr)
Xpca1 = pca1.transform(Xr)
pca2 = PCA(n_components=2).fit(StandardScaler().fit_transform(Xr))
Xpca2 = pca2.transform(StandardScaler().fit_transform(Xr))

fig, ax = plt.subplots(1, 2, figsize=(12,5))
ax[0].scatter(Xpca1[:,0], Xpca1[:,1], c=yr, cmap='viridis', s=20)
ax[0].set_title("PCA sin escalar → proyección distorsionada")
ax[1].scatter(Xpca2[:,0], Xpca2[:,1], c=yr, cmap='viridis', s=20)
ax[1].set_title("PCA con escalado → proyección informativa")
plt.suptitle("🔻 PCA antes y después del escalado")
plt.show()

# ---------- 3️⃣ DEMO DE EXPANSIÓN POLINÓMICA ----------
# Usamos solo una variable para visualizar la curvatura
Xp = np.linspace(-3, 3, 100).reshape(-1,1)
y_true = 2*Xp[:,0]**2 - 1 + np.random.randn(100)*0.3  # relación cuadrática
lin = LinearRegression().fit(Xp, y_true)
poly = PolynomialFeatures(degree=2)
Xp_poly = poly.fit_transform(Xp)
lin_poly = LinearRegression().fit(Xp_poly, y_true)

plt.figure(figsize=(8,5))
plt.scatter(Xp, y_true, s=15, label='Datos reales')
plt.plot(Xp, lin.predict(Xp), 'r--', label='Regresión lineal')
plt.plot(Xp, lin_poly.predict(Xp_poly), 'g-', label='Regresión polinómica (grado 2)')
plt.title("🔺 Expansión polinómica — captura de relaciones no lineales")
plt.legend(); plt.show()


<details>
<summary><b>📐 Regresión Logística — Fundamentos matemáticos</b> (clic para desplegar)</summary>

### 🧩 Modelo

Sea $\mathbf{x} \in \mathbb{R}^d$, con parámetros $(\mathbf{w}, b)$.  
El modelo de regresión logística busca estimar la **probabilidad** de que una observación pertenezca a la clase positiva (1).  
A diferencia de la regresión lineal, su salida se restringe al rango **[0,1]**, usando la **función sigmoide (o logística)**:

$$
\hat{p}(y=1\mid \mathbf{x}) = \sigma(z) = \frac{1}{1+e^{-z}},
\quad z = \mathbf{w}^\top \mathbf{x} + b
$$

- $\mathbf{w}$: vector de pesos o coeficientes asociados a cada variable independiente.  
- $b$: término de sesgo o intercepto.  
- $z$: combinación lineal de las variables de entrada (aún sin restringir).  
- $\sigma(z)$: transforma cualquier número real en un valor entre 0 y 1 (probabilidad).  

👉 **Interpretación:**  
Cuando $z$ crece (la combinación lineal indica mayor evidencia hacia la clase 1), la probabilidad $\hat{p}$ se acerca a 1.  
Si $z$ disminuye, $\hat{p}$ se acerca a 0.

---

### ⚙️ Función de pérdida (log-loss / entropía cruzada binaria)

El objetivo del entrenamiento es encontrar los parámetros $(\mathbf{w}, b)$ que minimicen la diferencia entre las predicciones del modelo $\hat{p}_i$ y las etiquetas reales $y_i$.  
Para ello, se utiliza la **entropía cruzada** (log-loss), una medida de error probabilística:

$$
\mathcal{L}(\mathbf{w}, b)
= -\frac{1}{N}\sum_{i=1}^{N}
\Big[y_i \log \hat{p}_i + (1-y_i)\log(1-\hat{p}_i)\Big]
$$

- El primer término penaliza los errores cuando la clase real es **1**.  
- El segundo término penaliza los errores cuando la clase real es **0**.  
- Cuanto más se acerque $\hat{p}_i$ a $y_i$, menor será la pérdida.  

👉 **Interpretación:**  
La log-loss mide el grado de "sorpresa" del modelo. Si el modelo asigna alta probabilidad a la clase correcta, la pérdida será baja.  
Si el modelo está muy seguro pero se equivoca, la penalización es severa (crece logarítmicamente).

---

### 🔢 Gradientes

Para optimizar los parámetros, se calcula cómo cambia la pérdida respecto a cada peso.  
Los **gradientes** indican la dirección en la que la pérdida aumenta; por lo tanto, al restarlos, movemos los pesos hacia donde la pérdida disminuye.

Usando $\hat{p}_i = \sigma(z_i)$ y $z_i = \mathbf{w}^\top \mathbf{x}_i + b$:

$$
\frac{\partial \mathcal{L}}{\partial \mathbf{w}}
= \frac{1}{N}\sum_{i=1}^{N} (\hat{p}_i - y_i)\mathbf{x}_i,
\qquad
\frac{\partial \mathcal{L}}{\partial b}
= \frac{1}{N}\sum_{i=1}^{N} (\hat{p}_i - y_i)
$$

- $(\hat{p}_i - y_i)$ representa el **error** en la predicción para el ejemplo $i$.  
- Al multiplicarlo por $\mathbf{x}_i$, se pondera la contribución de cada variable al error total.  

👉 **Interpretación:**  
Si una característica tiene correlación positiva con el error, su peso debe disminuir;  
si tiene correlación negativa, su peso debe aumentar.  
Esto guía el ajuste adaptativo de los parámetros.

---

### 🔽 Actualización por gradiente descendente

Con los gradientes calculados, se actualizan los pesos para reducir la pérdida, siguiendo la regla del **gradiente descendente**:

$$
\mathbf{w} \leftarrow \mathbf{w} - \eta \frac{\partial \mathcal{L}}{\partial \mathbf{w}},
\quad
b \leftarrow b - \eta \frac{\partial \mathcal{L}}{\partial b}
$$

- $\eta$ es la **tasa de aprendizaje**, que controla el tamaño del paso en cada iteración.  
- Si $\eta$ es muy grande → el modelo oscila y no converge.  
- Si $\eta$ es muy pequeña → el modelo tarda mucho en aprender.  

👉 **Interpretación visual:**  
Cada paso del algoritmo ajusta los pesos en dirección contraria a la pendiente de la función de pérdida, buscando el mínimo global o un mínimo local estable.

---

📘 **Resumen conceptual:**

1. La regresión logística transforma una combinación lineal ($z$) en una probabilidad.  
2. Optimiza sus parámetros minimizando la pérdida de entropía cruzada.  
3. El proceso de ajuste se basa en derivadas (gradientes) y en el descenso iterativo por la superficie de error.  
4. El resultado final es un modelo que estima $P(y=1|\mathbf{x})$ y permite clasificar según un umbral (típicamente 0.5).

</details>



In [ ]:

#@markdown ### 🧪 Implementación from scratch — Regresión Logística (binaria)
class LogisticRegressionFromScratch:
    def __init__(self, lr=0.1, n_iter=2000, fit_intercept=True):
        self.lr = lr
        self.n_iter = n_iter
        self.fit_intercept = fit_intercept

    @staticmethod
    def _sigmoid(z):
        return 1.0 / (1.0 + np.exp(-z))

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float)
        N, d = X.shape

        if self.fit_intercept:
            Xb = np.hstack([X, np.ones((N,1))])
        else:
            Xb = X

        self.w_ = np.zeros(Xb.shape[1])

        for _ in range(self.n_iter):
            z = Xb @ self.w_
            p = self._sigmoid(z)
            grad = (Xb.T @ (p - y)) / N
            self.w_ -= self.lr * grad

        return self

    def predict_proba(self, X):
        X = np.asarray(X, dtype=float)
        if self.fit_intercept:
            X = np.hstack([X, np.ones((X.shape[0],1))])
        z = X @ self.w_
        p = self._sigmoid(z)
        return np.c_[1-p, p]  # [P(y=0), P(y=1)]

    def predict(self, X, threshold=0.5):
        proba = self.predict_proba(X)[:,1]
        return (proba >= threshold).astype(int)


In [ ]:

#@markdown #### Entrenamiento y evaluación — LogReg (from scratch)
logreg_fs = LogisticRegressionFromScratch(lr=0.1, n_iter=4000)
logreg_fs.fit(X_train_sc, y_train)

y_pred_fs = logreg_fs.predict(X_test_sc)
acc_fs = accuracy_score(y_test, y_pred_fs)
prec_fs = precision_score(y_test, y_pred_fs)
rec_fs = recall_score(y_test, y_pred_fs)
f1_fs = f1_score(y_test, y_pred_fs)

acc_fs, prec_fs, rec_fs, f1_fs


In [ ]:

#@markdown #### Matriz de confusión — LogReg (from scratch)
cm = confusion_matrix(y_test, y_pred_fs, labels=[0,1])
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['maligno (0)', 'benigno (1)'])
disp.plot(values_format='d')
plt.title('Matriz de confusión — LogReg (from scratch)')
plt.show()


In [ ]:

#@markdown ### 🗺️ Frontera de decisión en 2D (PCA) — LogReg (from scratch)
pca = PCA(n_components=2, random_state=42)
X_train_2d = pca.fit_transform(X_train_sc)
X_test_2d = pca.transform(X_test_sc)

# Reentrenamos un modelo 2D para graficar frontera (sobre el espacio PCA)
logreg2d = LogisticRegressionFromScratch(lr=0.1, n_iter=4000)
logreg2d.fit(X_train_2d, y_train)

# mallado para la frontera
x_min, x_max = X_train_2d[:,0].min()-1, X_train_2d[:,0].max()+1
y_min, y_max = X_train_2d[:,1].min()-1, X_train_2d[:,1].max()+1
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200),
                     np.linspace(y_min, y_max, 200))
grid = np.c_[xx.ravel(), yy.ravel()]
Z = logreg2d.predict(grid).reshape(xx.shape)

fig, ax = plt.subplots()
ax.contourf(xx, yy, Z, alpha=0.3)
scatter0 = ax.scatter(X_train_2d[y_train==0,0], X_train_2d[y_train==0,1], label='maligno (0)', s=15)
scatter1 = ax.scatter(X_train_2d[y_train==1,0], X_train_2d[y_train==1,1], label='benigno (1)', s=15)
ax.set_title('Frontera de decisión (PCA 2D) — LogReg (from scratch)')
ax.legend()
plt.show()


In [ ]:

#@markdown ### 🧰 Regresión Logística con scikit-learn (baseline, sin tuning)
logreg_sk = LogisticRegression(max_iter=1000)
logreg_sk.fit(X_train_sc, y_train)
y_pred_sk = logreg_sk.predict(X_test_sc)

acc_sk = accuracy_score(y_test, y_pred_sk)
prec_sk = precision_score(y_test, y_pred_sk)
rec_sk = recall_score(y_test, y_pred_sk)
f1_sk = f1_score(y_test, y_pred_sk)

acc_sk, prec_sk, rec_sk, f1_sk


# Regresion polinomial

In [ ]:
#@markdown # 📚 Regresión Polinomial — Guía completa para entender cada fórmula
#@markdown
#@markdown ---
#@markdown
#@markdown ## 🎯 ¿Qué es la Regresión Polinomial?
#@markdown
#@markdown La **Regresión Polinomial** es una extensión de la regresión lineal que permite modelar relaciones **no lineales** entre las características y la variable objetivo.
#@markdown
#@markdown **Diferencia clave:**
#@markdown - **Regresión Lineal**: $\hat{y} = w_1x + b$ (línea recta)
#@markdown - **Regresión Polinomial**: $\hat{y} = w_1x + w_2x^2 + w_3x^3 + \cdots + w_dx^d + b$ (curva)
#@markdown
#@markdown **Aplicaciones:**
#@markdown - Predecir precios con tendencias no lineales
#@markdown - Modelar crecimiento biológico
#@markdown - Ajustar curvas a datos experimentales
#@markdown
#@markdown ---
#@markdown
#@markdown ## 🧩 La idea fundamental
#@markdown
#@markdown **Truco matemático:** Aunque el modelo es **no lineal en x**, sigue siendo **lineal en los pesos w**.
#@markdown
#@markdown **Ejemplo con una variable:**
#@markdown
#@markdown Modelo polinomial de grado 2:
#@markdown $$\hat{y} = w_1x + w_2x^2 + b$$
#@markdown
#@markdown Lo transformamos creando **nuevas características**:
#@markdown $$x_1 = x, \quad x_2 = x^2$$
#@markdown
#@markdown Ahora es un modelo lineal:
#@markdown $$\hat{y} = w_1x_1 + w_2x_2 + b$$
#@markdown
#@markdown **¡Podemos usar las mismas fórmulas de regresión lineal!**
#@markdown
#@markdown ---
#@markdown
#@markdown ## 🔢 Transformación Polinomial (Feature Engineering)
#@markdown
#@markdown Para un polinomio de **grado d** con características originales $\mathbf{x} = [x_1, x_2, \ldots, x_n]$:
#@markdown
#@markdown **Creamos nuevas features** incluyendo:
#@markdown - Términos lineales: $x_1, x_2, \ldots$
#@markdown - Términos cuadráticos: $x_1^2, x_1x_2, x_2^2, \ldots$
#@markdown - Términos cúbicos: $x_1^3, x_1^2x_2, \ldots$ (si d≥3)
#@markdown - Y así sucesivamente hasta grado $d$
#@markdown
#@markdown **Ejemplo con 2 features originales y grado 2:**
#@markdown
#@markdown Entrada original: $\mathbf{x} = [x_1, x_2]$
#@markdown
#@markdown Features polinomiales: $\mathbf{x}_{\text{poly}} = [1, x_1, x_2, x_1^2, x_1x_2, x_2^2]$
#@markdown
#@markdown **Número de features crece rápido:**
#@markdown - Original: $n$ features
#@markdown - Grado 2: $\frac{(n+1)(n+2)}{2}$ features
#@markdown - Grado 3: $\frac{(n+1)(n+2)(n+3)}{6}$ features
#@markdown
#@markdown ---
#@markdown
#@markdown ## 📐 Fórmulas (idénticas a regresión lineal)
#@markdown
#@markdown Una vez que tenemos $\mathbf{X}_{\text{poly}}$, usamos las mismas fórmulas:
#@markdown
#@markdown ### Hipótesis:
#@markdown $$\hat{y}_i = \mathbf{w}^\top \mathbf{x}_{\text{poly},i} + b$$
#@markdown
#@markdown ### Función de pérdida (MSE):
#@markdown $$\mathcal{L}(\mathbf{w}, b) = \frac{1}{N}\sum_{i=1}^N(\hat{y}_i - y_i)^2$$
#@markdown
#@markdown ### Gradiente:
#@markdown $$\frac{\partial \mathcal{L}}{\partial \mathbf{w}_b} = \frac{2}{N} \mathbf{X}_{\text{poly},b}^\top (\mathbf{X}_{\text{poly},b}\mathbf{w}_b - \mathbf{y})$$
#@markdown
#@markdown ### Actualización:
#@markdown $$\mathbf{w}_b \leftarrow \mathbf{w}_b - \eta \frac{\partial \mathcal{L}}{\partial \mathbf{w}_b}$$
#@markdown
#@markdown **La única diferencia:** Usamos $\mathbf{X}_{\text{poly}}$ en lugar de $\mathbf{X}$ original.
#@markdown
#@markdown ---
#@markdown
#@markdown ## ⚖️ Grado del polinomio: ¿Cuál elegir?
#@markdown
#@markdown | Grado | Modelo | Ventajas | Desventajas |
#@markdown |:------|:-------|:---------|:------------|
#@markdown | d=1 | Lineal | Simple, rápido, interpretable | No captura curvas |
#@markdown | d=2 | Cuadrático | Captura curvaturas básicas | Puede ser insuficiente |
#@markdown | d=3-4 | Cúbico/Cuártico | Flexibilidad moderada | Balance bueno |
#@markdown | d≥5 | Alto grado | Muy flexible | **Sobreajuste** (overfitting) |
#@markdown
#@markdown **Regla práctica:** Empieza con d=2 o d=3, aumenta solo si es necesario.
#@markdown
#@markdown ---
#@markdown
#@markdown ## ⚠️ Problemas comunes y soluciones
#@markdown
#@markdown ### 1. Sobreajuste (Overfitting)
#@markdown **Problema:** Grado muy alto → el modelo memoriza datos de entrenamiento
#@markdown
#@markdown **Síntomas:**
#@markdown - Error bajo en train, alto en test
#@markdown - Curva con muchas oscilaciones extrañas
#@markdown
#@markdown **Soluciones:**
#@markdown - Reducir el grado del polinomio
#@markdown - Usar **regularización** (Ridge, Lasso)
#@markdown - Aumentar datos de entrenamiento
#@markdown
#@markdown ### 2. Explosión numérica
#@markdown **Problema:** $x^{10}$ puede ser enorme si $x > 1$
#@markdown
#@markdown **Solución:** **SIEMPRE escalar las features** antes de crear polinomios
#@markdown ```python
#@markdown scaler = StandardScaler()
#@markdown X_scaled = scaler.fit_transform(X)
#@markdown X_poly = crear_features_polinomiales(X_scaled, grado=3)
#@markdown ```
#@markdown
#@markdown ### 3. Multicolinealidad
#@markdown **Problema:** $x$ y $x^2$ están muy correlacionados
#@markdown
#@markdown **Solución:** Usar regularización (Ridge es particularmente efectivo)
#@markdown
#@markdown ---
#@markdown
#@markdown ## 💡 Interpretación de resultados
#@markdown
#@markdown **Ejemplo:** Modelo cuadrático $\hat{y} = 2x - 0.5x^2 + 1$
#@markdown
#@markdown ```
#@markdown w1 = 2.0   (término lineal, positivo)
#@markdown w2 = -0.5  (término cuadrático, negativo)
#@markdown b = 1.0    (intercepto)
#@markdown ```
#@markdown
#@markdown **Interpretación:**
#@markdown - Coeficiente lineal positivo (+2) → inicialmente y aumenta con x
#@markdown - Coeficiente cuadrático negativo (-0.5) → parábola hacia abajo
#@markdown - Resultado: **curva que sube y luego baja** (forma de U invertida)
#@markdown
#@markdown **Aplicación práctica:**
#@markdown - Podría modelar: ingresos vs precio (aumenta hasta un punto óptimo, luego baja)
#@markdown - O: rendimiento vs esfuerzo (mejora inicialmente, luego agotamiento)
#@markdown
#@markdown ---
#@markdown
#@markdown ## 🎓 Resumen
#@markdown
#@markdown **Concepto clave:** Regresión Polinomial = Regresión Lineal con features transformadas
#@markdown
#@markdown **Proceso:**
#@markdown 1. Escalar features originales
#@markdown 2. Crear features polinomiales (x, x², x³, ...)
#@markdown 3. Aplicar regresión lineal estándar
#@markdown 4. Obtener predicciones (curva no lineal)
#@markdown
#@markdown **Ventajas:**
#@markdown - ✅ Modela relaciones no lineales
#@markdown - ✅ Fácil de implementar (reutiliza código lineal)
#@markdown - ✅ Matemáticamente bien entendido
#@markdown
#@markdown **Cuidados:**
#@markdown - ⚠️ SIEMPRE escalar antes de polinomios
#@markdown - ⚠️ Grado alto → sobreajuste
#@markdown - ⚠️ Usar regularización si es necesario
#@markdown - ⚠️ Validar en datos de prueba

In [ ]:
#@title 🔧 Funciones auxiliares para Regresión Polinomial

def crear_features_polinomiales(X, grado=2):
    """
    Crea features polinomiales manualmente

    Parámetros:
    -----------
    X : array de forma (N, d)
        Features originales
    grado : int
        Grado del polinomio

    Retorna:
    --------
    X_poly : array de forma (N, d_poly)
        Features polinomiales (incluye todas las combinaciones hasta el grado especificado)
    """
    from sklearn.preprocessing import PolynomialFeatures
    poly = PolynomialFeatures(degree=grado, include_bias=False)
    return poly.fit_transform(X)

def add_bias(X):
    """Agrega columna de 1s para el sesgo"""
    return np.c_[X, np.ones(X.shape[0])]

print("✅ Funciones auxiliares cargadas")

In [ ]:
#@markdown # ⚙️ Hiperparámetros en Regresión Polinomial — Guía Completa
#@markdown
#@markdown ---
#@markdown
#@markdown ## 🎛️ ¿Qué son los hiperparámetros?
#@markdown
#@markdown Los **hiperparámetros** son configuraciones que TÚ decides ANTES de entrenar el modelo.
#@markdown
#@markdown **Diferencia clave:**
#@markdown - **Parámetros** (w, b): Los aprende el modelo automáticamente durante el entrenamiento
#@markdown - **Hiperparámetros**: Tú los decides manualmente y controlan CÓMO aprende el modelo
#@markdown
#@markdown **Analogía:** Si entrenar un modelo es como hornear un pastel:
#@markdown - Los **parámetros** son el resultado (el pastel cocido)
#@markdown - Los **hiperparámetros** son las configuraciones del horno (temperatura, tiempo)
#@markdown
#@markdown ---
#@markdown
#@markdown ## 📊 Hiperparámetro 1: Grado del Polinomio (degree)
#@markdown
#@markdown ```python
#@markdown grado_polinomio = 3  # ¿Cuántas curvaturas puede hacer el modelo?
#@markdown ```
#@markdown
#@markdown ### ¿Qué es?
#@markdown El **grado** determina la **complejidad máxima** de la curva que el modelo puede aprender.
#@markdown
#@markdown ### Ejemplos visuales:
#@markdown
#@markdown **Grado 1 (Lineal):**
#@markdown $$\hat{y} = w_1x + b$$
#@markdown ```
#@markdown Forma: Línea recta  ──────
#@markdown Ejemplo: y = 2x + 1
#@markdown ```
#@markdown
#@markdown **Grado 2 (Cuadrático):**
#@markdown $$\hat{y} = w_1x + w_2x^2 + b$$
#@markdown ```
#@markdown Forma: Parábola (U o ∩)  ∪
#@markdown Ejemplo: y = x - 0.5x² + 1
#@markdown ```
#@markdown
#@markdown **Grado 3 (Cúbico):**
#@markdown $$\hat{y} = w_1x + w_2x^2 + w_3x^3 + b$$
#@markdown ```
#@markdown Forma: Curva S  ∿
#@markdown Ejemplo: y = x - x² + 0.2x³
#@markdown ```
#@markdown
#@markdown **Grado 5 (Alto):**
#@markdown $$\hat{y} = w_1x + w_2x^2 + w_3x^3 + w_4x^4 + w_5x^5 + b$$
#@markdown ```
#@markdown Forma: Curva muy ondulada  ∿∿∿
#@markdown ⚠️ Peligro: Puede ajustarse "demasiado bien" a los datos
#@markdown ```
#@markdown
#@markdown ### ¿Cómo afecta al modelo?
#@markdown
#@markdown | Grado | Flexibilidad | Cuándo usar | Riesgo |
#@markdown |:------|:-------------|:------------|:-------|
#@markdown | **1** | Muy baja | Relaciones lineales simples | ❌ Subajuste (underfitting) |
#@markdown | **2** | Baja | Curvas suaves (parábolas) | ✅ Generalmente seguro |
#@markdown | **3** | Media | Curvas con 1-2 inflexiones | ✅ Buen balance |
#@markdown | **4-5** | Alta | Datos muy complejos | ⚠️ Posible sobreajuste |
#@markdown | **≥6** | Muy alta | Casi nunca necesario | ❌ Sobreajuste (overfitting) |
#@markdown
#@markdown ### Síntomas de elegir mal:
#@markdown
#@markdown **Grado muy BAJO (subajuste):**
#@markdown ```
#@markdown Síntomas:
#@markdown - MSE alto tanto en train como en test
#@markdown - La curva no captura patrones obvios
#@markdown - Predicciones muy imprecisas
#@markdown
#@markdown Ejemplo: Usar línea recta (grado 1) para datos curvos
#@markdown ```
#@markdown
#@markdown **Grado muy ALTO (sobreajuste):**
#@markdown ```
#@markdown Síntomas:
#@markdown - MSE muy bajo en train, ALTO en test
#@markdown - La curva hace "zigzag" entre puntos
#@markdown - Excelente en entrenamiento, terrible en nuevos datos
#@markdown
#@markdown Ejemplo: Usar grado 10 con pocos datos
#@markdown ```
#@markdown
#@markdown ### ¿Cómo elegir el grado óptimo?
#@markdown
#@markdown **Método 1: Prueba y error sistemático**
#@markdown ```python
#@markdown # Probar grados 1, 2, 3, 4, 5
#@markdown for grado in [1, 2, 3, 4, 5]:
#@markdown     entrenar_modelo(grado)
#@markdown     calcular_mse_test()
#@markdown # Elegir el grado con MENOR MSE en test
#@markdown ```
#@markdown
#@markdown **Método 2: Regla práctica inicial**
#@markdown - Empieza con **grado 2** (cuadrático)
#@markdown - Si MSE test es alto → aumenta a grado 3
#@markdown - Si MSE test > MSE train significativamente → reduce el grado
#@markdown
#@markdown **Método 3: Validación cruzada** (avanzado)
#@markdown - Divide datos en múltiples "folds"
#@markdown - Entrena con diferentes grados
#@markdown - Elige el grado con mejor promedio
#@markdown
#@markdown ### Ejemplos prácticos:
#@markdown
#@markdown **Caso 1: Precio de casas vs área**
#@markdown ```
#@markdown Relación: Probablemente cuadrática
#@markdown Grado recomendado: 2
#@markdown Razón: A mayor área, mayor precio, pero con rendimientos decrecientes
#@markdown ```
#@markdown
#@markdown **Caso 2: Temperatura a lo largo del día**
#@markdown ```
#@markdown Relación: Cíclica/sinusoidal
#@markdown Grado recomendado: 3-4
#@markdown Razón: Sube en la mañana, pico al mediodía, baja en la noche
#@markdown ```
#@markdown
#@markdown **Caso 3: Crecimiento de bacterias**
#@markdown ```
#@markdown Relación: Exponencial (muy no lineal)
#@markdown Grado recomendado: Polinomios NO son ideales
#@markdown Mejor opción: Transformación logarítmica o modelo exponencial
#@markdown ```
#@markdown
#@markdown ---
#@markdown
#@markdown ## 🎓 Hiperparámetro 2: Usar Escalado (use_scaling)
#@markdown
#@markdown ```python
#@markdown use_scaling_poly = True  # ¿Normalizar features antes de entrenar?
#@markdown ```
#@markdown
#@markdown ### ¿Qué es?
#@markdown El **escalado** transforma todas las características para que tengan **media 0** y **desviación estándar 1**.
#@markdown
#@markdown ### ¿Por qué es CRÍTICO en polinomios?
#@markdown
#@markdown **Problema sin escalado:**
#@markdown ```
#@markdown Si x = 100, entonces:
#@markdown x²  = 10,000
#@markdown x³  = 1,000,000
#@markdown x⁴  = 100,000,000  ← ¡EXPLOSIÓN NUMÉRICA!
#@markdown
#@markdown Resultado: El modelo se rompe (overflow, NaN, etc.)
#@markdown ```
#@markdown
#@markdown **Con escalado (x normalizado entre -2 y +2):**
#@markdown ```
#@markdown Si x = 1.5, entonces:
#@markdown x²  = 2.25
#@markdown x³  = 3.375
#@markdown x⁴  = 5.063  ← Todo bajo control
#@markdown
#@markdown Resultado: Entrenamiento estable
#@markdown ```
#@markdown
#@markdown ### Comparación visual:
#@markdown
#@markdown | Aspecto | SIN escalado | CON escalado |
#@markdown |:--------|:-------------|:-------------|
#@markdown | Valores de x² | Pueden ser enormes | Rango controlado |
#@markdown | Convergencia | Lenta o falla | Rápida y estable |
#@markdown | Precisión numérica | Problemas de overflow | Sin problemas |
#@markdown | Interpretación | Difícil | Más fácil |
#@markdown
#@markdown ### ¿Cuándo DEBES usar escalado?
#@markdown
#@markdown **SIEMPRE en polinomios de grado ≥2** ✅
#@markdown
#@markdown Excepciones (casi nunca):
#@markdown - Variables ya están en rango [0, 1]
#@markdown - Grado = 1 (lineal simple)
#@markdown
#@markdown ### Tipos de escalado:
#@markdown
#@markdown **StandardScaler (recomendado para polinomios):**
#@markdown $$x_{\text{scaled}} = \frac{x - \mu}{\sigma}$$
#@markdown ```
#@markdown Resultado: Media=0, Desviación=1
#@markdown Ejemplo: [10, 20, 30] → [-1.22, 0, 1.22]
#@markdown ```
#@markdown
#@markdown **MinMaxScaler (alternativa):**
#@markdown $$x_{\text{scaled}} = \frac{x - x_{\min}}{x_{\max} - x_{\min}}$$
#@markdown ```
#@markdown Resultado: Rango [0, 1]
#@markdown Ejemplo: [10, 20, 30] → [0, 0.5, 1]
#@markdown ```
#@markdown
#@markdown ---
#@markdown
#@markdown ## 🚀 Hiperparámetro 3: Tasa de Aprendizaje (alpha / learning rate)
#@markdown
#@markdown ```python
#@markdown poly_alpha = 0.01  # ¿Qué tan grande es cada "paso" del gradiente?
#@markdown ```
#@markdown
#@markdown ### ¿Qué es?
#@markdown La **tasa de aprendizaje** (η o alpha) controla qué tan grande es cada actualización de los pesos.
#@markdown
#@markdown $$\mathbf{w}_{\text{nuevo}} = \mathbf{w}_{\text{viejo}} - \eta \cdot \nabla\mathcal{L}$$
#@markdown
#@markdown ### Analogía visual:
#@markdown
#@markdown Imagina que estás bajando una montaña con niebla (no ves el valle):
#@markdown
#@markdown **η muy pequeña (ej. 0.0001):**
#@markdown ```
#@markdown Comportamiento: Pasos muy pequeños 👣👣👣👣👣
#@markdown Ventaja: Seguro, no te caes
#@markdown Desventaja: LENTO, puede tardar horas
#@markdown ```
#@markdown
#@markdown **η moderada (ej. 0.01):**
#@markdown ```
#@markdown Comportamiento: Pasos normales 👟👟👟
#@markdown Ventaja: Balance entre velocidad y seguridad
#@markdown Desventaja: Requiere ajuste
#@markdown ```
#@markdown
#@markdown **η muy grande (ej. 1.0):**
#@markdown ```
#@markdown Comportamiento: Saltos enormes 🦘🦘
#@markdown Ventaja: Rápido... en teoría
#@markdown Desventaja: Puedes "saltar sobre" el valle (divergencia)
#@markdown ```
#@markdown
#@markdown ### Síntomas de mala elección:
#@markdown
#@markdown **α muy PEQUEÑA:**
#@markdown ```
#@markdown Síntomas:
#@markdown - Pérdida disminuye muy lentamente
#@markdown - Necesitas muchas iteraciones (10,000+)
#@markdown - Entrenamiento tarda mucho
#@markdown
#@markdown Solución: Aumentar α (multiplicar por 2-10)
#@markdown ```
#@markdown
#@markdown **α muy GRANDE:**
#@markdown ```
#@markdown Síntomas:
#@markdown - Pérdida AUMENTA en lugar de disminuir
#@markdown - Pérdida oscila violentamente
#@markdown - Valores NaN o Inf aparecen
#@markdown
#@markdown Solución: Reducir α (dividir por 10)
#@markdown ```
#@markdown
#@markdown ### Gráfica de pérdida según α:
#@markdown
#@markdown ```
#@markdown α = 0.0001 (muy pequeña)
#@markdown Pérdida │╲
#@markdown        │ ╲_______________  ← Converge muy lento
#@markdown        └─────────────────→ Iteraciones
#@markdown
#@markdown α = 0.01 (óptima)
#@markdown Pérdida │╲
#@markdown        │ ╲__  ← Converge rápido y suave
#@markdown        └─────→ Iteraciones
#@markdown
#@markdown α = 1.0 (muy grande)
#@markdown Pérdida │  ╱╲╱╲╱╲
#@markdown        │╱╲╱╲╱╲╱╲  ← Oscila, NO converge
#@markdown        └──────────→ Iteraciones
#@markdown ```
#@markdown
#@markdown ### ¿Cómo elegir α?
#@markdown
#@markdown **Método 1: Prueba logarítmica**
#@markdown ```python
#@markdown # Probar: 0.001, 0.01, 0.1, 1.0
#@markdown alphas = [10**i for i in range(-3, 1)]
#@markdown for alpha in alphas:
#@markdown     entrenar_y_graficar_perdida(alpha)
#@markdown # Elegir el que converge más rápido sin oscilar
#@markdown ```
#@markdown
#@markdown **Método 2: Regla inicial**
#@markdown - **Con escalado**: α = 0.01 - 0.1
#@markdown - **Sin escalado**: α = 0.0001 - 0.001
#@markdown
#@markdown **Método 3: Ajuste adaptativo**
#@markdown ```python
#@markdown # Si pérdida no baja después de 100 iters → α muy grande
#@markdown # Si pérdida baja muy lento → α muy pequeña
#@markdown ```
#@markdown
#@markdown ### Valores típicos por contexto:
#@markdown
#@markdown | Situación | α recomendada | Razón |
#@markdown |:----------|:--------------|:------|
#@markdown | Features escaladas, grado 2-3 | 0.01 - 0.1 | Gradientes bien condicionados |
#@markdown | Features sin escalar | 0.0001 - 0.001 | Gradientes muy grandes |
#@markdown | Muchas features (>100) | 0.001 - 0.01 | Evitar inestabilidad |
#@markdown | Pocas features (<10) | 0.01 - 0.5 | Puede permitirse más agresivo |
#@markdown | Grado muy alto (≥5) | 0.001 - 0.01 | Superficie de error compleja |
#@markdown
#@markdown ---
#@markdown
#@markdown ## 🔁 Hiperparámetro 4: Número de Iteraciones (n_iterations)
#@markdown
#@markdown ```python
#@markdown poly_iters = 2000  # ¿Cuántas veces actualizar los pesos?
#@markdown ```
#@markdown
#@markdown ### ¿Qué es?
#@markdown El número de **iteraciones** determina cuántas veces el algoritmo ajusta los pesos.
#@markdown
#@markdown ### Proceso en cada iteración:
#@markdown ```
#@markdown Iteración 1: w₁ = w₀ - α·∇L(w₀)
#@markdown Iteración 2: w₂ = w₁ - α·∇L(w₁)
#@markdown Iteración 3: w₃ = w₂ - α·∇L(w₂)
#@markdown ...
#@markdown Iteración N: wₙ = wₙ₋₁ - α·∇L(wₙ₋₁)
#@markdown ```
#@markdown
#@markdown ### ¿Cuántas necesitas?
#@markdown
#@markdown **Depende de α y complejidad:**
#@markdown
#@markdown | Tasa aprendizaje (α) | Iteraciones típicas | Tiempo |
#@markdown |:---------------------|:--------------------|:-------|
#@markdown | 0.0001 (muy pequeña) | 10,000 - 50,000 | Lento ⏱️⏱️⏱️ |
#@markdown | 0.001 | 5,000 - 10,000 | Moderado ⏱️⏱️ |
#@markdown | 0.01 (común) | 1,000 - 5,000 | Rápido ⏱️ |
#@markdown | 0.1 (grande) | 500 - 2,000 | Muy rápido ⚡ |
#@markdown
#@markdown ### Señales de cuándo parar:
#@markdown
#@markdown **Muy pocas iteraciones:**
#@markdown ```
#@markdown Síntomas:
#@markdown - Pérdida todavía está bajando al final
#@markdown - La curva de pérdida no se aplana
#@markdown - MSE test podría mejorar con más entrenamiento
#@markdown
#@markdown Ejemplo: Usar 100 iteraciones con α=0.001
#@markdown Solución: Aumentar iteraciones o aumentar α
#@markdown ```
#@markdown
#@markdown **Suficientes iteraciones:**
#@markdown ```
#@markdown Señales:
#@markdown - Pérdida se vuelve casi plana (cambios < 0.0001)
#@markdown - Últimas 500 iteraciones muestran poca mejora
#@markdown - MSE test se estabiliza
#@markdown
#@markdown ✅ Entrenamiento completo
#@markdown ```
#@markdown
#@markdown **Demasiadas iteraciones:**
#@markdown ```
#@markdown Síntomas:
#@markdown - Pérdida train muy baja, test empieza a SUBIR
#@markdown - Diferencia train-test aumenta con tiempo
#@markdown - Sobreajuste progresivo
#@markdown
#@markdown ⚠️ Early stopping recomendado
#@markdown ```
#@markdown
#@markdown ### Técnica: Early Stopping
#@markdown
#@markdown ```python
#@markdown mejor_perdida_test = infinito
#@markdown paciencia = 0
#@markdown
#@markdown for iteracion in range(max_iteraciones):
#@markdown     entrenar_una_epoca()
#@markdown     perdida_test_actual = evaluar_test()
#@markdown
#@markdown     if perdida_test_actual < mejor_perdida_test:
#@markdown         mejor_perdida_test = perdida_test_actual
#@markdown         guardar_modelo()
#@markdown         paciencia = 0
#@markdown     else:
#@markdown         paciencia += 1
#@markdown
#@markdown     if paciencia >= 100:  # Sin mejora en 100 iters
#@markdown         print("Early stopping!")
#@markdown         break
#@markdown ```
#@markdown
#@markdown ### Reglas prácticas:
#@markdown
#@markdown **Configuración inicial rápida (exploración):**
#@markdown ```
#@markdown iteraciones = 500-1000
#@markdown α = 0.1
#@markdown Objetivo: Ver si el modelo funciona
#@markdown ```
#@markdown
#@markdown **Configuración estándar:**
#@markdown ```
#@markdown iteraciones = 2000-5000
#@markdown α = 0.01
#@markdown Objetivo: Balance velocidad-calidad
#@markdown ```
#@markdown
#@markdown **Configuración cuidadosa (producción):**
#@markdown ```
#@markdown iteraciones = 10000+
#@markdown α = 0.001-0.01
#@markdown Early stopping activo
#@markdown Objetivo: Mejor modelo posible
#@markdown ```
#@markdown
#@markdown ---
#@markdown
#@markdown ## 🎯 Cómo Ajustar Todos los Hiperparámetros Juntos
#@markdown
#@markdown ### Proceso paso a paso:
#@markdown
#@markdown **Paso 1: Configuración base**
#@markdown ```python
#@markdown grado = 2
#@markdown use_scaling = True  # ¡SIEMPRE!
#@markdown alpha = 0.01
#@markdown iteraciones = 2000
#@markdown ```
#@markdown
#@markdown **Paso 2: Entrenar y observar la curva de pérdida**
#@markdown ```
#@markdown Si pérdida oscila → Reducir alpha ÷10
#@markdown Si pérdida baja muy lento → Aumentar alpha ×2
#@markdown Si pérdida se aplana rápido → Todo bien ✅
#@markdown ```
#@markdown
#@markdown **Paso 3: Evaluar en test**
#@markdown ```
#@markdown Si MSE_train << MSE_test → Sobreajuste
#@markdown   Solución: Reducir grado o usar regularización
#@markdown
#@markdown Si MSE_train ≈ MSE_test pero ambos altos → Subajuste
#@markdown   Solución: Aumentar grado
#@markdown ```
#@markdown
#@markdown **Paso 4: Búsqueda sistemática del grado**
#@markdown ```python
#@markdown for grado in [1, 2, 3, 4, 5]:
#@markdown     entrenar_con_grado(grado)
#@markdown     evaluar_test()
#@markdown elegir_grado_con_menor_mse_test()
#@markdown ```
#@markdown
#@markdown **Paso 5: Ajuste fino**
#@markdown ```
#@markdown Con el mejor grado encontrado:
#@markdown - Probar α ligeramente diferentes
#@markdown - Aumentar iteraciones si no convergió
#@markdown - Implementar early stopping
#@markdown ```
#@markdown
#@markdown ### Tabla de troubleshooting:
#@markdown
#@markdown | Problema | Posible causa | Solución |
#@markdown |:---------|:--------------|:---------|
#@markdown | Pérdida = NaN | α muy grande | α ÷ 10 |
#@markdown | Pérdida casi no baja | α muy pequeña | α × 5 |
#@markdown | Pérdida oscila | α grande | α ÷ 2 |
#@markdown | MSE test >> MSE train | Sobreajuste | Reducir grado |
#@markdown | MSE test ≈ MSE train (ambos altos) | Subajuste | Aumentar grado |
#@markdown | Entrenamiento muy lento | α pequeña | α × 2 o menos iters |
#@markdown | Curva zigzagueante | Grado muy alto | Reducir grado |
#@markdown
#@markdown ---
#@markdown
#@markdown ## 📋 Resumen Ejecutivo: Valores Recomendados
#@markdown
#@markdown ### Para empezar (configuración segura):
#@markdown ```python
#@markdown grado_polinomio = 2           # Cuadrático, suficiente para muchos casos
#@markdown use_scaling_poly = True       # SIEMPRE activar
#@markdown poly_alpha = 0.01             # Balance velocidad-estabilidad
#@markdown poly_iters = 2000             # Suficiente para convergencia
#@markdown ```
#@markdown
#@markdown ### Para optimizar:
#@markdown ```python
#@markdown # 1. Probar diferentes grados
#@markdown grados_a_probar = [1, 2, 3, 4, 5]
#@markdown
#@markdown # 2. Para cada grado, ajustar alpha si es necesario
#@markdown alphas_a_probar = [0.001, 0.01, 0.1]
#@markdown
#@markdown # 3. Usar el mejor según MSE en test
#@markdown ```
#@markdown
#@markdown ### Según tu objetivo:
#@markdown
#@markdown | Objetivo | Grado | α | Iteraciones |
#@markdown |:---------|:------|:--|:------------|
#@markdown | **Prototipo rápido** | 2 | 0.1 | 500 |
#@markdown | **Balance estándar** | 2-3 | 0.01 | 2000 |
#@markdown | **Máxima precisión** | Buscar óptimo | 0.001-0.01 | 5000+ con early stopping |
#@markdown | **Datos muy complejos** | 3-4 | 0.01 | 3000 |
#@markdown | **Pocos datos** | 2 | 0.01 | 1000 |
#@markdown
#@markdown ---
#@markdown
#@markdown ## 💡 Consejos Finales
#@markdown
#@markdown 1. **SIEMPRE escala tus features** antes de crear polinomios (use_scaling = True)
#@markdown 2. **Empieza simple** (grado 2) y aumenta solo si es necesario
#@markdown 3. **Observa la curva de pérdida** - te dice si α es apropiada
#@markdown 4. **Valida en test** - el MSE de test es tu métrica clave
#@markdown 5. **No te obsesiones** - un grado 3 bien entrenado suele ser suficiente
#@markdown 6. **Usa regularización** si el grado alto es inevitable (Ridge, Lasso)
#@markdown 7. **Documenta tus experimentos** - qué funcionó y qué no
#@markdown
#@markdown **Recuerda:** No existe una configuración perfecta universal. Los hiperparámetros óptimos dependen de tus datos específicos. ¡Experimenta sistemáticamente!

In [ ]:
#@title 🧮 Regresión Polinomial — Implementación desde cero

#@markdown **Configuración del modelo:**
#@markdown - Selecciona el grado del polinomio
#@markdown - Activa/desactiva escalado
#@markdown - Ajusta hiperparámetros de entrenamiento

grado_polinomio = 1  #@param {type:"slider", min:1, max:4, step:1}
use_scaling_poly = True  #@param {type:"boolean"}
poly_alpha = 0.001  #@param {type:"number"}
poly_iters = 2000  #@param {type:"integer"}

print("=" * 70)
print("CONFIGURACIÓN DEL MODELO POLINOMIAL")
print("=" * 70)
print(f"Grado del polinomio: {grado_polinomio}")
print(f"Usar escalado: {use_scaling_poly}")
print(f"Tasa de aprendizaje: {poly_alpha}")
print(f"Iteraciones: {poly_iters}")

# ADVERTENCIA sobre configuración
if grado_polinomio >= 3:
    print(f"\n⚠️  ADVERTENCIA: Grado {grado_polinomio} generará MUCHAS features")
    print(f"   Recomendación: Usar α muy pequeña (0.0001-0.001)")

if poly_alpha > 0.01 and grado_polinomio >= 3:
    print(f"\n❌ ERROR POTENCIAL: α={poly_alpha} es MUY ALTA para grado {grado_polinomio}")
    print(f"   El modelo probablemente divergirá (NaN)")
    print(f"   Usa α ≤ 0.001 para grado ≥ 3")

print()

# Copias de datos
from copy import deepcopy
Xtr, Xte = deepcopy(Xr_train), deepcopy(Xr_test)
ytr, yte = deepcopy(yr_train).values.reshape(-1, 1), deepcopy(yr_test).values.reshape(-1, 1)

# Paso 1: Escalar si es necesario
scaler_poly = None
if use_scaling_poly:
    scaler_poly = StandardScaler().fit(Xtr)
    Xtr = scaler_poly.transform(Xtr)
    Xte = scaler_poly.transform(Xte)
    print("✅ Features escaladas (StandardScaler)")
else:
    print("⚠️  Features NO escaladas - ALTO riesgo de overflow")

# Paso 2: Crear features polinomiales
print(f"\n📊 Creando features polinomiales de grado {grado_polinomio}...")
print(f"   Features originales: {Xtr.shape[1]}")

Xtr_poly = crear_features_polinomiales(Xtr, grado=grado_polinomio)
Xte_poly = crear_features_polinomiales(Xte, grado=grado_polinomio)

print(f"   Features polinomiales: {Xtr_poly.shape[1]}")
print(f"   Factor de expansión: {Xtr_poly.shape[1] / Xtr.shape[1]:.1f}x")

# ADVERTENCIA si hay demasiadas features
if Xtr_poly.shape[1] > 500:
    print(f"\n⚠️  MUCHAS features ({Xtr_poly.shape[1]})!")
    print(f"   Recomendaciones:")
    print(f"   - Usar α MUY pequeña (0.0001-0.001)")
    print(f"   - Considerar reducir grado o número de features originales")
    print(f"   - Usar regularización (Ridge)")

# Paso 3: Agregar bias
Xtr_poly_b = add_bias(Xtr_poly)
Xte_poly_b = add_bias(Xte_poly)

# Paso 4: Inicializar pesos
N, d_poly = Xtr_poly_b.shape
w_b_poly = np.zeros((d_poly, 1))

print(f"\n🎯 Dimensiones finales:")
print(f"   X_train_poly: {Xtr_poly_b.shape}")
print(f"   y_train: {ytr.shape}")
print(f"   Pesos iniciales: {w_b_poly.shape}")

# Ajuste automático de α si es necesario
alpha_ajustado = poly_alpha
if d_poly > 500 and poly_alpha > 0.001:
    alpha_ajustado = 0.001
    print(f"\n⚙️  AUTO-AJUSTE: α reducida de {poly_alpha} a {alpha_ajustado}")
    print(f"   Razón: Demasiadas features ({d_poly})")

# Paso 5: Entrenamiento con Gradiente Descendente
print(f"\n🔄 Entrenando modelo polinomial (grado {grado_polinomio})...")

loss_hist_poly = []
diverged = False

for it in range(poly_iters):
    # Forward pass
    y_pred = Xtr_poly_b @ w_b_poly
    residual = y_pred - ytr

    # Detectar divergencia temprana
    if np.any(np.isnan(residual)) or np.any(np.isinf(residual)):
        print(f"\n❌ DIVERGENCIA detectada en iteración {it}")
        print(f"   Los pesos explotaron (NaN o Inf)")
        print(f"   Causa probable: Tasa de aprendizaje MUY ALTA")
        print(f"   Solución: Reducir α al menos 10x")
        diverged = True
        break

    # Calcular gradiente
    grad = (2 / N) * (Xtr_poly_b.T @ residual)

    # Actualizar pesos con gradient clipping (prevenir explosión)
    grad_clipped = np.clip(grad, -1e6, 1e6)
    w_b_poly -= alpha_ajustado * grad_clipped

    # Calcular pérdida
    L = (residual ** 2).sum() / N
    loss_hist_poly.append(L)

    # Detectar explosión de pérdida
    if L > 1e10:
        print(f"\n❌ PÉRDIDA EXPLOTÓ en iteración {it}: {L:.2e}")
        print(f"   Tasa de aprendizaje demasiado alta")
        diverged = True
        break

    # Progreso cada 20%
    if (it + 1) % (poly_iters // 5) == 0:
        print(f"   Iteración {it+1}/{poly_iters} - Pérdida: {L:.6f}")

if not diverged:
    print(f"\n✅ Entrenamiento completado exitosamente")
    print(f"   Pérdida final: {loss_hist_poly[-1]:.6f}")
    print(f"   Pérdida inicial: {loss_hist_poly[0]:.6f}")
    print(f"   Reducción: {(1 - loss_hist_poly[-1]/loss_hist_poly[0])*100:.2f}%")
    print(f"   Pesos aprendidos (primeros 5): {w_b_poly.ravel()[:5]}")

    # Paso 6: Predicciones
    y_pred_train = Xtr_poly_b @ w_b_poly
    y_pred_test = Xte_poly_b @ w_b_poly

    # Calcular errores
    mse_train = ((y_pred_train - ytr) ** 2).mean()
    mse_test = ((y_pred_test - yte) ** 2).mean()

    print(f"\n📊 RESULTADOS:")
    print(f"   MSE Train: {mse_train:.6f}")
    print(f"   MSE Test:  {mse_test:.6f}")
    print(f"   Relación Test/Train: {mse_test/mse_train:.2f}x")

    if mse_test > mse_train * 2.0:
        print(f"   ⚠️  SOBREAJUSTE severo (test >> train)")
        print(f"      Solución: Reducir grado o usar regularización")
    elif mse_test > mse_train * 1.3:
        print(f"   ⚠️  Posible sobreajuste moderado")
        print(f"      Considera: Reducir grado o agregar más datos")
    elif mse_test < mse_train * 0.8:
        print(f"   ⚠️  Comportamiento inusual (test << train)")
    else:
        print(f"   ✅ Buen balance entre train y test")

    # Visualización de la curva de pérdida
    plt.figure(figsize=(14, 5))

    plt.subplot(1, 3, 1)
    plt.plot(loss_hist_poly, linewidth=2)
    plt.xlabel("Iteración", fontsize=11)
    plt.ylabel("MSE", fontsize=11)
    plt.title(f"Pérdida durante entrenamiento\n(Polinomio grado {grado_polinomio})", fontsize=12)
    plt.grid(True, alpha=0.3)

    plt.subplot(1, 3, 2)
    start_idx = max(0, len(loss_hist_poly) - 500)
    plt.plot(range(start_idx, len(loss_hist_poly)),
             loss_hist_poly[start_idx:], linewidth=2, color='orange')
    plt.xlabel("Iteración", fontsize=11)
    plt.ylabel("MSE", fontsize=11)
    plt.title("Últimas iteraciones (zoom)", fontsize=12)
    plt.grid(True, alpha=0.3)

    # CORREGIDO: Gráfico de barras en lugar de plot
    plt.subplot(1, 3, 3)
    categories = ['Train', 'Test']
    values = [mse_train, mse_test]
    colors = ['blue', 'red' if mse_test > mse_train * 1.5 else 'orange']

    bars = plt.bar(categories, values, color=colors, alpha=0.7, edgecolor='black', linewidth=2)
    plt.ylabel("MSE", fontsize=11)
    plt.title("Comparación Train vs Test", fontsize=12)
    plt.grid(True, alpha=0.3, axis='y')

    # Añadir valores en las barras
    for bar, val in zip(bars, values):
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2., height,
                f'{val:.2f}',
                ha='center', va='bottom', fontsize=10, fontweight='bold')

    # Línea de referencia
    if mse_test > mse_train:
        plt.axhline(y=mse_train, color='blue', linestyle='--', alpha=0.3, label='MSE Train')
        plt.legend()

    plt.tight_layout()
    plt.show()

    # Comparación: predicciones vs valores reales (sample)
    print("\n" + "=" * 70)
    print("COMPARACIÓN: Predicciones vs Valores Reales (primeros 10 casos de test)")
    print("=" * 70)
    print(f"{'Real':<12} {'Predicción':<12} {'Error':<12} {'Error %':<12}")
    print("-" * 70)

    for i in range(min(10, len(yte))):
        real = yte[i, 0]
        pred = y_pred_test[i, 0]
        error = abs(real - pred)
        error_pct = (error / real * 100) if real != 0 else 0
        print(f"{real:<12.4f} {pred:<12.4f} {error:<12.4f} {error_pct:<12.2f}%")

    # Análisis del sobreajuste si lo hay
    if mse_test > mse_train * 1.5:
        print("\n" + "=" * 70)
        print("🔍 ANÁLISIS DE SOBREAJUSTE")
        print("=" * 70)
        print(f"\n⚠️  El modelo tiene sobreajuste (test {mse_test/mse_train:.2f}x > train)")
        print("\n📚 ¿Qué significa esto?")
        print("   El modelo 'memorizó' los datos de entrenamiento pero no generaliza bien")
        print("\n🔧 Soluciones posibles:")
        print(f"   1️⃣  Reducir grado: Usar grado {max(1, grado_polinomio-1)} en lugar de {grado_polinomio}")
        print("   2️⃣  Usar regularización: Ridge (L2) o Lasso (L1)")
        print("   3️⃣  Obtener más datos de entrenamiento")
        print("   4️⃣  Usar validación cruzada para elegir mejor grado")
        print("\n💡 Próximo paso:")
        print("   Entrena el modelo con sklearn usando Ridge para comparar")

else:
    print("\n" + "=" * 70)
    print("❌ ENTRENAMIENTO FALLIDO")
    print("=" * 70)
    print("\n🔧 DIAGNÓSTICO Y SOLUCIONES:")
    print("\n1️⃣  Reducir la tasa de aprendizaje:")
    print(f"   Actual: α = {poly_alpha}")
    print(f"   Prueba: α = {poly_alpha/10:.6f} o menor")

    print("\n2️⃣  Reducir el grado del polinomio:")
    print(f"   Actual: grado = {grado_polinomio}")
    print(f"   Prueba: grado = {max(1, grado_polinomio-1)}")

    print("\n3️⃣  Verificar que el escalado esté activo:")
    print(f"   Actual: use_scaling = {use_scaling_poly}")
    if not use_scaling_poly:
        print(f"   ⚠️  ACTIVAR ESCALADO es CRÍTICO")

    print("\n4️⃣  Configuración recomendada para reintentar:")
    print(f"   grado_polinomio = {max(1, grado_polinomio-1)}")
    print(f"   use_scaling_poly = True")
    print(f"   poly_alpha = 0.0001")
    print(f"   poly_iters = 2000")

## regresion logistica polinomial clasificación

In [ ]:
#@markdown # 📚 Regresión Logística Polinomial — Clasificación No Lineal
#@markdown
#@markdown ---
#@markdown
#@markdown ## 🎯 ¿Qué es la Regresión Logística Polinomial?
#@markdown
#@markdown Combina lo mejor de dos mundos:
#@markdown - **Features polinomiales**: Capturan relaciones no lineales entre características
#@markdown - **Regresión logística**: Predice probabilidades de clases
#@markdown
#@markdown **Diferencia clave con regresión logística lineal:**
#@markdown
#@markdown **Logística Lineal:**
#@markdown $$z = w_1x_1 + w_2x_2 + b$$
#@markdown - Frontera de decisión: **línea recta** (o hiperplano)
#@markdown - Limitada a separaciones lineales
#@markdown
#@markdown **Logística Polinomial (grado 2):**
#@markdown $$z = w_1x_1 + w_2x_2 + w_3x_1^2 + w_4x_1x_2 + w_5x_2^2 + b$$
#@markdown - Frontera de decisión: **curva** (elipse, parábola, etc.)
#@markdown - Puede separar clases con patrones curvos
#@markdown
#@markdown ---
#@markdown
#@markdown ## 🧩 El proceso matemático
#@markdown
#@markdown ### Paso 1: Transformación polinomial
#@markdown
#@markdown Entrada original: $\mathbf{x} = [x_1, x_2]$
#@markdown
#@markdown Features polinomiales (grado 2): $\mathbf{x}_{\text{poly}} = [x_1, x_2, x_1^2, x_1x_2, x_2^2]$
#@markdown
#@markdown ### Paso 2: Combinación lineal (con features expandidas)
#@markdown
#@markdown $$z_i = \mathbf{w}^\top \mathbf{x}_{\text{poly},i} + b$$
#@markdown
#@markdown ### Paso 3: Función sigmoide (igual que antes)
#@markdown
#@markdown $$\hat{p}_i = \sigma(z_i) = \frac{1}{1 + e^{-z_i}}$$
#@markdown
#@markdown ### Paso 4: Función de pérdida (log-loss, igual que antes)
#@markdown
#@markdown $$\mathcal{L}(\mathbf{w}, b) = -\frac{1}{N}\sum_{i=1}^N \Big[y_i \log(\hat{p}_i) + (1-y_i)\log(1-\hat{p}_i)\Big]$$
#@markdown
#@markdown ### Paso 5: Gradiente (misma fórmula, diferentes features)
#@markdown
#@markdown $$\frac{\partial \mathcal{L}}{\partial \mathbf{w}_b} = \frac{1}{N} \mathbf{X}_{\text{poly},b}^\top (\hat{\mathbf{p}} - \mathbf{y})$$
#@markdown
#@markdown ### Paso 6: Actualización (igual que antes)
#@markdown
#@markdown $$\mathbf{w}_b \leftarrow \mathbf{w}_b - \eta \frac{\partial \mathcal{L}}{\partial \mathbf{w}_b}$$
#@markdown
#@markdown ---
#@markdown
#@markdown ## 🆚 Comparación visual de fronteras de decisión
#@markdown
#@markdown ### Logística Lineal (grado 1):
#@markdown ```
#@markdown     Clase 1 (○)         |      Clase 0 (×)
#@markdown                         |
#@markdown        ○  ○            |        × ×
#@markdown      ○      ○          |      ×     ×
#@markdown        ○  ○            |        × ×
#@markdown                         |
#@markdown   ─────────────────────┼────────────────
#@markdown                Línea recta
#@markdown ```
#@markdown
#@markdown ### Logística Polinomial (grado 2):
#@markdown ```
#@markdown     Clase 1 (○)         /‾‾\      Clase 0 (×)
#@markdown                        /    \
#@markdown        ○  ○           |  ×  |      × ×
#@markdown      ○      ○         |  ×  |    ×     ×
#@markdown        ○  ○           |  ×  |      × ×
#@markdown                        \    /
#@markdown                         \__/
#@markdown                    Curva (círculo/elipse)
#@markdown ```
#@markdown
#@markdown ---
#@markdown
#@markdown ## 🎯 ¿Cuándo usar Logística Polinomial?
#@markdown
#@markdown **Usa Logística Polinomial cuando:**
#@markdown
#@markdown 1. **Las clases NO son linealmente separables**
#@markdown    - Ejemplo: Clasificar puntos dentro vs fuera de un círculo
#@markdown    - Logística lineal: FALLA ❌
#@markdown    - Logística polinomial grado 2: FUNCIONA ✅
#@markdown
#@markdown 2. **Relaciones cuadráticas entre features**
#@markdown    - Ejemplo: Riesgo = edad × colesterol (interacción)
#@markdown    - El término $x_1 \cdot x_2$ captura esta interacción
#@markdown
#@markdown 3. **Patrones curvos en los datos**
#@markdown    - Ejemplo: Decisión de compra según precio e ingreso
#@markdown    - Relación en forma de U invertida
#@markdown
#@markdown **Mantén Logística Lineal cuando:**
#@markdown - Las clases YA están bien separadas linealmente
#@markdown - Pocas muestras (riesgo de sobreajuste con polinomios)
#@markdown - Necesitas máxima interpretabilidad
#@markdown
#@markdown ---
#@markdown
#@markdown ## ⚖️ Hiperparámetros para Logística Polinomial
#@markdown
#@markdown | Hiperparámetro | Valores típicos | Efecto |
#@markdown |:---------------|:----------------|:-------|
#@markdown | **Grado** | 2-3 | Complejidad de la frontera |
#@markdown | **α (learning rate)** | 0.001-0.1 | Velocidad de convergencia |
#@markdown | **Iteraciones** | 2000-5000 | Tiempo de entrenamiento |
#@markdown | **Escalado** | Siempre True | Estabilidad numérica |
#@markdown
#@markdown **Advertencia:** Grado ≥ 4 raramente necesario y causa sobreajuste
#@markdown
#@markdown ---
#@markdown
#@markdown ## 💡 Ventajas y Desventajas
#@markdown
#@markdown **✅ Ventajas:**
#@markdown - Captura patrones no lineales
#@markdown - Mantiene interpretabilidad probabilística (0-1)
#@markdown - Fácil de implementar (reutiliza código logístico)
#@markdown - Más flexible que logística lineal
#@markdown
#@markdown **⚠️ Desventajas:**
#@markdown - Muchas más features → más lento
#@markdown - Riesgo de sobreajuste con grado alto
#@markdown - Menos interpretable que modelo lineal
#@markdown - Necesita regularización si hay muchas features
#@markdown
#@markdown ---
#@markdown
#@markdown ## 🔑 Resumen
#@markdown
#@markdown **Concepto clave:** Regresión Logística Polinomial = Logística con features transformadas
#@markdown
#@markdown **Proceso:**
#@markdown 1. Escalar features originales
#@markdown 2. Crear features polinomiales
#@markdown 3. Aplicar regresión logística estándar
#@markdown 4. Obtener frontera de decisión curva
#@markdown
#@markdown **Frontera de decisión:** Donde $\hat{p} = 0.5$, que ahora es una **curva** en lugar de línea recta

In [ ]:
#@title 🔧 Funciones auxiliares (reutilizadas)

def sigmoid(z):
    """Función sigmoide con clip para estabilidad"""
    return 1.0 / (1.0 + np.exp(-np.clip(z, -500, 500)))

def crear_features_polinomiales(X, grado=2):
    """Crea features polinomiales"""
    from sklearn.preprocessing import PolynomialFeatures
    poly = PolynomialFeatures(degree=grado, include_bias=False)
    return poly.fit_transform(X)

def add_bias(X):
    """Agrega columna de 1s para el sesgo"""
    return np.c_[X, np.ones(X.shape[0])]

print("✅ Funciones auxiliares cargadas")

In [ ]:
#@title 🧮 Regresión Logística Polinomial — Implementación desde cero

#@markdown **Configuración del modelo:**
#@markdown - Selecciona el grado del polinomio
#@markdown - Activa/desactiva escalado
#@markdown - Ajusta hiperparámetros de entrenamiento

grado_poly_log = 2  #@param {type:"slider", min:1, max:4, step:1}
use_scaling_poly_log = True  #@param {type:"boolean"}
poly_log_alpha = 0.01  #@param {type:"number"}
poly_log_iters = 3000  #@param {type:"integer"}

print("=" * 70)
print("CONFIGURACIÓN DEL MODELO LOGÍSTICO POLINOMIAL")
print("=" * 70)
print(f"Grado del polinomio: {grado_poly_log}")
print(f"Usar escalado: {use_scaling_poly_log}")
print(f"Tasa de aprendizaje: {poly_log_alpha}")
print(f"Iteraciones: {poly_log_iters}")

# ADVERTENCIA sobre configuración
if grado_poly_log >= 3:
    print(f"\n⚠️  ADVERTENCIA: Grado {grado_poly_log} puede causar sobreajuste")
    print(f"   Observa la diferencia entre accuracy train y test")

print()

# Copias de datos de clasificación
from copy import deepcopy
Xct, Xce = deepcopy(Xc_train), deepcopy(Xc_test)
yct = deepcopy(yc_train).values.reshape(-1, 1)
yce = deepcopy(yc_test).values.reshape(-1, 1)

# Paso 1: Escalar si es necesario
scaler_poly_log = None
if use_scaling_poly_log:
    scaler_poly_log = StandardScaler().fit(Xct)
    Xct = scaler_poly_log.transform(Xct)
    Xce = scaler_poly_log.transform(Xce)
    print("✅ Features escaladas (StandardScaler)")
else:
    print("⚠️  Features NO escaladas - puede afectar convergencia")

# Paso 2: Crear features polinomiales
print(f"\n📊 Creando features polinomiales de grado {grado_poly_log}...")
print(f"   Features originales: {Xct.shape[1]}")

Xct_poly = crear_features_polinomiales(Xct, grado=grado_poly_log)
Xce_poly = crear_features_polinomiales(Xce, grado=grado_poly_log)

print(f"   Features polinomiales: {Xct_poly.shape[1]}")
print(f"   Factor de expansión: {Xct_poly.shape[1] / Xct.shape[1]:.1f}x")

# Ajuste de α si hay muchas features
if Xct_poly.shape[1] > 300 and poly_log_alpha > 0.01:
    print(f"\n⚙️  Reduciendo α de {poly_log_alpha} a 0.01 (muchas features)")
    poly_log_alpha = 0.01

# Paso 3: Agregar bias
Xct_poly_b = add_bias(Xct_poly)
Xce_poly_b = add_bias(Xce_poly)

# Paso 4: Inicializar pesos
N_log, d_poly_log = Xct_poly_b.shape
w_b_poly_log = np.zeros((d_poly_log, 1))

print(f"\n🎯 Dimensiones finales:")
print(f"   X_train_poly: {Xct_poly_b.shape}")
print(f"   y_train: {yct.shape}")
print(f"   Pesos iniciales: {w_b_poly_log.shape}")

# Paso 5: Entrenamiento con Gradiente Descendente
print(f"\n🔄 Entrenando modelo logístico polinomial (grado {grado_poly_log})...")

loss_hist_poly_log = []
diverged = False

for it in range(poly_log_iters):
    # Forward pass
    z = Xct_poly_b @ w_b_poly_log
    p_hat = sigmoid(z)

    # Detectar divergencia
    if np.any(np.isnan(p_hat)) or np.any(np.isinf(p_hat)):
        print(f"\n❌ DIVERGENCIA en iteración {it}")
        diverged = True
        break

    # Calcular gradiente
    grad = (Xct_poly_b.T @ (p_hat - yct)) / N_log

    # Actualizar pesos
    w_b_poly_log -= poly_log_alpha * grad

    # Calcular pérdida (log-loss con estabilidad numérica)
    eps = 1e-15
    p_hat_clipped = np.clip(p_hat, eps, 1 - eps)
    L = -np.mean(yct * np.log(p_hat_clipped) + (1 - yct) * np.log(1 - p_hat_clipped))
    loss_hist_poly_log.append(L)

    # Progreso cada 20%
    if (it + 1) % (poly_log_iters // 5) == 0:
        print(f"   Iteración {it+1}/{poly_log_iters} - Pérdida: {L:.6f}")

if not diverged:
    print(f"\n✅ Entrenamiento completado exitosamente")
    print(f"   Pérdida final: {loss_hist_poly_log[-1]:.6f}")
    print(f"   Pérdida inicial: {loss_hist_poly_log[0]:.6f}")
    print(f"   Reducción: {(1 - loss_hist_poly_log[-1]/loss_hist_poly_log[0])*100:.2f}%")

    # Paso 6: Predicciones
    # Train
    z_train = Xct_poly_b @ w_b_poly_log
    p_hat_train = sigmoid(z_train)
    y_pred_train_poly = (p_hat_train >= 0.5).astype(int)

    # Test
    z_test = Xce_poly_b @ w_b_poly_log
    p_hat_test = sigmoid(z_test)
    y_pred_test_poly = (p_hat_test >= 0.5).astype(int)

    # Calcular accuracy
    acc_train_poly = (y_pred_train_poly == yct).mean()
    acc_test_poly = (y_pred_test_poly == yce).mean()

    print(f"\n📊 RESULTADOS:")
    print(f"   Accuracy Train: {acc_train_poly:.4f} ({acc_train_poly*100:.2f}%)")
    print(f"   Accuracy Test:  {acc_test_poly:.4f} ({acc_test_poly*100:.2f}%)")
    print(f"   Diferencia: {abs(acc_train_poly - acc_test_poly):.4f}")

    # Análisis de sobreajuste
    if acc_train_poly > acc_test_poly + 0.05:
        print(f"   ⚠️  Posible sobreajuste (train >> test)")
        print(f"      Considera: Reducir grado o usar regularización")
    elif acc_train_poly < acc_test_poly:
        print(f"   ✅ Excelente generalización (test ≥ train)")
    else:
        print(f"   ✅ Buen balance entre train y test")

    # Comparación con logística lineal (grado 1)
    if grado_poly_log > 1:
        print(f"\n💡 Comparación con Logística Lineal:")
        print(f"   Grado actual: {grado_poly_log}")
        print(f"   Para ver si el polinomio ayuda, compara con grado=1")

    # Visualización
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    # Gráfica 1: Curva de pérdida
    axes[0].plot(loss_hist_poly_log, linewidth=2)
    axes[0].set_xlabel("Iteración", fontsize=11)
    axes[0].set_ylabel("Log-Loss", fontsize=11)
    axes[0].set_title(f"Pérdida durante entrenamiento\n(Polinomio grado {grado_poly_log})", fontsize=12)
    axes[0].grid(True, alpha=0.3)

    # Gráfica 2: Zoom últimas iteraciones
    start_idx = max(0, len(loss_hist_poly_log) - 500)
    axes[1].plot(range(start_idx, len(loss_hist_poly_log)),
                 loss_hist_poly_log[start_idx:], linewidth=2, color='orange')
    axes[1].set_xlabel("Iteración", fontsize=11)
    axes[1].set_ylabel("Log-Loss", fontsize=11)
    axes[1].set_title("Últimas iteraciones (zoom)", fontsize=12)
    axes[1].grid(True, alpha=0.3)

    # Gráfica 3: Comparación accuracy
    categories = ['Train', 'Test']
    values = [acc_train_poly * 100, acc_test_poly * 100]
    colors = ['blue', 'red' if acc_train_poly > acc_test_poly + 0.05 else 'green']

    bars = axes[2].bar(categories, values, color=colors, alpha=0.7, edgecolor='black', linewidth=2)
    axes[2].set_ylabel("Accuracy (%)", fontsize=11)
    axes[2].set_title("Comparación Train vs Test", fontsize=12)
    axes[2].set_ylim([0, 105])
    axes[2].grid(True, alpha=0.3, axis='y')

    # Añadir valores en las barras
    for bar, val in zip(bars, values):
        height = bar.get_height()
        axes[2].text(bar.get_x() + bar.get_width()/2., height + 1,
                    f'{val:.1f}%',
                    ha='center', va='bottom', fontsize=11, fontweight='bold')

    plt.tight_layout()
    plt.show()

    # Matriz de confusión
    from sklearn.metrics import confusion_matrix

    cm_train = confusion_matrix(yct, y_pred_train_poly)
    cm_test = confusion_matrix(yce, y_pred_test_poly)

    print("\n" + "=" * 70)
    print("MATRIZ DE CONFUSIÓN - TRAIN")
    print("=" * 70)
    print(cm_train)
    print(f"\nVerdaderos Negativos (Maligno correcto): {cm_train[0,0]}")
    print(f"Falsos Positivos (Maligno → Benigno):   {cm_train[0,1]} ⚠️")
    print(f"Falsos Negativos (Benigno → Maligno):   {cm_train[1,0]}")
    print(f"Verdaderos Positivos (Benigno correcto): {cm_train[1,1]}")

    print("\n" + "=" * 70)
    print("MATRIZ DE CONFUSIÓN - TEST")
    print("=" * 70)
    print(cm_test)
    print(f"\nVerdaderos Negativos (Maligno correcto): {cm_test[0,0]}")
    print(f"Falsos Positivos (Maligno → Benigno):   {cm_test[0,1]} ⚠️ PELIGROSO")
    print(f"Falsos Negativos (Benigno → Maligno):   {cm_test[1,0]}")
    print(f"Verdaderos Positivos (Benigno correcto): {cm_test[1,1]}")

    # Ejemplos de predicciones
    print("\n" + "=" * 70)
    print("EJEMPLOS DE PREDICCIONES (primeros 10 casos de test)")
    print("=" * 70)
    print(f"{'Real':<15} {'Predicción':<15} {'Prob(Benigno)':<20} {'Resultado':<15}")
    print("-" * 70)

    for i in range(min(10, len(yce))):
        real = yce[i, 0]
        pred = y_pred_test_poly[i, 0]
        prob = p_hat_test[i, 0]

        real_str = "Benigno (1)" if real == 1 else "Maligno (0)"
        pred_str = "Benigno (1)" if pred == 1 else "Maligno (0)"
        resultado = "✓ CORRECTO" if real == pred else "✗ ERROR"

        print(f"{real_str:<15} {pred_str:<15} {prob:<20.4f} {resultado:<15}")

else:
    print("\n❌ Entrenamiento fallido - Ajusta hiperparámetros")
#@markdown # ⚙️ Hiperparámetros en Regresión Logística Polinomial — Guía Completa
#@markdown
#@markdown ---
#@markdown
#@markdown ## 🎛️ ¿Qué son los hiperparámetros en clasificación?
#@markdown
#@markdown Los **hiperparámetros** son configuraciones que TÚ decides ANTES de entrenar el modelo de clasificación.
#@markdown
#@markdown **Diferencia con regresión:**
#@markdown - En **regresión**: Predecimos valores continuos (ej. precio, temperatura)
#@markdown - En **clasificación**: Predecimos categorías discretas (ej. benigno/maligno)
#@markdown
#@markdown **Los hiperparámetros afectan:**
#@markdown - Qué tan compleja es la frontera de decisión
#@markdown - Qué tan rápido aprende el modelo
#@markdown - Si el modelo sobreajusta o subajusta
#@markdown
#@markdown ---
#@markdown
#@markdown ## 📊 Hiperparámetro 1: Grado del Polinomio (degree)
#@markdown
#@markdown ```python
#@markdown grado_poly_log = 2  # ¿Qué tan curva puede ser la frontera de decisión?
#@markdown ```
#@markdown
#@markdown ### ¿Qué es?
#@markdown El **grado** determina la **complejidad de la frontera** que separa las clases.
#@markdown
#@markdown ### Ejemplos visuales de fronteras de decisión:
#@markdown
#@markdown **Grado 1 (Lineal):**
#@markdown ```
#@markdown Frontera: Línea recta (o hiperplano)
#@markdown
#@markdown    Clase 1 (○)    |    Clase 0 (×)
#@markdown                   |
#@markdown       ○  ○        |      × ×
#@markdown     ○      ○      |    ×     ×
#@markdown       ○  ○        |      × ×
#@markdown                   |
#@markdown ──────────────────┼────────────────
#@markdown        Línea recta vertical
#@markdown
#@markdown Ecuación: w₁x₁ + w₂x₂ + b = 0
#@markdown ```
#@markdown
#@markdown **Grado 2 (Cuadrático):**
#@markdown ```
#@markdown Frontera: Curva (círculo, elipse, parábola, hipérbola)
#@markdown
#@markdown         ╱‾‾‾╲
#@markdown        │  ×  │
#@markdown    ○  ○│  ×  │○  ○
#@markdown  ○      │  ×  │      ○
#@markdown    ○  ○│  ×  │○  ○
#@markdown        │  ×  │
#@markdown         ╲___╱
#@markdown        Círculo/Elipse
#@markdown
#@markdown Ecuación: w₁x₁ + w₂x₂ + w₃x₁² + w₄x₁x₂ + w₅x₂² + b = 0
#@markdown ```
#@markdown
#@markdown **Grado 3 (Cúbico):**
#@markdown ```
#@markdown Frontera: Curva compleja (forma de S, múltiples curvaturas)
#@markdown
#@markdown       ╱‾╲    ╱‾╲
#@markdown      │ × │  │ × │
#@markdown   ○ ○│ × │○○│ × │○ ○
#@markdown      │ × │  │ × │
#@markdown       ╲_╱    ╲_╱
#@markdown    Múltiples regiones curvas
#@markdown
#@markdown Ecuación: Incluye términos hasta x³
#@markdown ```
#@markdown
#@markdown ---
#@markdown
#@markdown ### ¿Cómo afecta al modelo?
#@markdown
#@markdown | Grado | Frontera | Cuándo usar | Riesgo |
#@markdown |:------|:---------|:------------|:-------|
#@markdown | **1** | Línea recta | Clases linealmente separables | ❌ Subajuste si no son lineales |
#@markdown | **2** | Curva suave | Patrones circulares, elípticos | ✅ Buen balance generalmente |
#@markdown | **3** | Curva compleja | Múltiples regiones curvas | ⚠️ Puede sobreajustar |
#@markdown | **≥4** | Muy compleja | Casi nunca necesario | ❌ Sobreajuste severo |
#@markdown
#@markdown ---
#@markdown
#@markdown ### Casos de uso por grado:
#@markdown
#@markdown **Grado 1 - Usa cuando:**
#@markdown ```
#@markdown Ejemplo 1: Clasificar emails spam/no-spam por longitud y frecuencia de palabras
#@markdown - Si hay una tendencia clara: "emails largos con muchas palabras clave = spam"
#@markdown - La separación es aproximadamente lineal
#@markdown
#@markdown Ejemplo 2: Aprobar/reprobar examen según horas de estudio
#@markdown - Más horas → más probabilidad de aprobar (relación directa)
#@markdown ```
#@markdown
#@markdown **Grado 2 - Usa cuando:**
#@markdown ```
#@markdown Ejemplo 1: Clasificar tumores benignos/malignos
#@markdown - Tumores muy pequeños: generalmente benignos
#@markdown - Tumores muy grandes: generalmente malignos
#@markdown - Tumores tamaño medio con alta textura: malignos
#@markdown → Frontera en forma de elipse o curva
#@markdown
#@markdown Ejemplo 2: Detectar fraude en transacciones
#@markdown - Monto muy bajo + frecuencia baja: normal
#@markdown - Monto muy alto + frecuencia alta: fraude
#@markdown - Región intermedia: depende de interacciones (término x₁·x₂)
#@markdown ```
#@markdown
#@markdown **Grado 3 - Usa cuando:**
#@markdown ```
#@markdown Ejemplo 1: Clasificar calidad de vino (bueno/malo)
#@markdown - Acidez muy baja: malo
#@markdown - Acidez media + azúcar moderado: bueno
#@markdown - Acidez alta + azúcar bajo: malo
#@markdown - Acidez alta + azúcar alto: bueno
#@markdown → Múltiples regiones no lineales
#@markdown
#@markdown ⚠️ Solo si grado 2 claramente no es suficiente
#@markdown ```
#@markdown
#@markdown ---
#@markdown
#@markdown ### Síntomas de grado incorrecto:
#@markdown
#@markdown **Grado muy BAJO (subajuste):**
#@markdown ```
#@markdown Síntomas:
#@markdown - Accuracy bajo en train Y test (<85%)
#@markdown - Muchos errores obvios
#@markdown - Frontera demasiado simple para los datos
#@markdown - Ejemplos mal clasificados tienen patrón claro
#@markdown
#@markdown Ejemplo visual:
#@markdown    Intentar separar un círculo con una línea recta
#@markdown    ○ ○ ○     |    × × ×
#@markdown  ○   ×   ○   |  ×       ×
#@markdown    ○ ○ ○     |    × × ×
#@markdown    ↑ Línea recta no puede capturar el círculo
#@markdown
#@markdown Solución: Aumentar grado a 2
#@markdown ```
#@markdown
#@markdown **Grado muy ALTO (sobreajuste):**
#@markdown ```
#@markdown Síntomas:
#@markdown - Accuracy train muy alto (>98%) pero test bajo (<90%)
#@markdown - Diferencia train-test > 5%
#@markdown - Frontera hace "zigzag" entre puntos
#@markdown - Memoriza ruido en lugar de patrón real
#@markdown
#@markdown Ejemplo visual:
#@markdown    Frontera que "persigue" cada punto de entrenamiento
#@markdown      ╱╲╱╲╱╲
#@markdown    ○╱×╲○╱×╲○
#@markdown    ╲○╱×╲○╱×
#@markdown     ╲╱╲╱╲╱
#@markdown    ↑ Frontera demasiado compleja
#@markdown
#@markdown Solución: Reducir grado o usar regularización
#@markdown ```
#@markdown
#@markdown ---
#@markdown
#@markdown ### ¿Cómo elegir el grado óptimo?
#@markdown
#@markdown **Método 1: Empezar simple y aumentar gradualmente**
#@markdown ```python
#@markdown # Paso 1: Probar lineal (grado 1)
#@markdown grado = 1
#@markdown entrenar_y_evaluar(grado)
#@markdown
#@markdown # Paso 2: Si accuracy test < 90%, probar grado 2
#@markdown if accuracy_test < 0.90:
#@markdown     grado = 2
#@markdown     entrenar_y_evaluar(grado)
#@markdown
#@markdown # Paso 3: Si mejora ≥2%, considerar grado 3
#@markdown if mejora >= 0.02 and no_hay_sobreajuste:
#@markdown     grado = 3
#@markdown     entrenar_y_evaluar(grado)
#@markdown ```
#@markdown
#@markdown **Método 2: Comparación sistemática**
#@markdown ```python
#@markdown for grado in [1, 2, 3]:
#@markdown     entrenar_modelo(grado)
#@markdown     calcular_accuracy_test()
#@markdown     calcular_diferencia_train_test()
#@markdown
#@markdown # Elegir grado con:
#@markdown # - MAYOR accuracy test
#@markdown # - MENOR diferencia train-test (<5%)
#@markdown ```
#@markdown
#@markdown **Método 3: Validación cruzada** (más confiable)
#@markdown ```python
#@markdown from sklearn.model_selection import cross_val_score
#@markdown
#@markdown for grado in [1, 2, 3]:
#@markdown     scores = cross_val_score(modelo, X, y, cv=5)
#@markdown     print(f"Grado {grado}: {scores.mean():.3f} ± {scores.std():.3f}")
#@markdown
#@markdown # Elegir grado con mejor promedio y menor varianza
#@markdown ```
#@markdown
#@markdown ---
#@markdown
#@markdown ### Regla práctica para este dataset:
#@markdown
#@markdown **Dataset de cáncer (tumores benignos/malignos):**
#@markdown ```
#@markdown Características: tamaño, textura, simetría del tumor
#@markdown
#@markdown Recomendación inicial: Grado 2
#@markdown
#@markdown Razón:
#@markdown - Tumores pequeños y lisos → Benigno
#@markdown - Tumores grandes y rugosos → Maligno
#@markdown - Zona intermedia con interacciones → Requiere términos cuadráticos
#@markdown
#@markdown Expectativa:
#@markdown - Grado 1: ~95% accuracy (probablemente suficiente)
#@markdown - Grado 2: ~97% accuracy (ligera mejora)
#@markdown - Grado 3: ~97.5% accuracy (no justifica complejidad)
#@markdown
#@markdown Decisión: Quedarse con grado 1 o 2 según diferencia
#@markdown ```
#@markdown
#@markdown ---
#@markdown
#@markdown ## 🎓 Hiperparámetro 2: Usar Escalado (use_scaling)
#@markdown
#@markdown ```python
#@markdown use_scaling_poly_log = True  # ¿Normalizar features?
#@markdown ```
#@markdown
#@markdown ### ¿Qué es?
#@markdown El **escalado** transforma todas las características a la misma escala (típicamente media=0, std=1).
#@markdown
#@markdown ### ¿Por qué es CRÍTICO en clasificación polinomial?
#@markdown
#@markdown **Sin escalado - Problema:**
#@markdown ```
#@markdown Feature 1 (tamaño tumor): rango [5, 30] mm
#@markdown Feature 2 (textura): rango [10, 40]
#@markdown
#@markdown Con grado 2:
#@markdown - tamaño²: [25, 900] → rango enorme
#@markdown - textura²: [100, 1600] → aún más enorme
#@markdown - tamaño × textura: [50, 1200]
#@markdown
#@markdown Resultado:
#@markdown - Gradientes explotan (valores gigantes)
#@markdown - Modelo no converge (diverge a NaN/Inf)
#@markdown - Features grandes dominan artificialmente
#@markdown ```
#@markdown
#@markdown **Con escalado - Solución:**
#@markdown ```
#@markdown Feature 1 escalada: rango [-2, +2]
#@markdown Feature 2 escalada: rango [-2, +2]
#@markdown
#@markdown Con grado 2:
#@markdown - feature1²: [0, 4] → controlado
#@markdown - feature2²: [0, 4] → controlado
#@markdown - feature1 × feature2: [-4, 4] → controlado
#@markdown
#@markdown Resultado:
#@markdown - Gradientes estables
#@markdown - Convergencia rápida
#@markdown - Todas las features tienen influencia justa
#@markdown ```
#@markdown
#@markdown ---
#@markdown
#@markdown ### Comparación visual:
#@markdown
#@markdown **Superficie de error SIN escalado:**
#@markdown ```
#@markdown            ↑ J(w)
#@markdown            │
#@markdown       ▓▓▓▓▓│▓▓▓▓▓     ← Valle muy alargado
#@markdown     ▓▓     │     ▓▓   (difícil de descender)
#@markdown   ▓▓       │       ▓▓
#@markdown  ▓         ●         ▓  ← Mínimo difícil de alcanzar
#@markdown   ▓▓       │       ▓▓
#@markdown     ▓▓     │     ▓▓
#@markdown       ▓▓▓▓▓│▓▓▓▓▓
#@markdown ────────────────────────→ w
#@markdown
#@markdown Problema: Gradiente descendente oscila y diverge
#@markdown ```
#@markdown
#@markdown **Superficie de error CON escalado:**
#@markdown ```
#@markdown            ↑ J(w)
#@markdown            │
#@markdown        ████│████        ← Valle circular
#@markdown      ██    │    ██      (fácil de descender)
#@markdown     ██     │     ██
#@markdown    ██      ●      ██    ← Mínimo fácil de alcanzar
#@markdown     ██     │     ██
#@markdown      ██    │    ██
#@markdown        ████│████
#@markdown ────────────────────────→ w
#@markdown
#@markdown Solución: Gradiente descendente converge directamente
#@markdown ```
#@markdown
#@markdown ---
#@markdown
#@markdown ### Tipos de escalado:
#@markdown
#@markdown **StandardScaler (recomendado):**
#@markdown $$x_{\text{scaled}} = \frac{x - \mu}{\sigma}$$
#@markdown ```
#@markdown Resultado: Media=0, Desviación estándar=1
#@markdown
#@markdown Ejemplo:
#@markdown Original: [10, 20, 30, 40, 50]
#@markdown Escalado: [-1.41, -0.71, 0, 0.71, 1.41]
#@markdown
#@markdown Ventajas:
#@markdown - Preserva distribución original
#@markdown - Funciona bien con outliers moderados
#@markdown - Estándar en ML
#@markdown ```
#@markdown
#@markdown **MinMaxScaler (alternativa):**
#@markdown $$x_{\text{scaled}} = \frac{x - x_{\min}}{x_{\max} - x_{\min}}$$
#@markdown ```
#@markdown Resultado: Rango [0, 1]
#@markdown
#@markdown Ejemplo:
#@markdown Original: [10, 20, 30, 40, 50]
#@markdown Escalado: [0, 0.25, 0.5, 0.75, 1.0]
#@markdown
#@markdown Ventajas:
#@markdown - Rango acotado [0,1]
#@markdown - Útil para redes neuronales
#@markdown
#@markdown Desventajas:
#@markdown - Sensible a outliers
#@markdown ```
#@markdown
#@markdown ---
#@markdown
#@markdown ### ¿Cuándo DEBES usar escalado?
#@markdown
#@markdown **SIEMPRE en estos casos:** ✅
#@markdown - Grado ≥ 2 (features polinomiales)
#@markdown - Gradiente descendente
#@markdown - Regularización L1/L2
#@markdown - SVM, KNN, Redes Neuronales
#@markdown
#@markdown **Opcional en estos casos:** ⚠️
#@markdown - Árboles de decisión (inmunes a escala)
#@markdown - Random Forest (inmunes a escala)
#@markdown - Naive Bayes
#@markdown
#@markdown **Regla de oro:** Si usas gradiente descendente o polinomios → **SIEMPRE escalar**
#@markdown
#@markdown ---
#@markdown
#@markdown ## 🚀 Hiperparámetro 3: Tasa de Aprendizaje (alpha / learning rate)
#@markdown
#@markdown ```python
#@markdown poly_log_alpha = 0.01  # ¿Qué tan grande es cada paso del gradiente?
#@markdown ```
#@markdown
#@markdown ### ¿Qué es?
#@markdown La **tasa de aprendizaje** (η o alpha) controla el tamaño de cada actualización de pesos.
#@markdown
#@markdown $$\mathbf{w}_{\text{nuevo}} = \mathbf{w}_{\text{viejo}} - \eta \cdot \nabla\mathcal{L}$$
#@markdown
#@markdown ---
#@markdown
#@markdown ### Analogía del montañista (clasificación):
#@markdown
#@markdown Imagina que estás en una montaña con niebla buscando el **valle** (frontera de decisión óptima):
#@markdown
#@markdown **η muy pequeña (ej. 0.0001):**
#@markdown ```
#@markdown Comportamiento: Pasos de hormiga 🐜🐜🐜
#@markdown
#@markdown Ventaja: Muy seguro, nunca te caes
#@markdown Desventaja: Tardarás HORAS en llegar al valle
#@markdown
#@markdown En ML:
#@markdown - Pérdida baja muy lentamente
#@markdown - Necesitas 10,000+ iteraciones
#@markdown - Entrenamiento MUY lento
#@markdown ```
#@markdown
#@markdown **η moderada (ej. 0.01-0.1):**
#@markdown ```
#@markdown Comportamiento: Pasos normales de humano 🚶‍♂️🚶‍♂️
#@markdown
#@markdown Ventaja: Balance entre velocidad y seguridad
#@markdown Desventaja: Requiere ajuste fino
#@markdown
#@markdown En ML:
#@markdown - Convergencia en 1,000-3,000 iteraciones
#@markdown - Pérdida disminuye suavemente
#@markdown - ✅ Configuración típica
#@markdown ```
#@markdown
#@markdown **η muy grande (ej. 1.0):**
#@markdown ```
#@markdown Comportamiento: Saltos de canguro 🦘🦘
#@markdown
#@markdown Ventaja: Rápido... en teoría
#@markdown Desventaja: Saltas SOBRE el valle (nunca llegas)
#@markdown
#@markdown En ML:
#@markdown - Pérdida OSCILA violentamente
#@markdown - Puede DIVERGER (NaN/Inf)
#@markdown - Modelo no aprende
#@markdown ```
#@markdown
#@markdown ---
#@markdown
#@markdown ### Gráficas de pérdida según α:
#@markdown
#@markdown ```
#@markdown α = 0.0001 (muy pequeña)
#@markdown Pérdida │╲
#@markdown (Loss)  │ ╲________________  ← Baja MUY lento
#@markdown        │
#@markdown        └────────────────────→ Iteraciones
#@markdown        0    5000   10000
#@markdown
#@markdown α = 0.01 (óptima)
#@markdown Pérdida │╲
#@markdown (Loss)  │ ╲___  ← Converge rápido y suave
#@markdown        │
#@markdown        └─────→ Iteraciones
#@markdown        0  3000
#@markdown
#@markdown α = 0.5 (demasiado grande)
#@markdown Pérdida │  ╱╲╱╲╱╲
#@markdown (Loss)  │╱╲╱╲╱╲╱╲  ← Oscila, NO converge
#@markdown        │
#@markdown        └──────→ Iteraciones
#@markdown ```
#@markdown
#@markdown ---
#@markdown
#@markdown ### Valores típicos según contexto:
#@markdown
#@markdown | Situación | α recomendada | Iteraciones típicas | Razón |
#@markdown |:----------|:--------------|:--------------------|:------|
#@markdown | **Features escaladas, grado 1-2** | 0.01 - 0.1 | 1000-3000 | Gradientes bien condicionados |
#@markdown | **Features NO escaladas** | 0.0001 - 0.001 | 5000-10000 | Gradientes muy grandes |
#@markdown | **Muchas features (>100)** | 0.001 - 0.01 | 2000-5000 | Evitar inestabilidad |
#@markdown | **Pocas features (<20)** | 0.01 - 0.5 | 1000-2000 | Puede ser más agresivo |
#@markdown | **Grado alto (≥3)** | 0.001 - 0.01 | 3000-5000 | Superficie compleja |
#@markdown | **Dataset pequeño (<500 muestras)** | 0.01 - 0.05 | 1000-2000 | Rápido pero cuidadoso |
#@markdown | **Dataset grande (>10000 muestras)** | 0.05 - 0.5 | 500-2000 | Gradientes más estables |
#@markdown
#@markdown ---
#@markdown
#@markdown ### ¿Cómo saber si α está bien?
#@markdown
#@markdown **Señales de α CORRECTA:** ✅
#@markdown ```
#@markdown Gráfica de pérdida:
#@markdown - Disminuye monotónicamente (siempre baja)
#@markdown - Curva suave sin saltos
#@markdown - Se aplana después de convergencia
#@markdown - No hay NaN ni Inf
#@markdown
#@markdown Resultados:
#@markdown - Accuracy mejora consistentemente
#@markdown - Converge en tiempo razonable (<5 minutos)
#@markdown ```
#@markdown
#@markdown **Señales de α MUY PEQUEÑA:** ⚠️
#@markdown ```
#@markdown Gráfica de pérdida:
#@markdown - Baja muy lentamente
#@markdown - Al final sigue bajando (no converge)
#@markdown - Línea casi horizontal después de muchas iters
#@markdown
#@markdown Solución:
#@markdown - Multiplicar α por 5-10
#@markdown - O aumentar iteraciones a 10,000+
#@markdown ```
#@markdown
#@markdown **Señales de α MUY GRANDE:** ❌
#@markdown ```
#@markdown Gráfica de pérdida:
#@markdown - Oscila (sube y baja)
#@markdown - Puede AUMENTAR en lugar de disminuir
#@markdown - Valores NaN o Inf aparecen
#@markdown - Modelo "explota"
#@markdown
#@markdown Solución:
#@markdown - Dividir α por 10
#@markdown - Verificar que escalado esté activo
#@markdown ```
#@markdown
#@markdown ---
#@markdown
#@markdown ### Método para encontrar α óptima:
#@markdown
#@markdown **Técnica: Búsqueda logarítmica**
#@markdown ```python
#@markdown # Probar múltiples valores en escala logarítmica
#@markdown alphas = [0.001, 0.01, 0.1, 0.5]
#@markdown
#@markdown for alpha in alphas:
#@markdown     entrenar_modelo(alpha, iteraciones=1000)
#@markdown     graficar_perdida()
#@markdown     print(f"α={alpha}: Loss final = {loss_final}")
#@markdown
#@markdown # Elegir α donde:
#@markdown # 1. Converge más rápido
#@markdown # 2. No oscila
#@markdown # 3. Loss final es mínima
#@markdown ```
#@markdown
#@markdown **Ejemplo de salida:**
#@markdown ```
#@markdown α=0.001: Loss final = 0.15 (muy lento, no convergió)
#@markdown α=0.01:  Loss final = 0.08 (✅ perfecto, convergió suave)
#@markdown α=0.1:   Loss final = 0.09 (oscila ligeramente)
#@markdown α=0.5:   Loss final = NaN   (divergió)
#@markdown
#@markdown Decisión: Usar α = 0.01
#@markdown ```
#@markdown
#@markdown ---
#@markdown
#@markdown ## 🔁 Hiperparámetro 4: Número de Iteraciones (n_iterations)
#@markdown
#@markdown ```python
#@markdown poly_log_iters = 3000  # ¿Cuántas veces actualizar pesos?
#@markdown ```
#@markdown
#@markdown ### ¿Qué es?
#@markdown El número de **iteraciones** (o epochs) determina cuántas veces el algoritmo recorre los datos ajustando los pesos.
#@markdown
#@markdown ---
#@markdown
#@markdown ### Proceso en cada iteración:
#@markdown ```
#@markdown Iteración 1:
#@markdown   1. Calcular probabilidades: p = σ(X·w)
#@markdown   2. Calcular error: error = p - y
#@markdown   3. Calcular gradiente: ∇L = (1/N)·X^T·error
#@markdown   4. Actualizar pesos: w = w - α·∇L
#@markdown
#@markdown Iteración 2:
#@markdown   (repetir con los nuevos pesos w)
#@markdown
#@markdown ...
#@markdown
#@markdown Iteración N:
#@markdown   (pesos finales óptimos)
#@markdown ```
#@markdown
#@markdown ---
#@markdown
#@markdown ### ¿Cuántas iteraciones necesitas?
#@markdown
#@markdown **Depende de α y complejidad del modelo:**
#@markdown
#@markdown | α (learning rate) | Iteraciones necesarias | Tiempo total |
#@markdown |:------------------|:-----------------------|:-------------|
#@markdown | 0.001 (muy pequeña) | 5,000 - 10,000 | Lento ⏱️⏱️⏱️ |
#@markdown | 0.01 (moderada) | 2,000 - 5,000 | Medio ⏱️⏱️ |
#@markdown | 0.1 (grande) | 500 - 2,000 | Rápido ⏱️ |
#@markdown
#@markdown **Regla práctica:** Más α → Menos iteraciones necesarias
#@markdown
#@markdown ---
#@markdown
#@markdown ### Señales de convergencia:
#@markdown
#@markdown **Convergencia completa:** ✅
#@markdown ```
#@markdown Gráfica de pérdida:
#@markdown │╲
#@markdown │ ╲_________  ← Pérdida se aplana (plateau)
#@markdown │
#@markdown └──────────→ Iteraciones
#@markdown
#@markdown Características:
#@markdown - Últimas 500 iteraciones: cambio < 0.001
#@markdown - Accuracy ya no mejora
#@markdown - Modelo alcanzó su óptimo
#@markdown
#@markdown ✅ Puedes parar aquí, más iteraciones no ayudan
#@markdown ```
#@markdown
#@markdown **Convergencia prematura (early stopping):** 🛑
#@markdown ```
#@markdown Train Loss │╲___  ← Sigue bajando
#@markdown Test Loss  │╲_╱   ← Empieza a SUBIR (sobreajuste)
#@markdown           │
#@markdown           └──→ Iteraciones
#@markdown              ↑
#@markdown           Parar aquí
#@markdown
#@markdown Técnica: Early Stopping
#@markdown - Monitorear loss en validación
#@markdown - Si no mejora en 100 iteraciones → parar
#@markdown - Guardar modelo del mejor punto
#@markdown ```
#@markdown
#@markdown **Insuficientes iteraciones:** ⚠️
#@markdown ```
#@markdown Gráfica de pérdida:
#@markdown │╲
#@markdown │ ╲
#@markdown │  ╲  ← Todavía bajando al terminar
#@markdown │   ╲
#@markdown └────→ Iteraciones
#@markdown      Termina aquí
#@markdown
#@markdown Problema:
#@markdown - Modelo no alcanzó su potencial
#@markdown - Accuracy podría mejorar más
#@markdown - Pesos no convergieron
#@markdown
#@markdown Solución:
#@markdown - Aumentar iteraciones (doblar)
#@markdown - O aumentar α para converger más rápido
#@markdown ```
#@markdown
#@markdown **Demasiadas iteraciones (desperdicio):** 💸
#@markdown ```
#@markdown Gráfica de pérdida:
#@markdown │╲
#@markdown │ ╲__________________  ← Ya convergió hace rato
#@markdown │         (desperdicio de tiempo)
#@markdown └────────────────────→ Iteraciones
#@markdown        ↑ Ya estaba listo aquí
#@markdown
#@markdown Problema:
#@markdown - Tiempo de entrenamiento innecesario
#@markdown - No hay mejora después de convergencia
#@markdown - Riesgo de sobreajuste mínimo
#@markdown
#@markdown Solución:
#@markdown - Reducir iteraciones
#@markdown - Implementar early stopping
#@markdown ```
#@markdown
#@markdown ---
#@markdown
#@markdown ### Configuración recomendada según contexto:
#@markdown
#@markdown **Exploración rápida (prototipos):**
#@markdown ```python
#@markdown poly_log_alpha = 0.1    # Alta para rapidez
#@markdown poly_log_iters = 1000   # Pocas iteraciones
#@markdown
#@markdown Objetivo: Ver si el modelo funciona
#@markdown Tiempo: ~30 segundos
#@markdown ```
#@markdown
#@markdown **Configuración estándar:**
#@markdown ```python
#@markdown poly_log_alpha = 0.01   # Moderada
#@markdown poly_log_iters = 3000   # Suficiente para convergencia
#@markdown
#@markdown Objetivo: Balance velocidad-calidad
#@markdown Tiempo: ~2-3 minutos
#@markdown ```
#@markdown
#@markdown **Entrenamiento cuidadoso (producción):**
#@markdown ```python
#@markdown poly_log_alpha = 0.01
#@markdown poly_log_iters = 5000-10000
#@markdown + Early stopping activado
#@markdown + Validación cruzada
#@markdown
#@markdown Objetivo: Mejor modelo posible
#@markdown Tiempo: ~5-10 minutos
#@markdown ```
#@markdown
#@markdown ---
#@markdown
#@markdown ### Implementación de Early Stopping:
#@markdown
#@markdown ```python
#@markdown mejor_loss_test = float('inf')
#@markdown paciencia = 0
#@markdown max_paciencia = 100  # Esperar 100 iters sin mejora
#@markdown
#@markdown for iteracion in range(max_iteraciones):
#@markdown     # Entrenar una iteración
#@markdown     entrenar_una_epoca()
#@markdown
#@markdown     # Evaluar en test (o validación)
#@markdown     loss_test_actual = calcular_loss_test()
#@markdown
#@markdown     # Verificar si mejoró
#@markdown     if loss_test_actual < mejor_loss_test:
#@markdown         mejor_loss_test = loss_test_actual
#@markdown         guardar_modelo_actual()
#@markdown         paciencia = 0  # Reiniciar contador
#@markdown     else:
#@markdown         paciencia += 1  # Aumentar contador
#@markdown
#@markdown     # Parar si no hay mejora
#@markdown     if paciencia >= max_paciencia:
#@markdown         print(f"Early stopping en iteración {iteracion}")
#@markdown         print(f"No hubo mejora en {max_paciencia} iteraciones")
#@markdown         break
#@markdown
#@markdown # Cargar el mejor modelo guardado
#@markdown cargar_mejor_modelo()
#@markdown ```
#@markdown
#@markdown ---
#@markdown
#@markdown ## 🎯 Cómo Ajustar Todos los Hiperparámetros Juntos
#@markdown
#@markdown ### Proceso paso a paso recomendado:
#@markdown
#@markdown **Fase 1: Configuración inicial segura**
#@markdown ```python
#@markdown # Valores conservadores que casi siempre funcionan
#@markdown grado_poly_log = 2              # Cuadrático
#@markdown use_scaling_poly_log = True     # SIEMPRE
#@markdown poly_log_alpha = 0.01           # Moderada
#@markdown poly_log_iters = 3000           # Suficiente
#@markdown
#@markdown # Entrenar y observar
#@markdown entrenar_modelo()
#@markdown ```
#@markdown
#@markdown **Fase 2: Diagnóstico de la curva de pérdida**
#@markdown ```python
#@markdown # Observar la gráfica de pérdida
#@markdown
#@markdown if perdida_oscila:
#@markdown     poly_log_alpha /= 10  # Reducir α
#@markdown     print("α muy grande, reduciendo...")
#@markdown
#@markdown elif perdida_baja_muy_lento:
#@markdown     poly_log_alpha *= 2   # Aumentar α
#@markdown     print("α muy pequeña, aumentando...")
#@markdown
#@markdown elif perdida_no_converge:
#@markdown     poly_log_iters *= 2   # Más iteraciones
#@markdown     print("Necesita más iteraciones...")
#@markdown
#@markdown else:
#@markdown     print("✅ Configuración correcta")
#@markdown ```
#@markdown
#@markdown **Fase 3: Evaluar sobreajuste**
#@markdown ```python
#@markdown acc_train = evaluar_train()
#@markdown acc_test = evaluar_test()
#@markdown diferencia = abs(acc_train - acc_test)
#@markdown
#@markdown if diferencia > 0.05:  # >5% diferencia
#@markdown     print("⚠️ SOBREAJUSTE detectado")
#@markdown
#@markdown     if grado_poly_log > 1:
#@markdown         grado_poly_log -= 1
#@markdown         print(f"Reduciendo grado a {grado_poly_log}")
#@markdown     else:
#@markdown         print("Usar regularización (Ridge/Lasso)")
#@markdown
#@markdown elif diferencia < 0.01:  # <1% diferencia
#@markdown     print("✅ Excelente balance")
#@markdown
#@markdown else:
#@markdown     print("✅ Balance aceptable")
#@markdown ```
#@markdown
#@markdown **Fase 4: Optimizar grado del polinomio**
#@markdown ```python
#@markdown # Probar diferentes grados sistemáticamente
#@markdown resultados = {}
#@markdown
#@markdown for grado in [1, 2, 3]:
#@markdown     modelo = entrenar_con_grado(grado)
#@markdown     acc_test = evaluar_test(modelo)
#@markdown     diferencia_train_test = calcular_diferencia(modelo)
#@markdown
#@markdown     resultados[grado] = {
#@markdown         'acc_test': acc_test,
#@markdown         'diferencia': diferencia_train_test
#@markdown     }
#@markdown
#@markdown # Elegir el mejor
#@markdown mejor_grado = max(resultados,
#@markdown                   key=lambda g: resultados[g]['acc_test'])
#@markdown
#@markdown # Verificar que no sobreajuste
#@markdown if resultados[mejor_grado]['diferencia'] < 0.05:
#@markdown     print(f"✅ Usar grado {mejor_grado}")
#@markdown else:
#@markdown     print(f"⚠️ Grado {mejor_grado} sobreajusta")
#@markdown     print(f"Usar grado {mejor_grado-1} en su lugar")
#@markdown ```
#@markdown
#@markdown ---
#@markdown
#@markdown ## 📊 Tabla de Troubleshooting (Solución de Problemas)
#@markdown
#@markdown | Problema | Síntoma | Causa probable | Solución |
#@markdown |:---------|:--------|:---------------|:---------|
#@markdown | **Pérdida = NaN** | Loss se vuelve NaN/Inf | α muy grande | α ÷ 10 |
#@markdown | **Pérdida oscila** | Sube y baja | α demasiado grande | α ÷ 2 |
#@markdown | **Pérdida no baja** | Casi plana desde inicio | α muy pequeña | α × 5 |
#@markdown | **No converge** | Sigue bajando al terminar | Pocas iteraciones | Iters × 2 |
#@markdown | **Sobreajuste** | Train>>Test (>5%) | Grado muy alto | Reducir grado |
#@markdown | **Subajuste** | Train≈Test (ambos bajos) | Grado muy bajo | Aumentar grado |
#@markdown | **Muy lento** | Tarda mucho | α muy pequeña | Aumentar α |
#@markdown | **Frontera zigzag** | Decisiones erráticas | Grado muy alto | Reducir grado |
#@markdown | **Accuracy no mejora** | Estancado en ~50% | Features no escaladas | Activar escalado |
#@markdown
#@markdown ---
#@markdown
#@markdown ## 📋 Recetas de Configuración para Casos Comunes
#@markdown
#@markdown ### Receta 1: Dataset pequeño (<500 muestras)
#@markdown ```python
#@markdown grado_poly_log = 1 o 2      # No usar grado alto
#@markdown use_scaling_poly_log = True # SIEMPRE
#@markdown poly_log_alpha = 0.01       # Moderada
#@markdown poly_log_iters = 2000       # No muchas (riesgo sobreajuste)
#@markdown
#@markdown # Consideración extra:
#@markdown # - Usar validación cruzada
#@markdown # - Considerar regularización
#@markdown ```
#@markdown
#@markdown ### Receta 2: Dataset grande (>5000 muestras)
#@markdown ```python
#@markdown grado_poly_log = 2 o 3      # Puede permitirse más complejidad
#@markdown use_scaling_poly_log = True # SIEMPRE
#@markdown poly_log_alpha = 0.05-0.1   # Puede ser más agresiva
#@markdown poly_log_iters = 2000       # Converge más rápido
#@markdown
#@markdown # Consideración extra:
#@markdown # - Mini-batch gradient descent
#@markdown # - Menos riesgo de sobreajuste
#@markdown ```
#@markdown
#@markdown ### Receta 3: Clases claramente lineales
#@markdown ```python
#@markdown grado_poly_log = 1          # Lineal suficiente
#@markdown use_scaling_poly_log = True # SIEMPRE
#@markdown poly_log_alpha = 0.1        # Puede ser alta
#@markdown poly_log_iters = 1000       # Converge rápido
#@markdown
#@markdown # Si accuracy >95%: NO necesitas polinomios
#@markdown ```
#@markdown
#@markdown ### Receta 4: Clases con patrones curvos
#@markdown ```python
#@markdown grado_poly_log = 2          # Cuadrático
#@markdown use_scaling_poly_log = True # CRÍTICO para grado 2
#@markdown poly_log_alpha = 0.01       # Conservadora
#@markdown poly_log_iters = 3000       # Dar tiempo a converger
#@markdown
#@markdown # Ejemplo: Clasificar dentro/fuera de círculo
#@markdown ```
#@markdown
#@markdown ### Receta 5: Muchas features (>50)
#@markdown ```python
#@markdown grado_poly_log = 1 o 2      # Evitar explosión de features
#@markdown use_scaling_poly_log = True # CRÍTICO
#@markdown poly_log_alpha = 0.001      # MUY pequeña
#@markdown poly_log_iters = 5000       # Más iteraciones
#@markdown
#@markdown # Con 50 features originales:
#@markdown # - Grado 2: genera ~1,300 features
#@markdown # - Grado 3: genera ~22,000 features ← NO HACER
#@markdown ```
#@markdown
#@markdown ### Receta 6: Diagnóstico médico (como cáncer)
#@markdown ```python
#@markdown grado_poly_log = 1 o 2      # Interpretabilidad importante
#@markdown use_scaling_poly_log = True # SIEMPRE
#@markdown poly_log_alpha = 0.01       # Estable
#@markdown poly_log_iters = 3000       # Asegurar convergencia
#@markdown
#@markdown # Consideraciones especiales:
#@markdown # - Minimizar falsos negativos (más crítico)
#@markdown # - Ajustar umbral de decisión a 0.3-0.4
#@markdown # - Priorizar recall sobre precision
#@markdown ```
#@markdown
#@markdown ---
#@markdown
#@markdown ## 💡 Tips Finales y Mejores Prácticas
#@markdown
#@markdown ### 1. Orden de ajuste recomendado:
#@markdown ```
#@markdown 1º: Activar escalado (SIEMPRE primero)
#@markdown 2º: Elegir grado inicial (empezar con 1 o 2)
#@markdown 3º: Ajustar α observando curva de pérdida
#@markdown 4º: Ajustar iteraciones según convergencia
#@markdown 5º: Comparar diferentes grados
#@markdown 6º: Elegir el mejor grado (balance accuracy/complejidad)
#@markdown ```
#@markdown
#@markdown ### 2. Reglas de oro:
#@markdown ```
#@markdown ✅ SIEMPRE escalar features con grado ≥2
#@markdown ✅ Empezar simple (grado 1-2)
#@markdown ✅ Observar curva de pérdida antes de cambiar
#@markdown ✅ Comparar train vs test para detectar sobreajuste
#@markdown ✅ Preferir modelo simple si funciona igual de bien
#@markdown
#@markdown ❌ NUNCA usar grado ≥4 sin razón muy clara
#@markdown ❌ NUNCA ignorar señales de sobreajuste
#@markdown ❌ NUNCA dejar escalado = False con polinomios
#@markdown ❌ NUNCA elegir hiperparámetros mirando solo train
#@markdown ```
#@markdown
#@markdown ### 3. Validación de tu configuración:
#@markdown ```python
#@markdown # Checklist antes de considerar el modelo listo:
#@markdown
#@markdown ✓ Pérdida converge suavemente (no oscila)
#@markdown ✓ Accuracy test ≥ 90% (o meta del proyecto)
#@markdown ✓ Diferencia train-test < 5%
#@markdown ✓ Curva de pérdida se aplana (convergió)
#@markdown ✓ No hay NaN ni Inf en ningún momento
#@markdown ✓ Matriz de confusión es aceptable
#@markdown ✓ Falsos negativos minimizados (si es crítico)
#@markdown ✓ Tiempo de entrenamiento razonable (<10 min)
#@markdown ```
#@markdown
#@markdown ### 4. Cuándo parar de optimizar:
#@markdown ```
#@markdown PARAR cuando:
#@markdown - Accuracy test > 95% y diferencia < 3%
#@markdown - Mejoras adicionales < 1%
#@markdown - Complejidad no justifica ganancia marginal
#@markdown - Ya probaste grados 1, 2, 3 y 2 es mejor
#@markdown
#@markdown SEGUIR optimizando cuando:
#@markdown - Accuracy test < 85%
#@markdown - Diferencia train-test > 10%
#@markdown - Pérdida no converge
#@markdown - Hay NaN o comportamiento errático
#@markdown ```
#@markdown
#@markdown ---
#@markdown
#@markdown ## 🎓 Ejercicio Práctico: Ajuste Guiado
#@markdown
#@markdown ### Paso 1: Configuración inicial
#@markdown ```python
#@markdown grado_poly_log = 2
#@markdown use_scaling_poly_log = True
#@markdown poly_log_alpha = 0.01
#@markdown poly_log_iters = 3000
#@markdown ```
#@markdown
#@markdown ### Paso 2: Entrenar y documentar
#@markdown ```
#@markdown Anota tus resultados:
#@markdown
#@markdown Grado 2:
#@markdown - Pérdida final: _______
#@markdown - Accuracy train: _______
#@markdown - Accuracy test: _______
#@markdown - Diferencia: _______
#@markdown - ¿Converge suavemente? Sí / No
#@markdown - ¿Hay sobreajuste? Sí / No
#@markdown ```
#@markdown
#@markdown ### Paso 3: Experimentar con grado
#@markdown ```
#@markdown Prueba grado 1 y 3, anota resultados
#@markdown
#@markdown Grado 1:
#@markdown - Accuracy test: _______
#@markdown - Mejora vs grado 2: _______
#@markdown
#@markdown Grado 3:
#@markdown - Accuracy test: _______
#@markdown - Mejora vs grado 2: _______
#@markdown - ¿Sobreajuste? _______
#@markdown ```
#@markdown
#@markdown ### Paso 4: Decisión final
#@markdown ```
#@markdown Basándote en tus resultados:
#@markdown
#@markdown 1. ¿Qué grado elegirías? _______
#@markdown 2. ¿Por qué? _______________________
#@markdown 3. ¿Qué accuracy esperas en producción? _______
#@markdown 4. ¿Es suficiente para tu aplicación? Sí / No
#@markdown ```
#@markdown
#@markdown ---
#@markdown
#@markdown ## 🔑 Resumen Ejecutivo
#@markdown
#@markdown | Hiperparámetro | Qué controla | Valor inicial | Cómo ajustar |
#@markdown |:---------------|:-------------|:--------------|:-------------|
#@markdown | **Grado** | Complejidad frontera | 2 | Comparar 1,2,3 sistemáticamente |
#@markdown | **Escalado** | Estabilidad numérica | True | SIEMPRE True con polinomios |
#@markdown | **α (alpha)** | Velocidad aprendizaje | 0.01 | Ver curva pérdida, ajustar ×2 o ÷2 |
#@markdown | **Iteraciones** | Tiempo convergencia | 3000 | Aumentar si no converge, reducir si desperdicia |
#@markdown
#@markdown **Configuración segura universal:**
#@markdown ```python
#@markdown grado_poly_log = 2
#@markdown use_scaling_poly_log = True
#@markdown poly_log_alpha = 0.01
#@markdown poly_log_iters = 3000
#@markdown ```
#@markdown
#@markdown **Recuerda:** Más complejo NO siempre es mejor. Un modelo simple que funciona bien es mejor que uno complejo que apenas mejora.

In [ ]:
#@title 🗺️ Visualización: Frontera de Decisión Polinomial en 2D

# Verificar variables necesarias
try:
    _ = Xct, yct, grado_poly_log, w_b_poly_log
    print("✅ Usando modelo ya entrenado")
except NameError:
    print("❌ ERROR: Ejecuta primero la celda de entrenamiento")
    print("   '🧮 Regresión Logística Polinomial'")
else:
    # Reducir a 2D con PCA
    pca_2d = PCA(n_components=2).fit(Xct)
    Z_train = pca_2d.transform(Xct)

    # Entrenar modelo 2D para visualización
    Z_poly = crear_features_polinomiales(Z_train, grado=grado_poly_log)
    Z_poly_b = add_bias(Z_poly)

    w_viz = np.zeros((Z_poly_b.shape[1], 1))
    for _ in range(1000):
        p = sigmoid(Z_poly_b @ w_viz)
        w_viz -= 0.1 * (Z_poly_b.T @ (p - yct)) / len(yct)

    # Crear malla
    x_min, x_max = Z_train[:,0].min()-1, Z_train[:,0].max()+1
    y_min, y_max = Z_train[:,1].min()-1, Z_train[:,1].max()+1
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300),
                         np.linspace(y_min, y_max, 300))

    # Calcular probabilidades en la malla
    grid = np.c_[xx.ravel(), yy.ravel()]
    grid_poly = crear_features_polinomiales(grid, grado=grado_poly_log)
    grid_poly_b = add_bias(grid_poly)
    probs = sigmoid(grid_poly_b @ w_viz).reshape(xx.shape)

    # Visualizar
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

    # Subplot 1: Regiones de decisión
    ax1.contourf(xx, yy, (probs > 0.5).astype(int), alpha=0.3, cmap='RdBu')
    ax1.contour(xx, yy, probs, levels=[0.5], linewidths=3, colors='black', linestyles='--')
    ax1.scatter(Z_train[:,0], Z_train[:,1], c=yc_train, cmap='RdBu',
               edgecolor='k', s=50, alpha=0.8)
    ax1.set_xlabel("PC1")
    ax1.set_ylabel("PC2")
    ax1.set_title(f"Frontera de Decisión (Grado {grado_poly_log})")
    ax1.grid(True, alpha=0.2)

    # Subplot 2: Mapa de probabilidades
    contour = ax2.contourf(xx, yy, probs, levels=20, cmap='RdBu', alpha=0.8)
    ax2.contour(xx, yy, probs, levels=[0.5], linewidths=3, colors='yellow', linestyles='--')
    ax2.scatter(Z_train[:,0], Z_train[:,1], c=yc_train, cmap='RdBu',
               edgecolor='white', s=50, alpha=0.9, linewidths=2)
    ax2.set_xlabel("PC1")
    ax2.set_ylabel("PC2")
    ax2.set_title(f"Mapa de Probabilidades P(Benigno)")
    plt.colorbar(contour, ax=ax2, label="P(Benigno)")
    ax2.grid(True, alpha=0.2)

    plt.tight_layout()
    plt.show()

    print(f"\n📖 Interpretación:")
    print(f"   • Línea negra/amarilla: Frontera donde P = 0.5")
    print(f"   • Región azul: Benigno | Región roja: Maligno")
    if grado_poly_log == 1:
        print(f"   • Frontera: LÍNEA RECTA (lineal)")
    elif grado_poly_log == 2:
        print(f"   • Frontera: CURVA (cuadrático)")
    else:
        print(f"   • Frontera: CURVA COMPLEJA (grado {grado_poly_log})")

#Metricas

In [ ]:
#@title 🧮 Introducción a las Métricas de Clasificación
#@markdown # 📊 Evaluación de Modelos de Clasificación
#@markdown ---
#@markdown
#@markdown La **clasificación** busca predecir **categorías discretas** (ej. *benigno* / *maligno*).
#@markdown Para evaluar su desempeño, necesitamos comparar las **predicciones del modelo (ŷ)** con las **etiquetas reales (y)**.
#@markdown
#@markdown La base de todas las métricas es la **Matriz de Confusión**, que resume los aciertos y errores:
#@markdown
#@markdown |                      | **Predicho 0** | **Predicho 1** |
#@markdown |----------------------|----------------|----------------|
#@markdown | **Real 0 (negativo)**| TN (Verdadero Negativo) | FP (Falso Positivo) |
#@markdown | **Real 1 (positivo)**| FN (Falso Negativo) | TP (Verdadero Positivo) |
#@markdown
#@markdown ---
#@markdown ## 🧩 Interpretación básica
#@markdown - **TP:** casos positivos bien detectados.
#@markdown - **TN:** casos negativos bien detectados.
#@markdown - **FP:** falsos positivos (alarma falsa).
#@markdown - **FN:** falsos negativos (casos no detectados).
#@markdown
#@markdown A partir de estos 4 valores, surgen las métricas más importantes.


In [ ]:
#@title ⚙️ Implementación: Accuracy, Precision y Recall
#@markdown Vamos a implementar las métricas básicas **desde su base matemática** y luego con **Scikit-Learn**.
#@markdown Deberás completar algunas métricas faltantes.

import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

# Ejemplo de etiquetas reales y predichas
y_true = np.array([1,0,1,1,0,1,0,0,1,0])
y_pred = np.array([1,0,1,0,0,1,1,0,1,0])

# 1️⃣ Matriz de confusión
TN, FP, FN, TP = confusion_matrix(y_true, y_pred).ravel()
print("TN:", TN, "FP:", FP, "FN:", FN, "TP:", TP)

# 2️⃣ Accuracy — Exactitud
# Fórmula: (TP + TN) / (TP + TN + FP + FN)
accuracy_manual = (TP + TN) / (TP + TN + FP + FN)
print(f"\nAccuracy (manual): {accuracy_manual:.4f}")

# Desde la librería
accuracy_lib = accuracy_score(y_true, y_pred)
print(f"Accuracy (sklearn): {accuracy_lib:.4f}")

# 3️⃣ Precision — Pureza de los positivos
# Fórmula: TP / (TP + FP)
precision_manual = TP / (TP + FP)
print(f"\nPrecision (manual): {precision_manual:.4f}")

# Desde la librería
precision_lib = precision_score(y_true, y_pred)
print(f"Precision (sklearn): {precision_lib:.4f}")

# 4️⃣ TODO: Completar métricas faltantes
# Recall (Sensibilidad): TP / (TP + FN)
# F1-Score: 2*(Precision*Recall)/(Precision + Recall)
# Especificidad: TN / (TN + FP)
# AUC: calcular con sklearn (roc_auc_score)

# === Espacio para los estudiantes ===
# recall_manual = ...
# f1_manual = ...
# specificity_manual = ...
# auc_lib = ...

# print("Recall:", recall_manual)
# print("F1-Score:", f1_manual)
# print("Specificidad:", specificity_manual)
# print("AUC (sklearn):", auc_lib)


In [ ]:
#@title 📊 Métricas de Clasificación — Logística Polinomial (manual + sklearn)
#@markdown Esta celda calcula **accuracy, precision, recall, F1, especificidad (specificity) y AUC**
#@markdown para **train** y **test**, tanto **manual** (desde la matriz de confusión) como con **Scikit-Learn**.
#@markdown Requiere las variables del bloque anterior: `yct, yce, y_pred_train_poly, y_pred_test_poly, p_hat_train, p_hat_test`.

import numpy as np
import pandas as pd
from sklearn.metrics import (
    confusion_matrix, accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score
)

def _safe_div(a, b):
    return a / b if b != 0 else 0.0

def metrics_from_confusion(TN, FP, FN, TP):
    """Métricas manuales desde TN, FP, FN, TP."""
    acc = _safe_div(TP + TN, TP + TN + FP + FN)
    prec = _safe_div(TP, TP + FP)
    rec = _safe_div()                 # Sensibilidad / Recall
    spec = _safe_div()                # Especificidad
    f1 = _safe_div() if () != 0 else 0.0
    return acc, prec, rec, f1, spec

def compute_block(y_true, y_pred, y_score):
    """Calcula métrica manual + sklearn para un conjunto (train o test)."""
    # Matriz de confusión y métricas manuales
    TN, FP, FN, TP = confusion_matrix(y_true, y_pred).ravel()
    acc_m, prec_m, rec_m, f1_m, spec_m = metrics_from_confusion(TN, FP, FN, TP)

    # Métricas sklearn (usar promedio='binary' por ser binario)
    acc_s  = accuracy_score(y_true, y_pred)
    prec_s = precision_score(y_true, y_pred, zero_division=0)
    rec_s  = recall_score(y_true, y_pred, zero_division=0)
    f1_s   = f1_score(y_true, y_pred, zero_division=0)

    # Para AUC necesitamos probabilidades/score continuo de la clase positiva
    # (en tu entrenamiento: p_hat_* es la prob de y=1)
    try:
        auc_s = roc_auc_score(y_true, y_score)
    except ValueError:
        auc_s = np.nan  # por si hay una sola clase en el conjunto

    df_manual = pd.Series({
        "Accuracy (manual)": acc_m,
        "Precision (manual)": prec_m,
        "Recall (manual)": rec_m,
        "F1-score (manual)": f1_m,
        "Especificidad (manual)": spec_m,
    })

    df_sklearn = pd.Series({
        "Accuracy (sklearn)": acc_s,
        "Precision (sklearn)": prec_s,
        "Recall (sklearn)": rec_s,
        "F1-score (sklearn)": f1_s,
        "AUC (sklearn)": auc_s,
    })

    cm = np.array([[TN, FP],
                   [FN, TP]])

    return df_manual, df_sklearn, cm

# === Train ===
y_true_tr = yct.ravel().astype(int)
y_pred_tr = y_pred_train_poly.ravel().astype(int)
y_score_tr = p_hat_train.ravel()  # prob de clase positiva

man_tr, skl_tr, cm_tr = compute_block(y_true_tr, y_pred_tr, y_score_tr)

# === Test ===
y_true_te = yce.ravel().astype(int)
y_pred_te = y_pred_test_poly.ravel().astype(int)
y_score_te = p_hat_test.ravel()

man_te, skl_te, cm_te = compute_block(y_true_te, y_pred_te, y_score_te)

# Mostrar resultados en tablas limpias
print("======== TRAIN ========")
display(pd.concat([man_tr, skl_tr], axis=0).to_frame("Valor").style.format("{:.4f}"))

print("Matriz de Confusión (TRAIN)  [[TN, FP],[FN, TP]]")
print(cm_tr, "\n")

print("======== TEST ========")
display(pd.concat([man_te, skl_te], axis=0).to_frame("Valor").style.format("{:.4f}"))

print("Matriz de Confusión (TEST)  [[TN, FP],[FN, TP]]")
print(cm_te)


# 📏 Evaluación de modelos de **Regresión**
## (aplicado a Regresión Polinómica)

En regresión predecimos un valor **continuo** (en nuestro caso, `y_reg = mean radius` del tumor).  
Comparamos los valores reales $y$ con las predicciones $\hat{y}$:

$$y=(y_1,\dots,y_m),\quad \hat{y}=(\hat{y}_1,\dots,\hat{y}_m)$$

---

## 📊 Métricas más importantes
### (qué miden, por qué, y cuándo usarlas)

### **1) MSE — Mean Squared Error (Error Cuadrático Medio)**

$$\text{MSE}=\frac{1}{m}\sum_{i=1}^{m}(y_i-\hat{y}_i)^2$$

**¿Qué mide?**  
El promedio de los errores al cuadrado. Penaliza mucho los errores grandes.

**¿Por qué usarla?**  
Es la métrica clásica de mínimos cuadrados; útil para optimizar modelos y comparar ajustes cuando te importa castigar fuerte los outliers.

**¿Cuándo?**
- Cuando los errores grandes son críticos (prefiero castigar 5 mm de error que varios de 1 mm).
- En validación de modelos lineales y polinomiales cuando buscas mínima energía del error.

**Contras:** Muy sensible a outliers (un solo punto raro puede inflarla).

**Ejemplo:** MSE = 4 → errores al cuadrado promedio de 4 (≈ errores típicos de 2 mm si fueran homogéneos).

---

### **2) RMSE — Root Mean Squared Error (Raíz del MSE)**

$$\text{RMSE}=\sqrt{\text{MSE}}$$

**¿Qué mide?**  
La raíz cuadrada del MSE. Vuelve el error a la **misma unidad** que la variable $y$ original.

**¿Por qué usarla?**  
Interpretación directa: "el error medio típico es de X unidades". Si predices `mean radius` en mm, RMSE = 2.5 significa que en promedio te equivocas en ±2.5 mm.

**¿Cuándo?**
- Siempre que necesites comunicar resultados a stakeholders (es intuitivo).
- Para reportes finales junto con MAE.
- Cuando quieres que errores grandes pesen más que los pequeños.

**Contras:** Igual de sensible a outliers que MSE (heredado del cuadrado).

**Ejemplo:** Si RMSE = 2.5 mm, significa que típicamente el modelo se equivoca en 2.5 mm.

---

### **3) MAE — Mean Absolute Error (Error Absoluto Medio)**

$$\text{MAE}=\frac{1}{m}\sum_{i=1}^{m}\left|y_i-\hat{y}_i\right|$$

**¿Qué mide?**  
El promedio del valor absoluto de los errores (sin elevar al cuadrado).

**¿Por qué usarla?**  
Más **robusto** ante outliers que MSE/RMSE. Trata errores grandes y pequeños de forma más balanceada.

**¿Cuándo?**
- Cuando tienes outliers potenciales pero aún quieres penalizarlos (menos que MSE, pero más que ignorarlos).
- Para datos del mundo real con ruido.
- Cuando quieres una métrica simétrica y fácil de interpretar.

**Contras:** No penaliza tanto los errores extremos (a veces es una ventaja, a veces no).

**Ejemplo:** MAE = 1.8 mm → en promedio, el error absoluto es 1.8 mm (directamente interpretable).

---

### **4) MedAE — Median Absolute Error (Mediana del Error Absoluto)**

$$\text{MedAE} = \text{mediana}\left(|y_i-\hat{y}_i|\right)$$

**¿Qué mide?**  
La mediana de los errores absolutos (el valor del medio cuando ordenas los errores).

**¿Por qué usarla?**  
Aún más robusto que MAE. El 50% de tus predicciones están dentro de MedAE de error.

**¿Cuándo?**
- Con datos muy ruidosos o con muchos outliers.
- Para entender el "error típico" del 50% central de predicciones.
- Cuando quieres eliminar influencia de colas largas.

**Contras:** Menos sensible a mejoras globales (ignora los extremos).

**Ejemplo:** MedAE = 1.2 mm → la mitad de tus predicciones están dentro de ±1.2 mm.

---

### **5) R² — Coeficiente de Determinación**

$$R^{2}=1-\frac{\sum_{i}(y_i-\hat{y}_i)^2}{\sum_{i}(y_i-\bar{y})^2}$$

**¿Qué mide?**  
Proporción de varianza en $y$ que tu modelo explica (vs. simplemente predecir la media $\bar{y}$).

**¿Por qué usarla?**  
Escala de 0 a 1 (típicamente). Fácil de entender: "mi modelo explica el X% de la variación".

**¿Cuándo?**
- Para reportar el "goodness of fit" general.
- Para comparar modelos rápidamente (el mayor R², mejor).
- Siempre en reportes profesionales.

**Interpretación:**
- $R^2 = 1$: ajuste perfecto.
- $R^2 = 0.85$: explicas el 85% de la varianza (bueno).
- $R^2 = 0$: tu modelo es tan malo como predecir la media.
- $R^2 < 0$: peor que predecir la media (¡muy malo!).

**Contras:** Sube siempre al agregar variables (incluso si no ayudan realmente) → usa R² ajustado.

**Ejemplo:** R² = 0.88 → el modelo explica el 88% de la variación en `mean radius`.

---

### **6) R² Ajustado (Adjusted R²)**
Para comparar modelos con **distinto número de features** $p$

$$R^2_{\text{adj}}=1-\left(1-R^2\right)\cdot\frac{n-1}{n-p-1}$$

**¿Qué mide?**  
R² pero penalizando por complejidad (número de variables). Penaliza agregar variables innecesarias.

**¿Por qué usarla?**  
Para comparar justamente: un modelo con 5 features vs. otro con 15 features.

**¿Cuándo?**
- Al elegir el grado del polinomio (grado 2, 3, 4...).
- Al comparar modelos con diferente complejidad.
- Cuando quieres evitar overfitting por exceso de variables.

**Ejemplo:**
- Polinomio grado 2: R² = 0.85, R²_adj = 0.84
- Polinomio grado 10: R² = 0.95, R²_adj = 0.80 ← ¡el grado 2 es mejor! (ajustado penalizó el grado 10)

---

### **7) MAPE — Mean Absolute Percentage Error**

$$\text{MAPE}=\frac{100\%}{m}\sum_{i=1}^{m}\left|\frac{y_i-\hat{y}_i}{y_i+\varepsilon}\right|$$

**¿Qué mide?**  
El error absoluto promedio en **porcentaje** (relativo al valor real).

**¿Por qué usarla?**  
Es interpretable en %. Si MAPE = 5%, te equivocas en ~5% en promedio.

**¿Cuándo?**
- En pronósticos de ventas, demanda, precios (donde % importa).
- Cuando quieres saber "qué tan lejos estoy en términos relativos".

**Contras:**
- **Muy inestable** si hay valores cercanos a 0 (división por casi cero).
- No es simétrico (un error del +50% y -50% no son equivalentes).
- Prefiere sobrepredicción a subpredicción.

**Ejemplo:** MAPE = 3.5% → en promedio te equivocas ~3.5% del valor real.

---

### **8) Explained Variance (Varianza Explicada)**

$$\text{EV} = 1 - \frac{\mathrm{Var}(y-\hat{y})}{\mathrm{Var}(y)}$$

**¿Qué mide?**  
Qué tanto reduce tu modelo la varianza del error vs. la varianza original.

**¿Por qué usarla?**  
Similar a R² pero con otra perspectiva: ¿cuánta incertidumbre elimino?

**¿Cuándo?**
- En problemas donde quieres entender reducción de varianza.
- Como validación adicional junto con R².

**Contras:** Muy similar a R², así que es redundante en muchos casos.

**Ejemplo:** EV = 0.87 → reduces la varianza en 87% vs. la original.

---

## ✅ Buenas Prácticas

1. **Reporta siempre dos métricas:**
   - Opción A: **RMSE + MAE** (te muestran el error desde dos perspectivas)
   - Opción B: **R² + RMSE** (explica varianza + error interpretable)
   
2. **Visualización obligatoria:**
   - Gráfico de `y_real` vs `y_predicho` (scatter plot con línea y=x)
   - Residuos vs. predicciones

3. **Usa R² ajustado** al comparar modelos con diferente complejidad (grados polinomiales, número de features).

4. **Evita MAPE** si tienes valores cercanos a 0; usa MAE o RMSE en su lugar.

5. **Con outliers:**
   - MSE/RMSE: sensibles
   - MAE: robusto
   - MedAE: muy robusto

---

## 🔗 Resumen Rápido: Cuál usar cuándo

| Escenario | Métrica Principal | Secundaria |
|-----------|-------------------|-----------|
| Comparar polinomios (grado 2 vs 3 vs 4) | R² ajustado | RMSE |
| Reportar a jefes/clientes | RMSE | R² |
| Datos con outliers | MAE | MedAE |
| Series de tiempo / pronósticos | MAPE | RMSE |
| Validación científica | R² | MAE |
| Publicación académica | RMSE + R² | MAE |

In [ ]:
#@title ⚙️ Métricas de Regresión (sklearn) para tu **modelo desde cero**
#@markdown Calcula MSE, RMSE, MAE, MedAE, R², **R² ajustado** y **Explained Variance**
#@markdown usando las salidas del modelo **implementado desde cero**: `y_pred_train`, `y_pred_test`, `ytr`, `yte`.

import numpy as np
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, median_absolute_error,
    r2_score, explained_variance_score
)

# --- Asegurar forma 1D ---
y_true_tr = np.asarray(ytr).ravel()
y_pred_tr = np.asarray(y_pred_train).ravel()
y_true_te = np.asarray(yte).ravel()
y_pred_te = np.asarray(y_pred_test).ravel()

# --- Métricas TRAIN (sklearn) ---
mse_tr  = mean_squared_error(y_true_tr, y_pred_tr)
rmse_tr = np.sqrt(mse_tr)  # ✅ RMSE manualmente (compatible con sklearn antiguo)
mae_tr  = mean_absolute_error(y_true_tr, y_pred_tr)
medae_tr = median_absolute_error(y_true_tr, y_pred_tr)
r2_tr   = r2_score(y_true_tr, y_pred_tr)
ev_tr   = explained_variance_score(y_true_tr, y_pred_tr)

# --- Métricas TEST (sklearn) ---
mse_te  = mean_squared_error(y_true_te, y_pred_te)
rmse_te = np.sqrt(mse_te)  # ✅ RMSE manualmente
mae_te  = mean_absolute_error(y_true_te, y_pred_te)
medae_te = median_absolute_error(y_true_te, y_pred_te)
r2_te   = r2_score(y_true_te, y_pred_te)
ev_te   = explained_variance_score(y_true_te, y_pred_te)

# --- R² ajustado (usa #features polinomiales p = d_poly - 1) ---
def r2_adjusted(r2, n, p):
    """
    Calcula R² ajustado para penalizar por complejidad del modelo.

    Parámetros:
    - r2: R² sin ajustar
    - n: número de muestras
    - p: número de features (excluyendo bias)

    Fórmula: R²_adj = 1 - (1-R²) * (n-1)/(n-p-1)
    """
    denom = (n - p - 1)
    if denom <= 0:
        return np.nan
    return 1 - (1 - r2) * (n - 1) / denom

# d_poly viene de tu celda anterior: Xtr_poly_b.shape[1] (= p + 1 por el bias)
# Ajusta 'd_poly' si tu variable se llama diferente
p_features = int(d_poly - 1)

r2adj_tr = r2_adjusted(r2_tr, n=len(y_true_tr), p=p_features)
r2adj_te = r2_adjusted(r2_te, n=len(y_true_te), p=p_features)

# --- Imprimir limpio ---
print("=" * 100)
print("MÉTRICAS DE REGRESIÓN - TRAIN vs TEST")
print("=" * 100)
print("\n📊 TRAIN:")
print(f"  MSE:      {mse_tr:.4f}")
print(f"  RMSE:     {rmse_tr:.4f}")
print(f"  MAE:      {mae_tr:.4f}")
print(f"  MedAE:    {medae_tr:.4f}")
print(f"  R²:       {r2_tr:.4f}")
print(f"  R² adj:   {r2adj_tr:.4f}")
print(f"  Var Expl: {ev_tr:.4f}")

print("\n📊 TEST:")
print(f"  MSE:      {mse_te:.4f}")
print(f"  RMSE:     {rmse_te:.4f}")
print(f"  MAE:      {mae_te:.4f}")
print(f"  MedAE:    {medae_te:.4f}")
print(f"  R²:       {r2_te:.4f}")
print(f"  R² adj:   {r2adj_te:.4f}")
print(f"  Var Expl: {ev_te:.4f}")

print("\n" + "=" * 100)
print("✅ COMPARATIVA TRAIN vs TEST (para detectar overfitting):")
print("=" * 100)
print(f"RMSE:   Train={rmse_tr:.4f} | Test={rmse_te:.4f} | Δ={(rmse_te - rmse_tr):.4f} " +
      ("(⚠️ OVERFITTING)" if rmse_te > rmse_tr * 1.1 else "✓ OK"))
print(f"R²:     Train={r2_tr:.4f} | Test={r2_te:.4f} | Δ={(r2_te - r2_tr):.4f} " +
      ("(⚠️ OVERFITTING)" if r2_te < r2_tr - 0.05 else "✓ OK"))
print(f"MAE:    Train={mae_tr:.4f} | Test={mae_te:.4f} | Δ={(mae_te - mae_tr):.4f} " +
      ("(⚠️ OVERFITTING)" if mae_te > mae_tr * 1.1 else "✓ OK"))


# 📏 Interpretación de Métricas de Regresión (TRAIN vs TEST)
*(Contexto: y = `mean radius` del tumor; modelo de **Regresión Polinómica**)*

A continuación se explica **qué mide** cada métrica, **cómo leer tus resultados** (con base en la tabla mostrada) y **qué hacer para mejorar** en caso de que el valor no sea el deseado.

---

## 1) MSE — *Mean Squared Error* (Error Cuadrático Medio)
**Fórmula:**  
$$\text{MSE}=\frac{1}{m}\sum_{i=1}^{m}(y_i-\hat{y}_i)^2$$

**Qué mide:** Promedio del **cuadrado** de los errores. Penaliza **mucho** los errores grandes (outliers).

**Cómo leerlo:** **Más bajo = mejor.**  
- En tu salida: **Train ≈ 1.91**, **Test ≈ 1.30** → El error medio cuadrático es **un poco menor en test** (bien; no hay señales de sobreajuste).

**Si es alto:**  
- Revisa **outliers** (winsorizar, transformar, o usar pérdida robusta).  
- **Incrementa capacidad** del modelo si está infraajustado (↑ grado polinómico con validación).  
- **Regulariza** si hay sobreajuste (Ridge/Lasso).  
- Mejora **features** (selección/creación), y asegúrate de **escalar**.

---

## 2) RMSE — *Root Mean Squared Error* (Raíz del MSE)
**Fórmula:**  
$$\text{RMSE}=\sqrt{\text{MSE}}$$

**Qué mide:** Error típico en **las mismas unidades** de la variable (mm). Interpretación directa.

**Cómo leerlo:** **Más bajo = mejor.**  
- En tu salida: **Train ≈ 1.38 mm**, **Test ≈ 1.14 mm** → El error típico en test es de ~**1.14 mm** (bueno).

**Si es alto:** Igual que MSE, pero piensa en **unidades reales**:  
- ¿1–2 mm es aceptable clínicamente? Si no, aplica las mejoras de MSE.

---

## 3) MAE — *Mean Absolute Error* (Error Absoluto Medio)
**Fórmula:**  
$$\text{MAE}=\frac{1}{m}\sum_{i=1}^{m}\left|y_i-\hat{y}_i\right|$$

**Qué mide:** Error promedio **absoluto** (robusto a outliers). Fácil de comunicar.

**Cómo leerlo:** **Más bajo = mejor.**  
- En tu salida: **Train ≈ 1.03 mm**, **Test ≈ 0.84 mm** → En promedio, el modelo se equivoca **menos de 1 mm** (muy razonable).

**Si es alto:**  
- Revisa **sesgos sistemáticos** (residuales vs predicción).  
- **Feature engineering**, **interacciones** o **no linealidades** (polinomios) si hay patrones.

---

## 4) MedAE — *Median Absolute Error* (Error Absoluto Mediano)
**Fórmula:**  
$$\text{MedAE}=\mathrm{median}\!\left(\,|y_i-\hat{y}_i|\,\right)$$

**Qué mide:** El error “típico” **mediano** (aún más robusto a outliers que MAE).

**Cómo leerlo:** **Más bajo = mejor.**  
- En tu salida: **Train ≈ 0.77 mm**, **Test ≈ 0.62 mm** → La mitad de los errores están por debajo de ~**0.6–0.8 mm** (excelente).

**Si es alto:**  
- Señal de **dispersión** general de errores → mejora de **features** o **capacidad del modelo**.

---

## 5) \(R^2\) — *Coeficiente de determinación*
**Fórmula:**  
$$R^{2}=1-\frac{\sum_{i}(y_i-\hat{y}_i)^2}{\sum_{i}(y_i-\bar{y})^2}$$

**Qué mide:** Proporción de **varianza explicada** por el modelo.  
**Rango:** 1 (perfecto), 0 (igual a la media), <0 (peor que la media).

**Cómo leerlo:** **Más alto = mejor.**  
- En tu salida: **Train ≈ 0.847**, **Test ≈ 0.893** → El modelo explica **~85–89%** de la variabilidad (muy bien). Que test sea **ligeramente mayor** que train es **aceptable** (no hay sobreajuste).

**Si es bajo:**  
- Posible **infraajuste** → añade **no linealidad** (↑ grado), **interacciones** o **otras features**.  
- Verifica **fugas de información** y **escalado**.

---

## 6) \(R^2_Ajustado (Adjusted R²)\) — *R² ajustado* (penaliza complejidad)
**Fórmula:**  
$$R^2_{\text{adj}}=1-(1-R^2)\cdot\frac{n-1}{n-p-1}$$

**Qué mide:** Ajusta \(R^2\) penalizando el número de **features** \(p\).  
**Cómo leerlo:** **Más alto = mejor** y útil para **comparar grados polinómicos**.

- En tu salida: **Train ≈ 0.841**, **Test ≈ 0.871** → La complejidad actual está **justificada** por la ganancia en varianza explicada.

**Si cae al subir el grado:**  
- Estás **sobreajustando** (exceso de términos). **Reduce grado** o **usa regularización** (Ridge/Lasso).

---

## 7) Varianza Explicada (*Explained Variance, EV*)
**Fórmula:**  
$$\text{EV}=1-\frac{\mathrm{Var}(y-\hat{y})}{\mathrm{Var}(y)}$$

**Qué mide:** Fracción de **varianza** que el modelo logra “explicar” (similar a \(R^2\)).  
**Cómo leerlo:** **Más alto = mejor.**

- En tu salida: **Train ≈ 0.853**, **Test ≈ 0.897** → Consistente con \(R^2\); buen ajuste **global**.

**Si es baja:**  
- Igual que \(R^2\): añade **capacidad** (hasta donde generalice), mejora **representación** de variables.

---

## 🧪 Diagnóstico rápido de *overfitting* / *underfitting*
- **Overfitting:** Métricas muy **buenas en TRAIN** y significativamente **peores en TEST**  
  **Acciones:** ↓ grado polinómico, **Ridge/Lasso**, más datos, validación cruzada, simplificar features.
- **Underfitting:** Métricas **malas en ambos** (alto error, bajo \(R^2\))  
  **Acciones:** ↑ capacidad (grado/interacciones), **feature engineering**, probar modelos más ricos.

> **Tu caso:** TEST está **igual o mejor** que TRAIN → **no** hay señales de sobreajuste; ajuste **sano**.

---

## 🔧 Recetas para mejorar (según lo que veas)

- **Bajar MSE/RMSE/MAE/MedAE:**  
  - **Features** mejores (interacciones, transformaciones),  
  - **Grado** polinómico con **validación cruzada**,  
  - **Regularización** (Ridge si hay muchas features o colinealidad),  
  - Manejo de **outliers** (robust scaling, winsorización),  
  - **Más datos** y **mejor preprocesamiento** (escalado consistente).

- **Subir \(R^2\)/EV sin sobreajuste:**  
  - Aumenta complejidad **moderadamente** + **Ridge/Lasso**,  
  - Selección de variables, **PCA** si hay multicolinealidad,  
  - Validación cruzada para **elegir grado** óptimo.

- **Sospecha de patrón en residuales (no aleatorios):**  
  - Añade **términos no lineales** o **interacciones**,  
  - Considera **modelos no lineales** (árboles/boosting) si la relación no es bien capturada por polinomios.

---

## 🧭 Regla práctica para clase
- Reporta **RMSE + MAE** (unidades reales) y **\(R^2\)/\(R^2_Ajustado (Adjusted R²))** (explicación de varianza).  
- Acompaña con dos gráficos: **y vs ŷ** (línea identidad) y **residuales vs ŷ** (ruido alrededor de 0).  
- Decide el **grado** mirando **\(R^2_Ajustado (Adjusted R²)\)** y **errores en TEST** (no solo en TRAIN).

